# BLAZE — configuração, firmware e ensaios do protótipo

Execute as seções na ordem. As seções 1–5 configuram e simulam no PC. A seção 6 define o hardware do ESP32, a 7 salva a combinação completa, a 8 prepara os programas e grava o firmware, e as seções 9–11 fazem os ensaios de conexão, sensores e integração. A seção 12 abre a central independente.


## 1. Verificar ambiente

**Objetivo:** confirmar que o kernel preparado foi selecionado e carregar as funções compartilhadas com o runtime.

**Entrada:** Kernel de `.bootstrap/python312` selecionado no VS Code.

**Resultado:** caminho do Python em `.bootstrap/python312`. Esta célula não acessa hardware.

**Código chamado:** `scripts/preparar_notebooks.py`, as funções deste notebook, `pc_operador/runtime.py`.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pc_operador' / 'runtime.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra a pasta blaze_prototipo_2026_10 no VS Code.')
EXPECTED = ROOT / '.bootstrap' / 'python312'
if not Path(sys.executable).resolve().is_relative_to(EXPECTED.resolve()):
    raise RuntimeError('Rode PRIMEIRO scripts/preparar_notebooks_windows.cmd no Windows ou a tarefa de preparação no Ubuntu; então selecione o kernel em .bootstrap/python312.')
from IPython.display import display, Image
import ipywidgets as widgets
sys.path.insert(0, str(ROOT / 'pc_operador'))
from runtime import AppConfig, discover_services, estimate
results = {}
print('Kernel pronto:', sys.executable)

In [ ]:
# Funções e verificações deste notebook: execução sem arquivos Python de teste auxiliares.
import csv, io, math, os, shutil, socket, subprocess, tempfile, threading, time, types, unittest
from unittest.mock import patch
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
import runtime as config
import runtime
from runtime import (AppConfig, OperatorInputs, ESPConnection, VisionSettings,
                     VisionEngine, VisionSession, VISION_DEFAULTS, box_iou, VisionWorker, CameraSource, control_message,
                     decode_telemetry, encode_message, pixel_to_servos, estimate,
                     DeviceScanner, Probe)
import cv2
import numpy as np
HERE = ROOT / 'pc_operador'

class NotebookSetupTests(unittest.TestCase):
    def test_setup_and_twelve_point_calibration_are_read_per_combination(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / "parameters_setups/user").mkdir(parents=True)
            (root / "models").mkdir()
            (root / "models/fire_yolo_best.pt").write_bytes(b"placeholder")
            (root / "parameters_setups/user/rgb_monocular_fire_monitor_setups.json").write_text(
                json.dumps({"ajuste": {"saved_at": "2026-10-05T10:30:00", "config": {
                    "jet_drop_px": 37, "fire_model_path": "/outro/pc/fire_yolo_best.pt"}}}),
                encoding="utf-8")
            angles={str(angle):{corner:[90,90] for corner in ('tl','tr','bl','br')}
                    for angle in (0,90,180)}
            (root / "parameters_setups/user/camera_cannon_calibrations.json").write_text(
                json.dumps({"combinacao": {"enabled": True, "angles": angles}}), encoding="utf-8")
            (root / "config.example.json").write_text(json.dumps({
                "camera": {"source": 0}, "esp": {}, "xbox_axes": {}, "xbox_buttons": {},
                "vision": {"legacy_setup_file": "parameters_setups/user/rgb_monocular_fire_monitor_setups.json",
                           "fire_model": "models/fire_yolo_best.pt", "person_model": "models/yolo11n.pt"}}),
                encoding="utf-8")
            setup=root / "config.json"
            setup.write_text(json.dumps({"_official":{"name":"combinacao"}}),encoding="utf-8")
            with patch.object(config, "ROOT", root):
                cfg = config.AppConfig(setup)
            self.assertEqual(cfg.vision.params["jet_drop_px"], 37)
            self.assertEqual(cfg.vision.source_name, "ajuste")
            self.assertTrue(runtime.camera_calibration_complete(cfg.vision.calibration))
            self.assertEqual(cfg.vision.fire_model, (root / "models/fire_yolo_best.pt").resolve())


class ProtocolTests(unittest.TestCase):
    def test_manual_command_does_not_invent_a_valid_camera(self):
        cmd = control_message({"requested_mode": "manual", "drive_x": 10, "drive_y": -10}, {})
        self.assertEqual(cmd["drive_x"], 1)
        self.assertEqual(cmd["drive_y"], -1)
        self.assertFalse(cmd["vision_valid"])
        self.assertFalse(cmd["jet_enable"])
        data = encode_message(cmd)
        self.assertTrue(data.endswith(b"\n"))

    def test_telemetry_requires_expected_type(self):
        self.assertEqual(decode_telemetry(b'{"type":"telemetry","armed":false}')["armed"], False)
        with self.assertRaises(ValueError):
            decode_telemetry(b'{"type":"control"}')


class KinematicsTests(unittest.TestCase):
    def test_without_calibration_no_speed_is_invented(self):
        self.assertIsNone(estimate(1, 1, {"track_width_m": .4,
                                        "sprocket_radius_m": .05, "output_max_rpm": 0}))

    def test_straight_and_turn(self):
        geometry = {"track_width_m": .4, "sprocket_radius_m": .05, "output_max_rpm": 60}
        straight = estimate(1, 1, geometry)
        self.assertAlmostEqual(straight["vx_body_mps_est"], 2 * math.pi * .05)
        self.assertEqual(straight["yaw_rad_s_est"], 0)
        turn = estimate(-1, 1, geometry)
        self.assertEqual(turn["vx_body_mps_est"], 0)
        self.assertGreater(turn["yaw_rad_s_est"], 0)
        self.assertEqual(turn["vy_body_mps_est"], 0)

    def test_link_loss_zeros_local_control_and_demands_rearm(self):
        state = OperatorInputs({}, {})
        state.pygame = None  # isola teclado mesmo se houver Xbox conectado ao PC
        state.joy = None
        state._want_link = True
        state.lt = 1.
        state.set_link(True)
        state.key("w", True)
        state.poll()
        self.assertNotEqual(state.snapshot()["drive_y"], 0)
        state.set_link(False)
        self.assertEqual(state.snapshot()["drive_y"], 0)
        self.assertTrue(state.snapshot()["stop"])
        state.action("rearm")
        self.assertTrue(state.snapshot()["stop"])
        state.key("w", False)


class LinkTest(unittest.TestCase):
    def test_control_goes_out_and_telemetry_returns(self):
        result = {}
        arrived = threading.Event()

        class FakeSocket:
            sent = []
            received = False

            def __enter__(self): return self
            def __exit__(self, *_): return False
            def settimeout(self, *_): pass
            def close(self): pass
            def sendall(self, message): self.sent.append(message)

            def recv(self, _):
                if not self.received:
                    self.received = True
                    return b'{"type":"telemetry","mode":"manual","armed":false}\n'
                raise socket.timeout

        fake = FakeSocket()
        link = ESPConnection("fake-esp.local", 8765,
                             lambda: {"type": "control", "mode": "manual"},
                             lambda data: (result.setdefault("telemetry", data), arrived.set()),
                             .1, transport='wifi')
        with patch("runtime.socket.create_connection", return_value=fake):
            link.start()
            self.assertTrue(arrived.wait(2), "telemetria não chegou")
            link.stop()
        self.assertIn(b'"type":"control"', fake.sent[0])
        self.assertTrue(json.loads(fake.sent[-1])["stop"])
        self.assertEqual(result["telemetry"]["armed"], False)

    def test_auto_uses_usb_telemetry_without_opening_wifi(self):
        arrived = threading.Event()
        sent = []

        class FakeSerial:
            def reset_input_buffer(self): pass
            def write(self, packet): sent.append(json.loads(packet)); return len(packet)
            def read(self, _):
                if sent:
                    return b'{"type":"telemetry","armed":false}\n'
                return b''
            def close(self): pass

        link = ESPConnection('unreachable', 8765, lambda: {'type': 'control', 'stop': False},
                             lambda _: arrived.set(), .1)
        with patch('runtime.serial_candidates', return_value=['COM8']), \
             patch.object(link, '_usb', return_value=FakeSerial()), \
             patch.object(link, '_wifi_local', side_effect=AssertionError('AP tentou abrir')):
            link.start()
            self.assertTrue(arrived.wait(2))
            self.assertEqual(link.active_transport, 'usb')
            link.stop()
        self.assertTrue(sent[0]['stop'])  # primeiro comando sempre neutro

    def test_auto_falls_back_to_local_ap_when_usb_cannot_open(self):
        arrived = threading.Event()

        class FakeSocket:
            def sendall(self, packet): pass
            def recv(self, _):
                arrived.set()
                return b'{"type":"telemetry","armed":false}\n'
            def close(self): pass

        link = ESPConnection('fake', 8765, lambda: {'type':'control'}, lambda _: None, .1)
        with patch('runtime.serial_candidates', return_value=['COM8']), \
             patch.object(link, '_usb', side_effect=OSError('porta indisponível')), \
             patch.object(link, '_wifi_local', return_value=FakeSocket()):
            link.start()
            self.assertTrue(arrived.wait(2))
            link.stop()
        self.assertEqual(link.connected, False)


class ActuatorProfileTests(unittest.TestCase):
    def test_default_combination_uses_no_saved_section(self):
        import profile_manager as profiles
        with tempfile.TemporaryDirectory() as directory:
            user=Path(directory)
            selection={section:'' for section in profiles.SECTIONS}
            (user/'combinations.json').write_text(json.dumps({
                'default*default*default*default':{'sections':selection}}),encoding='utf-8')
            with patch.object(profiles,'USER',user):
                data=profiles.assemble('default*default*default*default')
        self.assertEqual(data['_official']['profile'],'simulacao')
        self.assertEqual(data['hardware']['drive_kind'],'dc_3_6v')
        self.assertEqual(data['hardware']['drive_driver_model'],'BTS7960')
        self.assertFalse(data['hardware']['physical_outputs'])

    def test_selected_motor_models_define_isolated_firmware_backend(self):
        import profile_manager as profiles
        with tempfile.TemporaryDirectory() as directory:
            temp=Path(directory);user=temp/'user';user.mkdir()
            hardware=copy.deepcopy(profiles._read(profiles.DEFAULTS/'hardware.json'))
            hardware['hardware'].update(drive_kind='bldc_ktc86',
                drive_driver_model='KTC-J-DR-BLDC750',cannon_kind='ht23_ec1000',
                cannon_driver_model='KTC-DR23EC-K')
            (user/'hardware.json').write_text(json.dumps({'perfil_teste':hardware}),encoding='utf-8')
            selection={'connection':'','vision':'','control':'','hardware':'perfil_teste'}
            (user/'combinations.json').write_text(json.dumps({'perfil_teste':{'sections':selection}}),encoding='utf-8')
            with patch.object(profiles,'USER',user), \
                 patch.object(profiles,'RUNTIME_SETUPS',temp/'setups'), \
                 patch.object(profiles,'FIRMWARE_BUILDS',temp/'builds'), \
                 patch.object(profiles,'local_link_credentials',return_value=('TEST','password','TEST','123456')):
                project,data=profiles.firmware_project('perfil_teste')
            header=(project/'include/setup_generated.h').read_text(encoding='utf-8')
            self.assertIn('#define BLAZE_DRIVE_KIND 3',header)
            self.assertIn('#define BLAZE_CANNON_KIND 2',header)
            self.assertEqual(data['_official']['actuator_backend'],'bldc_ktc86/ht23_ec1000')
            self.assertIn('BLAZE_DRIVE_KIND == 1 || BLAZE_DRIVE_KIND == 2',
                          (project/'src/main.cpp').read_text(encoding='utf-8'))

    def test_both_dc_motor_options_use_bts_enable_pins(self):
        import profile_manager as profiles
        source=(profiles.ROOT/'esp32/src/main.cpp').read_text(encoding='utf-8')
        self.assertIn('if (BLAZE_DRIVE_KIND == 1 || BLAZE_DRIVE_KIND == 2)',source)
        self.assertEqual(profiles.DRIVE_KINDS['dc_3_6v'],1)
        self.assertEqual(profiles.DRIVE_KINDS['bts7960_12v'],2)
        with tempfile.TemporaryDirectory() as directory:
            user=Path(directory)
            hw=copy.deepcopy(profiles._read(profiles.DEFAULTS/'hardware.json'))
            hw['hardware'].update(physical_outputs=True,drive_kind='dc_3_6v',
                                  drive_driver_model='BTS7960')
            vision=copy.deepcopy(profiles._read(profiles.DEFAULTS/'vision.json'))
            vision['vision']['combat_mode']='real'
            (user/'hardware.json').write_text(json.dumps({'motor3v':hw}),encoding='utf-8')
            (user/'vision.json').write_text(json.dumps({'real':vision}),encoding='utf-8')
            (user/'combinations.json').write_text(json.dumps({'3V*BTS':{'sections':{
                'connection':'','vision':'real','control':'','hardware':'motor3v'}}}),encoding='utf-8')
            with patch.object(profiles,'USER',user):
                data=profiles.assemble('3V*BTS')
        self.assertEqual(data['hardware']['drive_driver_model'],'BTS7960')
        self.assertEqual(data['_official']['profile'],'combate_real')

    def test_real_profile_rejects_wrong_traction_driver(self):
        import profile_manager as profiles
        with tempfile.TemporaryDirectory() as directory:
            user=Path(directory)
            hw=copy.deepcopy(profiles._read(profiles.DEFAULTS/'hardware.json'))
            hw['hardware'].update(physical_outputs=True,drive_kind='dc_3_6v',
                                  drive_driver_model='DRV8833')
            vision=copy.deepcopy(profiles._read(profiles.DEFAULTS/'vision.json'))
            vision['vision']['combat_mode']='real'
            (user/'hardware.json').write_text(json.dumps({'wrong':hw}),encoding='utf-8')
            (user/'vision.json').write_text(json.dumps({'real':vision}),encoding='utf-8')
            (user/'combinations.json').write_text(json.dumps({'invalid':{'sections':{
                'connection':'','vision':'real','control':'','hardware':'wrong'}}}),encoding='utf-8')
            with patch.object(profiles,'USER',user):
                with self.assertRaisesRegex(ValueError,'BTS7960'):
                    profiles.assemble('invalid')

@unittest.skipIf(cv2 is None, "OpenCV não instalado neste ambiente")
class VisionSimulationTests(unittest.TestCase):
    def test_calibration_interpolates_twelve_points_and_camera_angle(self):
        from runtime import camera_calibration_complete
        angles={}
        for camera_angle in (0,90,180):
            angles[str(camera_angle)]={
                'tl':[25+camera_angle/3,145], 'tr':[45+camera_angle/3,145],
                'bl':[25+camera_angle/3,35], 'br':[45+camera_angle/3,35]}
        calibration={'enabled':True,'angles':angles,'frame_size':[100,100]}
        self.assertTrue(camera_calibration_complete(calibration))
        pan,tilt=pixel_to_servos((49.5,49.5),(100,100,3),calibration,{},45)
        self.assertAlmostEqual(pan,50)
        self.assertAlmostEqual(tilt,90)
        self.assertFalse(camera_calibration_complete({'enabled':True,'angles':{'90':angles['90']}}))

    def test_fire_boxes_respect_area_and_do_not_overlap(self):
        engine=VisionSession.__new__(VisionSession)
        engine.params={'max_fire_box_area_px':100}
        mask=np.zeros((40,40),dtype=np.uint8)
        mask[5:25,5:25]=255
        engine._fire_mask=lambda frame,raw,cell:mask
        boxes,regions=engine._small_boxes_and_regions(np.zeros((40,40,3),np.uint8),
            [{'box':(5,5,25,25),'conf':.8}])
        self.assertGreater(len(boxes),1)
        self.assertTrue(regions)
        for i,a in enumerate(boxes):
            self.assertLessEqual(a['area'],100)
            for b in boxes[i+1:]:
                self.assertEqual(box_iou(a['box'],b['box']),0.)

    def test_person_blocks_jet_and_clears_trail(self):
        engine=VisionSession.__new__(VisionSession)
        engine.fire_model=engine.person_model=object()
        engine.firefighter_model=None
        engine.params={**VISION_DEFAULTS,'input_fps':25}
        engine.mode='simulation';engine.source='test';engine.calibration={}
        engine.journal=io.StringIO()
        engine.reset()
        engine.trail=[(time.monotonic(),20,20)]
        def detection(painted,now,force=False):
            engine.people=[{'box':(1,1,12,12),'conf':.9}]
            engine.firefighters=[];engine.fires=[];engine.regions=[]
            engine.last_person=now
        engine._detect=detection
        _,record=engine.process(np.zeros((40,40,3),np.uint8))
        self.assertTrue(record['blocked'])
        self.assertFalse(record['jet_permitted'])
        self.assertEqual(engine.trail,[])

    def test_missing_models_does_not_validate_vision(self):
        settings=VisionSettings({}, {}, None, None, 'test')
        engine=VisionEngine(settings,{})
        output=engine.process(np.zeros((40,40,3),np.uint8),time.monotonic(),
                              'auto',False,(0,0))
        self.assertFalse(output.valid)
        self.assertFalse(output.target_valid)


class RuntimeControlTests(unittest.TestCase):
    def test_manual_camera_and_cannon_reciprocal_following(self):
        from runtime import coupled_manual_pose
        pan,tilt,camera,aim_x,aim_y=0.,45.,90.,0.,0.
        for _ in range(85):
            pan,tilt,camera,aim_x,aim_y=coupled_manual_pose(
                pan,tilt,camera,1.,0.,0,.1)
        self.assertGreater(camera,90.)
        self.assertLess(abs(aim_x),1.)
        for _ in range(85):
            pan,tilt,camera,aim_x,aim_y=coupled_manual_pose(
                pan,tilt,camera,0.,0.,-1,.1)
        self.assertLess(camera,90.)
        self.assertLess(pan,0.)
        self.assertLess(abs(aim_x),1.)

    def test_rpm_limit_caps_both_track_commands(self):
        state=OperatorInputs({}, {}, speed_limit_rpm=20, calibrated_max_rpm=80)
        state.pygame=None;state.joy=None;state.set_link(True)
        state.key('w',True);state.key('d',True);state.poll()
        snap=state.snapshot()
        self.assertLessEqual(max(abs(-snap['drive_y']+snap['drive_x']),
                                 abs(-snap['drive_y']-snap['drive_x'])),.25+1e-6)

    def test_uncalibrated_rpm_blocks_traction(self):
        state=OperatorInputs({}, {}, speed_limit_rpm=40, calibrated_max_rpm=0)
        state.pygame=None;state.joy=None;state.set_link(True)
        state.key('w',True);state.poll()
        self.assertEqual(state.snapshot()['drive_y'],0.)

class DeviceScannerTests(unittest.TestCase):
    def test_scan_reports_result_and_completes(self):
        updates = []
        completed = threading.Event()
        def callback(label, state, detail):
            updates.append((label, state, detail))
            if label == '__overall__': completed.set()
        with tempfile.TemporaryDirectory() as directory:
            scanner = DeviceScanner(Path(directory), callback, command_factory=lambda task: [
                sys.executable, '-c', 'print("BLAZE_DEVICES:" + __import__("json").dumps([{"index": 0}]))'])
            self.assertTrue(scanner.start([Probe('controle', 'joystick', timeout=2)]))
            self.assertTrue(completed.wait(3))
        self.assertIn(('controle', 'detectado', [{'index': 0}]), updates)
        self.assertEqual(updates[-1][1], 'concluída')

    def test_stop_kills_active_probe(self):
        updates = []
        started = threading.Event()
        completed = threading.Event()
        def callback(label, state, detail):
            updates.append((label, state))
            if state == 'buscando': started.set()
            if label == '__overall__': completed.set()
        with tempfile.TemporaryDirectory() as directory:
            scanner = DeviceScanner(Path(directory), callback, command_factory=lambda task: [
                sys.executable, '-c', 'import time; time.sleep(20)'])
            self.assertTrue(scanner.start([Probe('driver lento', 'camera', timeout=20), Probe('outro', 'camera')]))
            self.assertTrue(started.wait(2))
            time.sleep(0.1)
            began = time.monotonic()
            scanner.stop()
            self.assertTrue(completed.wait(2))
        self.assertLess(time.monotonic() - began, 2)
        self.assertIn(('__overall__', 'interrompida'), updates)
        self.assertNotIn(('outro', 'buscando'), updates)


def pause(message, automatic):
    if not automatic and sys.stdin.isatty():
        try:
            input(message + '  [Enter para continuar] ')
        except EOFError:
            pass


def check_structure():
    required = [HERE / 'runtime.py', HERE / 'requirements.txt',
                HERE / 'config.example.json']
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError('Arquivos ausentes: ' + ', '.join(missing))
    print('Estrutura do programa PC: OK')


def stage_config(automatic):
    from runtime import AppConfig

    target = HERE / 'config.json'
    if not target.exists():
        shutil.copy2(HERE / 'config.example.json', target)
        print('Configuração criada:', target)
    if sys.stdin.isatty() and not automatic:
        data = json.loads(target.read_text(encoding='utf-8'))
        try:
            source = input('Fonte de vídeo (0/1 para USB ou URL IP; Enter mantém): ').strip()
            host = input('IP do ESP32 (Enter mantém descoberta automática): ').strip()
            setup = input('Nome do setup calibrado do notebook (Enter mantém): ').strip()
        except EOFError:
            source = host = setup = ''
        if source:
            data.setdefault('camera', {})['source'] = int(source) if source.isdigit() else source
        if host:
            data.setdefault('esp', {})['host'] = host
        if setup:
            data.setdefault('vision', {})['legacy_setup_name'] = setup
        if source or host or setup:
            temporary = target.with_suffix('.json.tmp')
            temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
            temporary.replace(target)
            print('Ajustes preliminares registrados em config.json.')
    cfg = AppConfig(target)
    if cfg.vision.person_model is None:
        # Os pesos públicos YOLO11n são baixados pela própria Ultralytics.
        # O peso de chama é específico da equipe e não pode ser substituído.
        models = HERE / 'models'
        models.mkdir(exist_ok=True)
        current = Path.cwd()
        try:
            from ultralytics import YOLO
            os.chdir(models)
            print('Obtendo yolo11n.pt pelo mecanismo oficial da Ultralytics...')
            YOLO('yolo11n.pt')
        except Exception as exc:
            print('Modelo público de pessoas não disponível:', exc)
        finally:
            os.chdir(current)
        cfg.reload()
    print('Câmera configurada:', cfg.camera_source())
    print('ESP32 configurado:', cfg.data['esp']['host'], cfg.data['esp']['port'])
    print('Setup do notebook:', cfg.vision.source_name)
    print('Modelo chama:', cfg.vision.fire_model or 'ausente')
    print('Modelo pessoa:', cfg.vision.person_model or 'ausente')
    print('Calibração física de 12 pontos:', 'completa' if runtime.camera_calibration_complete(cfg.vision.calibration) else 'pendente no runtime')
    print('Ajuste config.json no VS Code se quiser trocar câmera, setup, modelos ou IP.')
    return cfg


def stage_unit_tests():
    suite = unittest.TestSuite()
    for case in (NotebookSetupTests, ProtocolTests, KinematicsTests,
                 LinkTest, VisionSimulationTests, DeviceScannerTests, RuntimeControlTests):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
    outcome = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
    if not outcome.wasSuccessful():
        raise RuntimeError('Um ou mais testes automatizados falharam; veja o nome e o erro acima.')
    return 'aprovado'


def stage_camera(cfg, automatic):
    import cv2

    source = cfg.camera_source()
    if source == 'auto-ip':
        from runtime import find_service
        cam = cfg.data['camera']
        service = find_service('_rtsp._tcp.local.', cam.get('mdns_name', ''), seconds=2.0)
        source = ('rtsp://{}:{}{}'.format(service[0], service[1], cam.get('stream_path', '/live'))
                  if service else cam.get('fallback_source', ''))
    if not source and source != 0:
        print('Câmera IP ainda não localizada. Configure camera.source em config.json.')
        return None
    pause('Posicione a câmera diante da imagem impressa da chama.', automatic)
    cap = cv2.VideoCapture(source)
    try:
        if not cap.isOpened():
            print('Câmera não abriu:', source)
            return None
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        frame = None
        for _ in range(8):
            ok, image = cap.read()
            if ok and image is not None:
                frame = image
        if frame is None:
            print('Câmera abriu, porém nenhum quadro foi recebido.')
            return None
        folder = ROOT / 'output' / 'testes_pc'
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / 'quadro_camera.jpg'
        if not cv2.imwrite(str(path), frame):
            raise RuntimeError('Não foi possível salvar a amostra da câmera.')
        print('Quadro recebido: {} x {}. Amostra: {}'.format(frame.shape[1], frame.shape[0], path))
        return frame
    finally:
        cap.release()


def stage_vision(cfg, frame):
    if frame is None:
        print('Visão com câmera real adiada; a etapa sintética já rodou nos testes.')
        return 'sem_camera'
    from runtime import VisionEngine
    import cv2

    engine = VisionEngine(cfg.vision, cfg.data['vision'])
    result = engine.process(frame, time.monotonic(), 'auto', False, (0.0, 0.0))
    folder = ROOT / 'output' / 'testes_pc'
    cv2.imwrite(str(folder / 'visao_ao_vivo.jpg'), result.live)
    cv2.imwrite(str(folder / 'visao_simulada.jpg'), result.combat)
    print('Visão:', result.status, '| válida:', result.valid,
          '| pessoa:', result.person_detected, '| alvo:', result.target_valid)
    if engine.model_error:
        print('Falha ao carregar modelo:', engine.model_error)
    if not cfg.vision.person_model:
        print('Sem peso de pessoas; procure yolo11n.pt no checkout do projeto.')
    if not cfg.vision.fire_model:
        print('Sem peso de chama; procure fire_yolo_best.pt no checkout do projeto.')
    return result.status


def xbox_plot(samples, axes, buttons, mapping, title):
    import cv2
    import numpy as np

    canvas = np.full((730, 1280, 3), (34, 36, 39), dtype=np.uint8)
    cv2.putText(canvas, title, (25, 32), cv2.FONT_HERSHEY_SIMPLEX, .65, (255, 255, 255), 2)
    x0, x1, y0, y1 = 185, 1250, 70, 470
    cv2.rectangle(canvas, (x0, y0), (x1, y1), (90, 90, 90), 1)
    cv2.line(canvas, (x0, (y0 + y1) // 2), (x1, (y0 + y1) // 2), (90, 90, 90), 1)
    cv2.putText(canvas, '+1', (130, y0 + 6), cv2.FONT_HERSHEY_SIMPLEX, .5, (210, 210, 210), 1)
    cv2.putText(canvas, ' 0', (130, (y0 + y1) // 2 + 6), cv2.FONT_HERSHEY_SIMPLEX, .5, (210, 210, 210), 1)
    cv2.putText(canvas, '-1', (130, y1), cv2.FONT_HERSHEY_SIMPLEX, .5, (210, 210, 210), 1)
    palette = [(70, 210, 255), (100, 230, 100), (255, 150, 80), (220, 120, 230),
               (120, 180, 255), (160, 200, 100), (255, 120, 160), (220, 220, 120)]
    axis_names = {int(index): name for name, index in mapping.get('xbox_axes', {}).items()
                  if isinstance(index, int)}
    visible = min(axes, 8)
    for i in range(visible):
        color = palette[i]
        label = 'E{} {}'.format(i, axis_names.get(i, ''))
        cv2.putText(canvas, label[:18], (8, 91 + i * 42), cv2.FONT_HERSHEY_SIMPLEX, .43, color, 1)
        if samples:
            points = []
            newest = samples[-1]['t']
            for sample in samples:
                px = int(x1 - min(20.0, newest - sample['t']) * (x1 - x0) / 20.0)
                py = int((y0 + y1) / 2 - sample['axes'][i] * (y1 - y0) / 2)
                points.append((px, py))
            if len(points) > 1:
                cv2.polylines(canvas, [np.array(points, dtype=np.int32)], False, color, 2)
            cv2.putText(canvas, '{:+.2f}'.format(samples[-1]['axes'][i]),
                        (8, 108 + i * 42), cv2.FONT_HERSHEY_SIMPLEX, .43, color, 1)
    button_names = {int(index): name for name, index in mapping.get('xbox_buttons', {}).items()
                    if isinstance(index, int)}
    for i in range(min(buttons, 16)):
        col, row = i % 8, i // 8
        bx, by = 30 + col * 155, 515 + row * 77
        pressed = bool(samples and samples[-1]['buttons'][i])
        color = (60, 170, 90) if pressed else (78, 78, 78)
        cv2.rectangle(canvas, (bx, by), (bx + 142, by + 59), color, -1)
        cv2.putText(canvas, 'B{} {}'.format(i, button_names.get(i, ''))[:18], (bx + 5, by + 24),
                    cv2.FONT_HERSHEY_SIMPLEX, .38, (255, 255, 255), 1)
        cv2.putText(canvas, 'pressionado' if pressed else 'solto', (bx + 5, by + 47),
                    cv2.FONT_HERSHEY_SIMPLEX, .38, (255, 255, 255), 1)
    cv2.putText(canvas, 'Eixos: -1 a +1 | janela: 20 s | botao Q ou Esc encerra',
                (30, 703), cv2.FONT_HERSHEY_SIMPLEX, .51, (230, 230, 230), 1)
    return canvas


def stage_controller(cfg, seconds=20.0):
    import pygame
    import cv2

    pygame.init()
    pygame.joystick.init()
    window_open = False
    try:
        count = pygame.joystick.get_count()
        print('Controles detectados:', count)
        for index in range(count):
            joy = pygame.joystick.Joystick(index)
            joy.init()
            print('  {}: {} ({} eixos, {} botões)'.format(
                index, joy.get_name(), joy.get_numaxes(), joy.get_numbuttons()))
        if not count:
            return {'estado': 'ausente', 'controles': 0}
        selected = int(cfg.data.get('xbox_index', 0))
        if selected == -1:
            print('Controle desativado na configuração; entrada por teclado disponível no runtime.')
            return {'estado': 'desativado_teclado', 'controles': count}
        if selected < 0 or selected >= count:
            print('Controle configurado {} indisponível; revise xbox_index.'.format(selected))
            return {'estado': 'indice_indisponivel', 'controles': count}
        joy = pygame.joystick.Joystick(selected)
        joy.init()
        axes, buttons = joy.get_numaxes(), joy.get_numbuttons()
        folder = ROOT / 'output' / 'testes_pc'
        folder.mkdir(parents=True, exist_ok=True)
        csv_path, png_path = folder / 'xbox_sinais.csv', folder / 'xbox_sinais.png'
        gui = os.name == 'nt' or bool(os.environ.get('DISPLAY') or os.environ.get('WAYLAND_DISPLAY'))
        print('Mova cada joystick e pressione cada botão separadamente durante {:.0f} s.'.format(seconds))
        print('Mapa configurado:', cfg.data.get('xbox_axes'), cfg.data.get('xbox_buttons'))
        if not gui:
            print('Sem tela gráfica: captura CSV e imagem PNG, sem janela ao vivo.')
        samples = []
        start = time.monotonic()
        with csv_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            writer.writerow(['segundos'] + ['eixo_{}'.format(i) for i in range(axes)] +
                            ['botao_{}'.format(i) for i in range(buttons)])
            while time.monotonic() - start < seconds:
                pygame.event.pump()
                sample = {'t': time.monotonic() - start,
                          'axes': [round(joy.get_axis(i), 4) for i in range(axes)],
                          'buttons': [joy.get_button(i) for i in range(buttons)]}
                samples.append(sample)
                writer.writerow([round(sample['t'], 3)] + sample['axes'] + sample['buttons'])
                if gui and len(samples) % 2 == 0:
                    frame = xbox_plot(samples, axes, buttons, cfg.data, joy.get_name())
                    cv2.imshow('BLAZE - teste do controle Xbox', frame)
                    window_open = True
                    if cv2.waitKey(1) & 0xff in (27, ord('q')):
                        break
                time.sleep(1 / 30)
        if samples:
            frame = xbox_plot(samples, axes, buttons, cfg.data, joy.get_name())
            if not cv2.imwrite(str(png_path), frame):
                raise RuntimeError('Falha ao salvar gráfico do controle Xbox.')
        print('Sinais do controle:', csv_path, '| gráfico:', png_path)
        return {'estado': 'capturado', 'nome': joy.get_name(), 'eixos': axes,
                'botoes': buttons, 'amostras': len(samples), 'csv': str(csv_path),
                'grafico': str(png_path)}
    finally:
        if window_open:
            cv2.destroyWindow('BLAZE - teste do controle Xbox')
        pygame.joystick.quit()
        pygame.quit()


def stage_esp_link(cfg):
    from runtime import ESPConnection
    from runtime import control_message
    import threading

    settings = cfg.data['esp']
    message = control_message({'requested_mode': 'manual', 'stop': True},
                              {'valid': False, 'target_valid': False, 'person_detected': False})
    received = []
    event = threading.Event()
    link = ESPConnection(str(settings['host']), int(settings['port']), lambda: message,
                         lambda data: (received.append(data), event.set()),
                         float(settings['reconnect_seconds']),
                         str(settings.get('transport', 'auto')),
                         str(settings.get('serial_port', 'auto')),
                         int(settings.get('baudrate', 115200)),
                         str(settings.get('bluetooth_port','auto')))
    link.start()
    try:
        event.wait(9)
        if not received:
            print('ESP32 indisponível por USB e/ou Wi-Fi; confira cabo de dados, driver e rede.')
            return None
        print('Telemetria do ESP32 por', link.active_transport, ':',
              received[-1].get('reason'), received[-1].get('mode'))
        return received[-1]
    finally:
        link.stop()


def stage_integrated(cfg, seconds=45):
    from runtime import CameraSource
    from runtime import OperatorInputs
    from runtime import ESPConnection
    from runtime import control_message
    from runtime import VisionWorker

    inputs = OperatorInputs(cfg.data['xbox_axes'], cfg.data['xbox_buttons'],
                            preferred_index=int(cfg.data.get('xbox_index', 0)),
                            speed_limit_rpm=float(cfg.data.get('control', {}).get('speed_limit_rpm', 80)),
                            calibrated_max_rpm=float(cfg.data['geometry'].get('output_max_rpm', 0)))
    cam = cfg.data['camera']
    camera = CameraSource(cfg.camera_source(), cam['reconnect_seconds'], cam.get('mdns_name', ''),
                          cam.get('stream_path', '/live'), cam.get('fallback_source', ''))
    vision = VisionWorker(camera, cfg.vision, cfg.data['vision'], inputs)
    received = []
    sent = []

    def command():
        value = control_message(inputs.snapshot(), vision.snapshot().state())
        sent.append({'t': round(time.monotonic() - start, 3), 'data': value})
        return value

    def telemetry(message):
        received.append({'t': round(time.monotonic() - start, 3), 'data': message})
        inputs.acknowledge_mode(str(message.get('mode', '')), bool(message.get('jet_virtual', False)))

    esp = cfg.data['esp']
    link = ESPConnection(str(esp['host']), int(esp['port']), command, telemetry,
                         float(esp['reconnect_seconds']), str(esp.get('transport', 'auto')),
                         str(esp.get('serial_port', 'auto')), int(esp.get('baudrate', 115200)),
                         str(esp.get('bluetooth_port','auto')),
                         str(cfg.data.get('_official',{}).get('firmware_signature','')),
                         preserve_wifi=(isinstance(cfg.data.get('vision',{}).get('camera_source'),str) and cfg.data['vision']['camera_source'].lower().startswith(('rtsp://','rtsps://','http://','https://'))))
    print('Ensaio real: controle Xbox e vídeo da câmera devem estar ligados.')
    print('Siga as instruções por fase. O jato permanece virtual. Duração:', seconds, 's.')
    start = time.monotonic()
    camera.start()
    vision.start()
    link.start()
    last_print = -5.
    phase_index = -1
    phases = [(0, 'Aguarde câmera e telemetria, deixe os joysticks neutros.'),
              (5, 'Pressione o botão configurado para REARMAR; mantenha os joysticks neutros.'),
              (10, 'Mova o joystick esquerdo: frente, trás e giros; depois centralize.'),
              (20, 'Mova o joystick direito: pan e tilt; depois use o botão de alternar modo.'),
              (30, 'Teste os botões configurados para arrefecimento, LEDs e jato virtual; verifique os estados.'),
              (36, 'Deixe o robô parado e apresente uma pessoa à câmera; observe o bloqueio.'),
              (41, 'Teste o botão configurado para PARAR e mantenha os comandos neutros.')]
    reconnect_ok = False
    disarmed_after_reconnect = False
    try:
        while time.monotonic() - start < seconds:
            inputs.poll()
            inputs.set_link(link.connected)
            elapsed = time.monotonic() - start
            while phase_index + 1 < len(phases) and elapsed >= phases[phase_index + 1][0]:
                phase_index += 1
                print('FASE {}: {}'.format(phase_index + 1, phases[phase_index][1]))
            if elapsed - last_print >= 5:
                recent = received[-1]['data'] if received else {}
                print('t={:.0f}s ESP={} câmera={} visão={} bloqueio={} E/D={}/{}'.format(
                    elapsed, link.connected, bool(vision.snapshot().frame_stamp),
                    vision.snapshot().valid, recent.get('reason', 'sem telemetria'),
                    recent.get('drive_left_commanded'), recent.get('drive_right_commanded')))
                last_print = elapsed
            time.sleep(.05)
        if received:
            print('Testando desconexão/reconexão TCP controlada; comandos locais serão zerados.')
            previous = len(received)
            link.stop()
            inputs.set_link(False)
            time.sleep(1.0)  # excede o timeout de 750 ms do firmware
            link.start()
            deadline = time.monotonic() + 5.0
            while time.monotonic() < deadline:
                inputs.poll()
                inputs.set_link(link.connected)
                if len(received) > previous:
                    reconnect_ok = True
                    disarmed_after_reconnect = not received[previous]['data'].get('armed', True)
                    break
                time.sleep(.05)
    finally:
        inputs.action('stop')
        link.stop()
        vision.stop()
        camera.stop()
    states = [row['data'] for row in received]
    commands = [row['data'] for row in sent]
    result = {
        'pacotes_enviados': len(sent), 'pacotes_recebidos': len(received),
        'enlace_real': bool(received),
        'camera_recente': any(cmd.get('vision_valid') for cmd in commands),
        'rearmado': any(msg.get('armed') for msg in states),
        'tracao_comandada': any(abs(float(msg.get('drive_left_commanded') or 0)) > .02 or
                                 abs(float(msg.get('drive_right_commanded') or 0)) > .02 for msg in states),
        'servo_comandado': any(abs(float(msg.get('servo_pan_commanded') or 90) - 90) > 2 or
                                abs(float(msg.get('servo_tilt_commanded') or 90) - 90) > 2 for msg in states),
        'modo_automatico': any(msg.get('mode') == 'auto' for msg in states),
        'sensor_interno_valido': any(msg.get('sensor_internal_valid') for msg in states),
        'sensor_externo_valido': any(msg.get('sensor_external_valid') for msg in states),
        'arrefecimento_comandado': any(msg.get('cool_internal_commanded') or
                                       msg.get('cool_external_commanded') for msg in states),
        'jato_virtual': any(msg.get('jet_virtual') for msg in states),
        'parada_operador': any(msg.get('reason') == 'operator_stop' for msg in states),
        'pessoa_bloqueada': any(msg.get('reason') == 'person_detected' for msg in states),
        'reconectou': reconnect_ok,
        'desarmado_apos_reconexao': disarmed_after_reconnect,
        'observacao': 'Atuação e velocidades físicas exigem confirmação visual/encoder; telemetria é comando.'}
    folder = ROOT / 'output' / 'testes_pc'
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'integrado_pacotes.json').write_text(
        json.dumps({'enviados': sent, 'recebidos': received}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Resumo do teste integrado:', json.dumps(result, ensure_ascii=False))
    return result


test = types.SimpleNamespace(pause=pause, check_structure=check_structure, stage_config=stage_config, stage_unit_tests=stage_unit_tests, stage_camera=stage_camera, stage_vision=stage_vision, xbox_plot=xbox_plot, stage_controller=stage_controller, stage_esp_link=stage_esp_link, stage_integrated=stage_integrated)
test.check_structure()

# Setups editáveis e combinação oficial usados pelos dois notebooks.
import copy, datetime as _setup_datetime, html, re
SETUPS_DIR = ROOT / 'pc_operador' / 'parameters_setups'
SETUP_SECTIONS = ('connection', 'vision', 'control', 'hardware')

def _setup_name(name):
    name = str(name).strip()
    if not name or any(ord(ch)<32 for ch in name) or '/' in name or '\\' in name:
        raise ValueError('Informe um nome sem barras ou caracteres de controle.')
    return name

def _setup_path(section):
    if section not in (*SETUP_SECTIONS, 'combinations'):
        raise ValueError('Tipo de setup desconhecido.')
    return SETUPS_DIR / 'user' / (section + '.json')

def _json_atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def setup_all(section):
    path = _setup_path(section)
    if not path.is_file(): return {}
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, dict): raise ValueError(f'JSON de setups inválido: {path}')
    return data

def setup_names(section):
    return sorted(setup_all(section), key=str.casefold)

def setup_get(section, name):
    name = _setup_name(name)
    return copy.deepcopy(setup_all(section)[name])

def setup_save(section, name, payload):
    name = _setup_name(name)
    if section != 'combinations' and '*' in name:
        raise ValueError('O caractere * separa os grupos na combinação; escolha outro caractere para o nome do setup.')
    records = setup_all(section)
    payload = copy.deepcopy(payload)
    if section != 'combinations':
        payload['_meta'] = {'saved_at': _setup_datetime.datetime.now().astimezone().isoformat()}
    records[name] = payload
    _json_atomic(_setup_path(section), records)
    return name

def setup_delete(section, name):
    name = _setup_name(name)
    records = setup_all(section)
    if name not in records: raise KeyError(name)
    if section == 'combinations' and official_name() == name:
        raise ValueError('Este é o setup oficial. Ative outro antes de excluir.')
    for combo_name, combo in setup_all('combinations').items():
        if section in SETUP_SECTIONS and combo.get('sections', {}).get(section) == name:
            raise ValueError(f'Está em uso na combinação {combo_name!r}; edite ou exclua a combinação primeiro.')
    del records[name]
    _json_atomic(_setup_path(section), records)

def official_name():
    path = SETUPS_DIR / 'official' / 'active.json'
    if not path.is_file(): return ''
    return str(json.loads(path.read_text(encoding='utf-8')).get('_official', {}).get('name', ''))

def assemble_combination(selections):
    base_path = ROOT / 'pc_operador' / 'config.example.json'
    data = json.loads(base_path.read_text(encoding='utf-8'))
    for section in SETUP_SECTIONS:
        selected = selections.get(section, '')
        payload = (setup_get(section, selected) if selected else
                   json.loads((SETUPS_DIR/'defaults'/f'{section}.json').read_text(encoding='utf-8')))
        for key, value in payload.items():
            if key in ('_meta', 'saved_at'): continue
            if isinstance(value, dict) and isinstance(data.get(key), dict):
                data[key].update(copy.deepcopy(value))
            else:
                data[key] = copy.deepcopy(value)
    if selections.get('vision'):
        data.setdefault('camera', {})['source'] = data['vision'].get(
            'camera_source', data.get('camera', {}).get('source', 0))
    rpm = data.get('control', {}).get('speed_limit_rpm', 0)
    max_rpm = data.get('geometry', {}).get('output_max_rpm', 0)
    if not 10 <= float(rpm) <= 80:
        raise ValueError('O limite de velocidade deve ficar entre 10 e 80 RPM.')
    if float(max_rpm) < 0:
        raise ValueError('RPM máxima calibrada não pode ser negativa.')
    return data

def activate_combination(name):
    name = _setup_name(name)
    combo = setup_get('combinations', name)
    selections = combo['sections']
    data = assemble_combination(selections)
    data['_official'] = {'name': name, 'sections': selections,
                         'activated_at': _setup_datetime.datetime.now(_setup_datetime.timezone.utc).isoformat()}
    _json_atomic(SETUPS_DIR / 'official' / 'active.json', data)
    return data

def status_box(title, state, description):
    colors = {'OK': '#17803d', 'FALHA': '#b42318', 'PENDENTE': '#9a6700', 'TESTANDO': '#1e65aa'}
    color = colors.get(state, '#6b7280')
    return widgets.HTML('<div style="border:1px solid #d1d5db;border-radius:7px;padding:8px;margin:3px">'
                        f'<b style="color:{color};display:inline-block;width:78px">{html.escape(state)}</b>'
                        f'<strong>{html.escape(title)}</strong><span style="margin-left:10px;color:#444">'
                        f'{html.escape(description)}</span></div>')


## 2. Detectar conexões e salvar parâmetros locais

**Objetivo:** escolher e testar uma webcam USB ou um vídeo salvo, identificar a COM do ESP32 e definir o transporte de comandos. A imagem é adquirida pelo PC; o ESP32 recebe apenas comandos e devolve sensores.

1. Clique **Buscar conexões**, teste a webcam USB desejada ou escolha **Arquivo de vídeo** e informe o caminho completo.
2. Escolha **Rede local criada pelo ESP32** (`192.168.4.1`), **Somente USB** ou **Automática** (USB, depois rede local). Se usar USB, selecione a porta do ESP32 identificada ao conectar e desconectar o cabo.
3. Preencha a geometria medida ou deixe zero para medições ainda não feitas e salve o setup. O controle Xbox é configurado e testado na seção 5. A comunicação real é medida na seção 9, depois da gravação por USB na seção 8.


In [ ]:
import html, json, math, subprocess, sys, threading, time
from runtime import DeviceScanner, Probe, redacted_source

# Ao executar a célula novamente, encerra a busca e a câmera anteriores.
if 'scanner' in globals():
    scanner.stop()
if 'stop_camera_preview' in globals():
    stop_camera_preview()

config_path = ROOT / 'pc_operador' / 'config.json'
cfg = AppConfig(config_path)
FORM_SAVED = False
saved_source = cfg.data['camera'].get('source', 0)
saved_host = cfg.data['esp'].get('host', 'auto')
saved_serial = cfg.data['esp'].get('serial_port', 'auto')
saved_transport = cfg.data['esp'].get('transport', 'wifi_local')
if saved_transport not in ('auto','usb','wifi_local'): saved_transport='wifi_local'
saved_index = int(cfg.data.get('xbox_index', 0))

camera = widgets.Dropdown(description='Vídeo:', layout=widgets.Layout(width='650px'))
secondary_camera = widgets.Text(value=str(cfg.data['camera'].get('fallback_source', '')), description='Outra USB:', placeholder='índice USB opcional', layout=widgets.Layout(width='500px'))
camera_manual = widgets.Text(description='Vídeo salvo:', placeholder='Caminho do arquivo .mp4, .avi, .mov ou .mkv', layout=widgets.Layout(width='570px'))
transport = widgets.Dropdown(description='Comunicação:', options=[
    ('Rede local criada pelo ESP32 · 192.168.4.1', 'wifi_local'),
    ('Somente USB', 'usb'), ('Automática: USB, depois rede local do ESP32', 'auto')], value=saved_transport)
serial_port = widgets.Dropdown(description='Porta USB:', layout=widgets.Layout(width='650px'))
serial_manual = widgets.Text(description='COM manual:', placeholder='COM3 ou /dev/ttyUSB0', layout=widgets.Layout(width='650px'))
esp = widgets.Dropdown(description='ESP32 Wi-Fi:', layout=widgets.Layout(width='650px'))
controller = widgets.Dropdown(description='Controle:', layout=widgets.Layout(width='410px'))
width = widgets.FloatText(value=float(cfg.data['geometry']['track_width_m']), description='Esteiras (m):')
radius = widgets.FloatText(value=float(cfg.data['geometry']['sprocket_radius_m']), description='Raio (m):')
rpm = widgets.FloatText(value=float(cfg.data['geometry']['output_max_rpm']), description='Saída (RPM):')
scan_button = widgets.Button(description='Buscar conexões', button_style='info')
stop_button = widgets.Button(description='Interromper busca', button_style='warning', disabled=True)
save_button = widgets.Button(description='Salvar parâmetros', button_style='success')
stop_video_button = widgets.Button(description='Parar vídeo', button_style='warning', disabled=True)
feedback = widgets.Output()
camera_rows = widgets.VBox()
other_status = widgets.HTML()
video_state = widgets.HTML(value='<b>Vídeo:</b> selecione a webcam USB ou um arquivo e clique em Testar.')
video_image = widgets.Image(format='jpeg', layout=widgets.Layout(width='520px', height='300px', border='1px solid #888'))
video_image.layout.display = 'none'

camera_base = [(f'Webcam USB {i}', i) for i in range(6)] + [('Arquivo de vídeo', '__file__')]
if isinstance(saved_source, str):
    camera_manual.value = saved_source
    camera.options = camera_base
    camera.value = '__file__'
else:
    if saved_source not in [v for _, v in camera_base]:
        camera_base.append((f'Webcam USB {saved_source}', saved_source))
    camera.options = camera_base
    camera.value = saved_source
serial_base = [('Detectar porta USB compatível', 'auto'), ('Porta manual', '__manual__')]
if saved_serial not in ('auto', '__manual__'):
    serial_base.append((f'Porta salva: {saved_serial}', saved_serial))
serial_port.options = serial_base
serial_port.value = saved_serial
controller_base = [('Configurar Xbox na seção 5', saved_index)]
controller.options = controller_base
controller.value = saved_index
camera_found, serial_found, joystick_found = [], [], []
tasks = [Probe(f'Webcam USB {i}', 'camera', str(i), 3.0) for i in range(4)] + [
    Probe('Portas USB seriais', 'serial', '', 3.0)]
states = {t.label: ('aguardando', None) for t in tasks}
overall = 'pronta para buscar'

def normalized_video_source(raw):
    value = str(raw).strip()
    if value.lower().startswith(('rtsp://', 'rtsps://', 'http://', 'https://')):
        raise ValueError('Esta versão aceita apenas câmera USB ou arquivo de vídeo local.')
    return int(value) if value.isdigit() else value

# Subprocesso isolado: se o driver de vídeo travar, o kernel continua utilizável.
CAPTURE_SCRIPT = r'''
import base64, cv2, os, sys, time
raw = sys.argv[1]
encoded = sys.argv[2] if len(sys.argv) > 2 else raw
source = int(raw) if raw.isdigit() else raw
cap = None
first = None
selected = 0
backend_name = ''

if isinstance(source, int):
    backends = ([cv2.CAP_DSHOW, cv2.CAP_MSMF] if sys.platform == 'win32'
                else [cv2.CAP_ANY])
    for api in backends:
        candidate = cv2.VideoCapture(source, api)
        if candidate.isOpened():
            ok, frame = candidate.read()
            if ok and frame is not None:
                cap, first = candidate, frame
                backend_name = candidate.getBackendName()
                break
        candidate.release()
else:
    candidates = [raw] + ([encoded] if encoded != raw else [])
    is_rtsp = raw.lower().startswith(('rtsp://', 'rtsps://'))
    # Tenta o endereço literal antes de alterar caracteres da senha.
    transports = ['tcp', 'udp'] if is_rtsp else ['default']
    for transport in transports:
        for index, candidate_source in enumerate(candidates):
            print('INFO:Abrindo fonte ' + ('original' if index == 0 else 'codificada') +
                  ' por ' + transport.upper() + '…', flush=True)
            if is_rtsp:
                os.environ['OPENCV_FFMPEG_CAPTURE_OPTIONS'] = 'rtsp_transport;' + transport
            else:
                os.environ.pop('OPENCV_FFMPEG_CAPTURE_OPTIONS', None)
            params = [cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, 5000,
                      cv2.CAP_PROP_READ_TIMEOUT_MSEC, 5000]
            try:
                candidate = cv2.VideoCapture(candidate_source, cv2.CAP_FFMPEG, params)
            except (TypeError, cv2.error):
                candidate = cv2.VideoCapture(candidate_source, cv2.CAP_FFMPEG)
            if candidate.isOpened():
                ok, frame = candidate.read()
                if ok and frame is not None:
                    cap, first, selected = candidate, frame, index
                    backend_name = candidate.getBackendName() + '/' + transport.upper()
                    break
            candidate.release()
        if cap is not None:
            break

if cap is None:
    if isinstance(source, int):
        print('ERR:Webcam USB não abriu; confira a câmera e outros programas.', flush=True)
    else:
        print('ERR:Não foi possível abrir o arquivo de vídeo. Confira o caminho e o formato.', flush=True)
    sys.exit(1)

print('OPEN:' + str(selected) + ':' + backend_name, flush=True)
try:
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    frame = first
    while True:
        if frame is None:
            ok, frame = cap.read()
            if not ok or frame is None:
                print('ERR:Fonte aberta, mas não há mais quadros.', flush=True)
                break
        h, w = frame.shape[:2]
        if w > 640:
            frame = cv2.resize(frame, (640, max(1, round(h * 640 / w))))
        ok, jpeg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
        if ok:
            print('FRAME:' + base64.b64encode(jpeg).decode('ascii'), flush=True)
        frame = None
        time.sleep(0.08)
finally:
    cap.release()
'''

preview = {'process': None, 'generation': 0, 'last_frame': 0.0, 'raw_source': None, 'effective_source': None, 'backend': ''}

def stop_camera_preview(_=None):
    preview['generation'] += 1
    process = preview.get('process')
    preview['process'] = None
    if process is not None and process.poll() is None:
        process.kill()
    stop_video_button.disabled = True
    indicator = globals().get('video_state')
    if hasattr(indicator, 'value'):
        indicator.value = '<b>Vídeo:</b> parado. O último quadro permanece abaixo.'

def start_camera_preview(source):
    import base64
    stop_camera_preview()
    generation = preview['generation']
    preview['raw_source'] = str(source)
    preview['effective_source'] = None
    preview['backend'] = ''
    video_image.value = b''
    video_image.layout.display = 'block'
    video_state.value = '<b>Vídeo:</b> abrindo ' + html.escape(redacted_source(source)) + '…'
    stop_video_button.disabled = False
    try:
        process = subprocess.Popen(
            [sys.executable, '-u', '-c', CAPTURE_SCRIPT, str(source), str(normalized_video_source(source))],
            cwd=str(ROOT / 'pc_operador'), stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL, text=True, bufsize=1)
    except OSError as exc:
        stop_video_button.disabled = True
        video_state.value = '<b>Erro ao iniciar vídeo:</b> ' + html.escape(str(exc))
        return
    preview['process'] = process
    preview['last_frame'] = time.monotonic()

    def read_frames():
        had_error = False
        try:
            for raw in process.stdout:
                if preview['generation'] != generation:
                    break
                line = raw.strip()
                if line.startswith('INFO:'):
                    video_state.value = '<b>Vídeo:</b> ' + html.escape(line[5:])
                elif line.startswith('OPEN:'):
                    _, variant, backend = line.split(':', 2)
                    preview['backend'] = backend
                    preview['effective_source'] = (str(source) if variant == '0'
                        else str(normalized_video_source(source)))
                elif line.startswith('FRAME:'):
                    try:
                        video_image.value = base64.b64decode(line[6:], validate=True)
                        preview['last_frame'] = time.monotonic()
                        video_state.value = ('<b>Vídeo ao vivo:</b> ' + html.escape(redacted_source(source))
                                             + ' — ' + html.escape(preview['backend']))
                    except ValueError:
                        continue
                elif line.startswith('ERR:'):
                    had_error = True
                    video_state.value = '<b>Vídeo:</b> ' + html.escape(line[4:])
                    break
        finally:
            if preview['generation'] == generation:
                if process.poll() is None:
                    process.kill()
                preview['process'] = None
                stop_video_button.disabled = True
                if not video_image.value and not had_error:
                    video_state.value = '<b>Vídeo:</b> sem quadros. Teste outra câmera/porta.'

    def watch_timeout():
        while process.poll() is None and preview['generation'] == generation:
            limit = 50 if isinstance(source, str) and source.lower().startswith(('rtsp://', 'rtsps://')) else 8
            if time.monotonic() - preview['last_frame'] > limit:
                video_state.value = '<b>Vídeo:</b> prazo de abertura esgotado; confira a câmera USB ou o arquivo.'
                process.kill()
                return
            time.sleep(0.3)

    threading.Thread(target=read_frames, daemon=True).start()
    threading.Thread(target=watch_timeout, daemon=True).start()

camera_buttons = {}
def camera_button(source):
    key = str(source)
    if key not in camera_buttons:
        button = widgets.Button(description='Testar vídeo', button_style='info',
                                layout=widgets.Layout(width='125px'))
        button.on_click(lambda _, source=source: start_camera_preview(source))
        camera_buttons[key] = button
    return camera_buttons[key]

def test_manual(_):
    source = camera_manual.value.strip()
    if source and not source.lower().startswith(('rtsp://','rtsps://','http://','https://')):
        from pathlib import Path
        if not Path(source).expanduser().is_file():
            video_state.value = '<b>Vídeo:</b> arquivo não encontrado. Informe o caminho completo.'
            return
        camera.value = '__file__'
        start_camera_preview(source)
    else:
        video_state.value = '<b>Vídeo:</b> informe o caminho de um vídeo local primeiro.'

manual_button = widgets.Button(description='Testar vídeo salvo', button_style='info',
                               layout=widgets.Layout(width='180px'))
manual_button.on_click(test_manual)
stop_video_button.on_click(stop_camera_preview)

def update_choices(dropdown, base, found):
    old = dropdown.value
    merged = list(base)
    known = {str(v) for _, v in merged}
    for label, value in found:
        if str(value) not in known:
            merged.append((label, value))
            known.add(str(value))
        else:
            merged = [(label, v) if str(v) == str(value) else (l, v) for l, v in merged]
    dropdown.options = merged
    if old in [v for _, v in merged]:
        dropdown.value = old

def render_status():
    colors = {'detectado': '#176b3d', 'ausente': '#697586', 'buscando': '#a25c00',
              'erro': '#ab2929', 'tempo esgotado': '#ab2929',
              'aguardando': '#697586', 'interrompido': '#697586'}
    rows = []
    for index in range(4):
        name = f'Webcam USB {index}'
        state, detail = states[name]
        label = widgets.HTML(
            f'<span style="display:inline-block;width:180px">{name}</span>'
            f'<span style="display:inline-block;width:120px;color:{colors.get(state, "#333")}">{state}</span>'
            f'<span>{"câmera acessível" if isinstance(detail, dict) and detail.get("available") else ""}</span>',
            layout=widgets.Layout(width='510px'))
        rows.append(widgets.HBox([label, camera_button(index)]))
    camera_rows.children = tuple(rows)
    others = []
    for name in ('Portas USB seriais',):
        state, detail = states[name]
        if isinstance(detail, list) and name == 'Portas USB seriais':
            description = ', '.join(f"{x['device']}: {x['description']}" for x in detail)
        elif isinstance(detail, list) and name == 'Controles Xbox/USB':
            description = ', '.join(f"{x['index']}: {x['name']}" for x in detail)
        elif isinstance(detail, list):
            description = ', '.join(f'{x[0]} ({x[1]})' for x in detail)
        else:
            description = str(detail or '')
        others.append('<tr><td style="padding:4px 12px 4px 0">' + html.escape(name) + '</td>'
                      '<td style="padding-right:12px;color:' + colors.get(state, '#333') + '">' +
                      html.escape(state) + '</td><td>' + html.escape(description) + '</td></tr>')
    other_status.value = '<b>Busca: ' + html.escape(overall) + '</b><table>' + ''.join(others) + '</table>'

def on_scan_update(label, state, detail):
    global overall
    if label == '__overall__':
        overall = state
        if state == 'interrompida':
            states.update({k: ('interrompido', None) for k, (v, _) in states.items()
                           if v in ('buscando', 'aguardando')})
        stop_button.disabled = True
        scan_button.disabled = False
    else:
        states[label] = (state, detail)
        if state == 'detectado':
            if label.startswith('Webcam USB '):
                index = int(label.split()[-1])
                camera_found.append((f'Webcam USB {index} (detectada)', index))
                update_choices(camera, camera_base, camera_found)
            elif label == 'Portas USB seriais':
                serial_found.extend((f"{item['device']}: {item['description']}", item['device'])
                                    for item in detail)
                update_choices(serial_port, serial_base, serial_found)
                # COM1 Nuvoton é normalmente do PC; prioriza a ponte CP210x do ESP32.
                if serial_port.value == 'auto':
                    candidate = next((item['device'] for item in detail
                                      if 'cp210' in item['description'].lower()
                                      or 'silicon labs' in item['description'].lower()), None)
                    if candidate:
                        serial_port.value = candidate
    render_status()

scanner = DeviceScanner(ROOT / 'pc_operador', on_scan_update)
def start_scan(_=None):
    global overall
    if scanner.running:
        return
    camera_found.clear(); serial_found.clear()
    states.update({t.label: ('aguardando', None) for t in tasks})
    overall = 'buscando; o formulário continua disponível'
    scan_button.disabled = True
    stop_button.disabled = False
    render_status()
    scanner.start(tasks)

def stop_scan(_):
    global overall
    scanner.stop()
    overall = 'interrompendo a sonda ativa…'
    render_status()

def save_pc(_):
    global cfg, FORM_SAVED
    with feedback:
        feedback.clear_output()
        chosen_source = camera_manual.value.strip() if camera.value == '__file__' else camera.value
        chosen_host = '192.168.4.1'
        chosen_serial = serial_manual.value.strip() if serial_port.value == '__manual__' else serial_port.value
        if not chosen_source and chosen_source != 0 or (serial_port.value == '__manual__' and not chosen_serial):
            print('Informe o caminho do vídeo ou a porta COM manual selecionada.')
            return
        dimensions = [width.value, radius.value, rpm.value]
        if not all(math.isfinite(v) and v >= 0 for v in dimensions):
            print('Use valores não negativos; deixe zero apenas o parâmetro ainda não medido.')
            return
        data = json.loads(config_path.read_text(encoding='utf-8')) if config_path.exists() else {}
        if camera.value == '__file__':
            from pathlib import Path
            if not Path(str(chosen_source)).expanduser().is_file():
                print('Arquivo de vídeo não encontrado:', chosen_source)
                return
            if str(chosen_source).lower().startswith(('rtsp://','rtsps://','http://','https://')):
                print('Apenas arquivos locais e câmeras USB são aceitos.')
                return
        if camera.value == '__file__':
            chosen_source = (preview['effective_source'] if
                preview['raw_source'] == str(chosen_source) and preview['effective_source'] is not None
                else chosen_source)
        chosen_source = int(chosen_source) if str(chosen_source).isdigit() else chosen_source
        data.setdefault('camera', {})['source'] = chosen_source
        alternate = secondary_camera.value.strip()
        data['camera']['fallback_source'] = int(alternate) if alternate.isdigit() else alternate
        data.setdefault('esp', {}).update({'host': chosen_host or 'auto', 'transport': transport.value,
                                           'serial_port': chosen_serial or 'auto'})
        data['xbox_index'] = controller.value
        data.setdefault('geometry', {}).update(dict(zip(
            ('track_width_m', 'sprocket_radius_m', 'output_max_rpm'), dimensions)))
        temporary = config_path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
        temporary.replace(config_path)
        cfg = AppConfig(config_path)
        FORM_SAVED = True
        print('Salvo em:', config_path)
        print('Vídeo:', redacted_source(cfg.camera_source()), '| ESP32:', transport.value, chosen_serial,
              chosen_host, '| controle:', controller.value)
        print('Mapeie os eixos e botões na seção 5; ela possui editor próprio.')

def field_with_help(control, explanation):
    return widgets.HBox([
        control,
        widgets.HTML(
            '<span style="color:#505c69;font-size:13px">' + explanation + '</span>',
            layout=widgets.Layout(width='520px')
        )
    ], layout=widgets.Layout(align_items='center'))

connection_setup_pick = widgets.Dropdown(options=[('Novo setup', '')] + [(n,n) for n in setup_names('connection')], description='Salvos:')
connection_setup_name = widgets.Text(value='ESP32', description='Nome do setup:', placeholder='Ex.: ESP32, CP2102', layout=widgets.Layout(width='330px'))
connection_setup_load = widgets.Button(description='Carregar para editar', button_style='info')
connection_setup_delete = widgets.Button(description='Excluir selecionado', button_style='danger')

def _choose_widget(control, selected, label):
    values = [v for _, v in control.options]
    if selected not in values:
        control.options = list(control.options) + [(label, selected)]
    control.value = selected

def load_connection_setup(_):
    with feedback:
        feedback.clear_output()
        try:
            name = connection_setup_pick.value
            if not name: raise ValueError('Escolha um setup salvo.')
            data = setup_get('connection', name)
            connection_setup_name.value = name
            cam = data['camera']
            source = cam.get('source', 0)
            _choose_widget(camera, source if isinstance(source,int) else '__file__', f'Fonte salva: {redacted_source(source)}')
            camera_manual.value = str(source) if isinstance(source,str) else ''
            secondary_camera.value = str(cam.get('fallback_source', ''))
            transport.value = data['esp'].get('transport', 'wifi_local') if data['esp'].get('transport') in ('usb','auto','wifi_local') else 'wifi_local'
            ser = data['esp'].get('serial_port', 'auto')
            host = data['esp'].get('host', 'auto')
            _choose_widget(serial_port, ser, f'Porta salva: {ser}')
            
            _choose_widget(controller, int(data.get('xbox_index', 0)), 'Controle salvo')
            width.value = float(data['geometry']['track_width_m'])
            radius.value = float(data['geometry']['sprocket_radius_m'])
            rpm.value = float(data['geometry']['output_max_rpm'])
            print('Campos carregados para edição:', name)
        except (KeyError, ValueError, OSError) as exc:
            print('Falha ao carregar:', exc)

def refresh_connection_setups(selected=''):
    connection_setup_pick.options = [('Novo setup', '')] + [(n,n) for n in setup_names('connection')]
    connection_setup_pick.value = selected if selected in setup_names('connection') else ''

def delete_connection_setup(_):
    with feedback:
        feedback.clear_output()
        try:
            setup_delete('connection', connection_setup_pick.value)
            refresh_connection_setups()
            print('Setup de conexão excluído. A seleção ativa nos campos não foi alterada.')
        except (ValueError, KeyError, OSError) as exc: print('Falha ao excluir:', exc)

def save_connection_setup(_):
    # Só persiste os campos se a validação da célula 2 tiver sido aprovada.
    global FORM_SAVED
    FORM_SAVED = False
    save_pc(None)
    if not FORM_SAVED: return
    try:
        name = setup_save('connection', connection_setup_name.value,
                          {'camera': {'source': cfg.camera_source(), 'fallback_source': cfg.data['camera'].get('fallback_source', '')},
                           'esp': dict(cfg.data['esp']), 'xbox_index': int(cfg.data['xbox_index']),
                           'geometry': dict(cfg.data['geometry'])})
        refresh_connection_setups(name)
        with feedback: print('Setup editável salvo:', name)
    except (ValueError, OSError) as exc:
        FORM_SAVED = False
        with feedback: print('Falha ao salvar setup:', exc)

connection_setup_load.on_click(load_connection_setup)
connection_setup_delete.on_click(delete_connection_setup)
scan_button.on_click(start_scan)
stop_button.on_click(stop_scan)
save_button.on_click(save_connection_setup)
render_status()
display(widgets.VBox([
    widgets.HTML('<h4>Conexões e fonte de vídeo</h4><p>Busque a câmera USB e a porta serial. Clique em Testar ao lado da câmera ou selecione um arquivo de vídeo e teste-o. O controle Xbox será testado na seção 5.</p>'),
    widgets.HBox([scan_button, stop_button]),
    camera_rows, other_status,
    widgets.HTML('<h4>Vídeo do operador</h4>'),
    field_with_help(camera, 'Escolha a webcam USB ou Arquivo de vídeo.'),
    field_with_help(widgets.HBox([camera_manual, manual_button]), 'Para arquivo salvo, informe o caminho completo e confira a prévia ao lado.'),
    video_state, widgets.HBox([stop_video_button]), video_image,
    field_with_help(secondary_camera, 'Fonte alternativa opcional do runtime; deixe vazia se houver só uma câmera.'),
    widgets.HTML('<h4>Comunicação PC ↔ ESP32</h4><p>A rede Wi-Fi é criada pelo ESP32 no endereço fixo 192.168.4.1. A primeira gravação exige USB. No modo automático, tenta USB e depois a rede local.</p>'),
    field_with_help(transport, 'Rede local do ESP32, USB ou automático. Os comandos e sensores usam esta conexão; o vídeo é capturado diretamente pelo PC.'),
    field_with_help(serial_port, 'Escolha a porta que aparece ao conectar o ESP32, geralmente Silicon Labs CP210x; '
                    'confirme no Gerenciador de Dispositivos. O número COM muda entre PCs.'),
    field_with_help(serial_manual, 'Preencha somente se escolher “Porta manual” acima '
                    'e souber a COM correta pelo Gerenciador de Dispositivos.'),
    widgets.HTML('<h4>Geometria</h4><p>A configuração do controle Xbox e seu teste estão na seção 5.</p>'),
    field_with_help(width, 'Distância entre os centros das esteiras, em metros. '
                    'Deixe 0 até medir.'),
    field_with_help(radius, 'Raio efetivo da roda motriz, em metros. Deixe 0 até medir.'),
    field_with_help(rpm, 'Rotação máxima na saída do redutor, em RPM. Deixe 0 até medir.'),
    widgets.HTML('<h4>Setups de conexão</h4><p>Carregue um registro para editar os campos acima. Salvar com o mesmo nome atualiza o registro; excluir remove-o. Um setup só vira oficial na última célula.</p>'),
    widgets.HBox([connection_setup_pick, connection_setup_name]),
    widgets.HBox([connection_setup_load, connection_setup_delete]),
    widgets.HBox([save_button,
                  widgets.HTML('<span style="margin-left:12px">Grava o rascunho em config.json e o setup nomeado em parameters_setups/user.</span>')]),
    feedback
]))
start_scan()


## 3. Lógica, segurança e protocolo

Execute esta célula sem motores, câmera ou ESP32 conectados. As 18 verificações mostram **OK**, **FALHA** ou **PENDENTE** ao lado do que conferem: leitura do setup RGB, comandos e telemetria, parada/rearme após perda do enlace, cinemática, alternativas USB/Wi-Fi, rota virtual e busca de dispositivos. O teste de teclado simula `W` com o joystick desativado só durante a verificação, para que um controle conectado não substitua a tecla.

Uma falha abre os detalhes daquela verificação abaixo das caixas. O exemplo de movimento no final é apenas uma estimativa quando geometria e RPM calibradas estiverem disponíveis; este ensaio não aciona o robô.


In [ ]:
# Seção 3 — lógica, segurança e protocolo
# Descobre os testes existentes na célula de definições, evitando nomes obsoletos.
import contextlib
import html
import io
import unittest

import ipywidgets as widgets
from IPython.display import display

if "results" not in globals():
    results = {}

GRUPOS = [
    ("Configuração", "NotebookSetupTests"),
    ("Protocolo", "ProtocolTests"),
    ("Cinemática e segurança", "KinematicsTests"),
    ("Perfis de motores", "ActuatorProfileTests"),
    ("Comunicação USB/Wi-Fi", "LinkTest"),
    ("Visão e segurança", "VisionSimulationTests"),
    ("Controle e RPM", "RuntimeControlTests"),
    ("Busca de dispositivos", "DeviceScannerTests"),
]

DESCRICOES = {
    "test_selected_motor_models_define_isolated_firmware_backend":
        "A escolha BLDC/HT23 gera firmware isolado com saídas de tração incompatíveis bloqueadas.",
    "test_default_combination_uses_no_saved_section":
        "Monta uma combinação completa somente com os quatro grupos default e sem entradas legadas.",
    "test_both_dc_motor_options_use_bts_enable_pins":
        "Confere dois BTS7960 e pinos EN nos perfis DC 3–6 V e Mabuchi 12 V.",
    "test_real_profile_rejects_wrong_traction_driver":
        "Impede gravar um driver incompatível com o par de BTS7960 do perfil físico.",
    "test_setup_and_twelve_point_calibration_are_read_per_combination":
        "Lê o setup de visão e a calibração física de 12 pontos vinculada à combinação completa.",
    "test_manual_command_does_not_invent_a_valid_camera":
        "Um comando manual não declara a câmera válida sem receber um quadro.",
    "test_telemetry_requires_expected_type":
        "Rejeita mensagens que não tenham o formato de telemetria esperado.",
    "test_link_loss_zeros_local_control_and_demands_rearm":
        "Zera a tração ao perder a comunicação e exige novo rearme.",
    "test_straight_and_turn":
        "Confere avanço, giro e velocidade lateral do modelo diferencial.",
    "test_without_calibration_no_speed_is_invented":
        "Não estima RPM quando a velocidade máxima não foi calibrada.",
    "test_auto_falls_back_to_wifi_when_usb_cannot_open":
        "Tenta Wi-Fi quando a porta USB não pode ser aberta.",
    "test_auto_uses_usb_telemetry_without_opening_wifi":
        "Mantém a USB quando ela fornece telemetria válida.",
    "test_control_goes_out_and_telemetry_returns":
        "Confere o envio de comando e o recebimento de telemetria.",
    "test_calibration_interpolates_twelve_points_and_camera_angle":
        "Interpola os quatro cantos e os ângulos da câmera entre os 12 registros.",
    "test_fire_boxes_respect_area_and_do_not_overlap":
        "Limita a área das caixas pequenas de fogo e impede sobreposição.",
    "test_person_blocks_jet_and_clears_trail":
        "Bloqueia o jato ao detectar pessoa e limpa o rastro virtual.",
    "test_missing_models_does_not_validate_vision":
        "Não declara a visão válida quando faltam modelos obrigatórios.",
    "test_manual_camera_and_cannon_reciprocal_following":
        "Confere que canhão e plataforma se acompanham na margem lateral do campo de 120°.",
    "test_rpm_limit_caps_both_track_commands":
        "Respeita o teto de RPM definido para as duas esteiras.",
    "test_uncalibrated_rpm_blocks_traction":
        "Bloqueia a tração sem RPM máxima calibrada.",
    "test_scan_reports_result_and_completes":
        "Conclui a busca de dispositivos e publica o resultado.",
    "test_stop_kills_active_probe":
        "Interrompe uma busca de dispositivo ainda em execução.",
}

if not globals().get("FORM_SAVED", False):
    results["logica"] = "pendente"
    display(status_box(
        "Pré-requisito da seção 3",
        "PENDENTE",
        "Volte à seção 2 e salve os parâmetros de conexão antes de executar os testes."
    ))
else:
    caixas = []
    detalhes = {}
    contagem = {"OK": 0, "FALHA": 0, "PENDENTE": 0}

    for nome_grupo, nome_classe in GRUPOS:
        classe = globals().get(nome_classe)

        if classe is None:
            contagem["PENDENTE"] += 1
            caixas.append(status_box(
                nome_grupo, "PENDENTE",
                f"A classe {nome_classe} não foi definida. Reexecute a célula de funções da seção 1."
            ))
            continue

        metodos = unittest.defaultTestLoader.getTestCaseNames(classe)

        if not metodos:
            contagem["PENDENTE"] += 1
            caixas.append(status_box(
                nome_grupo, "PENDENTE", "Nenhum método de teste foi encontrado."
            ))
            continue

        for metodo in metodos:
            titulo = metodo.removeprefix("test_").replace("_", " ")
            descricao = DESCRICOES.get(
                metodo,
                f"Verifica a função «{titulo}» no grupo {nome_grupo.lower()}."
            )

            log = io.StringIO()
            try:
                with contextlib.redirect_stdout(log):
                    resultado = unittest.TextTestRunner(
                        stream=log, verbosity=1
                    ).run(unittest.TestSuite([classe(metodo)]))

                if resultado.errors or resultado.failures:
                    estado = "FALHA"
                elif resultado.skipped:
                    estado = "PENDENTE"
                else:
                    estado = "OK"

                texto = log.getvalue().strip()
            except Exception as exc:
                estado = "FALHA"
                texto = f"{type(exc).__name__}: {exc}"

            contagem[estado] += 1
            detalhes[f"{nome_classe}.{metodo}"] = {
                "estado": estado,
                "descricao": descricao,
                "log": texto,
            }

            cartao = status_box(f"{nome_grupo}: {titulo}", estado, descricao)
            if estado == "FALHA":
                explicacao = widgets.HTML(
                    "<details style='margin:0 8px 10px 90px'>"
                    "<summary>Ver detalhe da falha</summary>"
                    "<pre style='white-space:pre-wrap;max-height:240px;overflow:auto'>"
                    + html.escape(texto)
                    + "</pre></details>"
                )
                caixas.append(widgets.VBox([cartao, explicacao]))
            else:
                caixas.append(cartao)

    results["logica_detalhes"] = detalhes
    results["logica"] = (
        "aprovado" if contagem["FALHA"] == 0 and contagem["PENDENTE"] == 0
        else "falha" if contagem["FALHA"]
        else "pendente"
    )

    resumo = (
        f"{contagem['OK']} OK · {contagem['FALHA']} falha(s) · "
        f"{contagem['PENDENTE']} pendente(s). "
        "Os testes desta seção não acionam o ESP32 nem os motores."
    )
    estado_resumo = (
        "FALHA" if contagem["FALHA"] else
        "PENDENTE" if contagem["PENDENTE"] else
        "OK"
    )

    display(widgets.VBox([
        status_box("Resultado da seção 3", estado_resumo, resumo),
        *caixas,
    ], layout=widgets.Layout(width="100%")))

    exemplo = estimate(0.3, 0.5, cfg.data["geometry"])
    if exemplo is None:
        display(status_box(
            "Exemplo cinemático", "PENDENTE",
            "Informe geometria e RPM máxima calibrada para calcular as velocidades."
        ))
    else:
        display(status_box(
            "Exemplo cinemático", "OK",
            f"Comandos E/D = 0,3/0,5 → "
            f"RPM estimadas E/D = "
            f"{exemplo['left_rpm_est']:.1f}/{exemplo['right_rpm_est']:.1f}; "
            f"avanço = {exemplo['vx_body_mps_est']:.3f} m/s; "
            f"giro = {exemplo['yaw_rad_s_est']:.3f} rad/s."
        ))

## 4. Visão RGB, segurança e jato virtual

Escolha **Câmera USB** ou **Arquivo de vídeo** e teste a imagem nesta seção. Se usar arquivo, coloque-o na pasta de vídeos indicada na interface. Selecione os modelos obrigatórios de fogo e pessoas, e opcionalmente um modelo de bombeiros. A inferência e o controle da mira seguem o modo simulação ou combate real; o ensaio nesta célula não abre a válvula física. Ajuste FPS e parâmetros, execute a visão e salve um setup para a combinação completa. A calibração física de câmera e canhão acontece no runtime depois da gravação do ESP32.


In [ ]:
# SEÇÃO 4 — substitua a célula inteira por este código.
import cv2, numpy as np, ipywidgets as widgets
import html, json, math, os, shutil, subprocess, sys, threading, time
from datetime import datetime as _vision_datetime
from pathlib import Path
from IPython.display import display

PC_DIR = ROOT / 'pc_operador'
DATA_DIR, MODELS_DIR, RUNS_DIR = (PC_DIR / name for name in ('data', 'models', 'runs'))
VIDEO_DIR = PC_DIR / 'videos_input'
LEGACY_DIR = PC_DIR / 'parameters_setups' / 'user'
VISION_FILE = LEGACY_DIR / 'rgb_monocular_fire_monitor_setups.json'
EXPORT_DIR = ROOT / 'output' / 'visao'
for folder in (DATA_DIR, MODELS_DIR, RUNS_DIR, VIDEO_DIR, LEGACY_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# São exemplos de estrutura, não datasets anotados. Nenhum treinamento começa automaticamente.
for kind, class_name in (('fire', 'fire'), ('person', 'person'), ('firefighter', 'firefighter')):
    base = DATA_DIR / kind
    for split in ('train', 'val'):
        for sub in ('images', 'labels'):
            (base / sub / split).mkdir(parents=True, exist_ok=True)
    template = base / 'dataset.yaml'
    if not template.exists():
        template.write_text(
            'path: .\ntrain: images/train\nval: images/val\nnames:\n  0: ' + class_name +
            ('\n  1: helmet\n  2: protective_clothing' if kind == 'firefighter' else '') + '\n',
            encoding='utf-8')

def note(control, message):
    return widgets.HBox([control, widgets.HTML(html.escape(message),
                         layout=widgets.Layout(width='560px'))])

def status(label, state, message):
    return status_box(label, state, message).value

def within_package(path):
    try:
        return str(Path(path).resolve().relative_to(PC_DIR.resolve()))
    except ValueError:
        return str(Path(path).resolve())

def camera_value(value):
    value = str(value).strip()
    return int(value) if value.isdecimal() else value

def choices(paths, label):
    return [(within_package(path), str(path.resolve())) for path in sorted(set(paths))]

def model_paths():
    return sorted(MODELS_DIR.glob('*.pt'))

def data_paths():
    return sorted(set(DATA_DIR.rglob('*.yaml')) | set(DATA_DIR.rglob('*.yml')))

def read_legacy():
    try:
        result = json.loads(VISION_FILE.read_text(encoding='utf-8'))
        return result if isinstance(result, dict) else {}
    except (OSError, ValueError):
        return {}

def refresh_dropdown(dropdown, values, preferred=None):
    old = preferred if preferred is not None else dropdown.value
    dropdown.options = [('Selecione...', '')] + values
    allowed = [value for _, value in dropdown.options]
    dropdown.value = old if old in allowed else ''

def resolved_path(dropdown, manual=''):
    value = manual.strip() if manual else dropdown.value
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = PC_DIR / path
    if not path.is_file():
        basename = str(value).replace('\\', '/').split('/')[-1]
        for folder in (VIDEO_DIR, MODELS_DIR):
            candidate = folder / basename
            if candidate.is_file():
                return candidate.resolve()
    return path.resolve()

def open_capture(source, stop, mode='camera', requested_fps=None):
    backends = ([('DirectShow', cv2.CAP_DSHOW), ('Media Foundation', cv2.CAP_MSMF),
                 ('automático', cv2.CAP_ANY)] if sys.platform == 'win32' and
                mode == 'camera' and isinstance(source, int) else [('automático', cv2.CAP_ANY)])
    errors = []
    for backend_name, backend in backends:
        if stop.is_set():
            break
        cap = None
        try:
            cap = (cv2.VideoCapture(source, backend, _blaze_stop=stop)
                   if isinstance(source,str) and source.lower().startswith(('rtsp://','rtsps://'))
                   else cv2.VideoCapture(source, backend))
            if cap.isOpened():
                if requested_fps and mode == 'camera':
                    cap.set(cv2.CAP_PROP_FPS, float(requested_fps))
                deadline = time.monotonic() + 3
                while time.monotonic() < deadline and not stop.is_set():
                    ok, sample = cap.read()
                    if ok and sample is not None and sample.size:
                        return cap, sample, backend_name, errors
            detail=getattr(cap,'error','') if cap is not None else ''
            errors.append(f'{backend_name}: {detail or "não forneceu quadros"}')
        except Exception as exc:
            errors.append(f'{backend_name}: {type(exc).__name__}: {exc}')
        finally:
            if cap is not None and (not cap.isOpened() or errors and
                                    errors[-1].startswith(backend_name + ':')):
                cap.release()
    return None, None, None, errors

def dataset_report(path, kind):
    if path is None or not path.is_file():
        raise ValueError(f'Selecione o dataset YAML de {kind}.')
    import yaml
    specification = yaml.safe_load(path.read_text(encoding='utf-8'))
    if not isinstance(specification, dict):
        raise ValueError('YAML inválido.')
    names = specification.get('names', {})
    if isinstance(names, dict):
        class_names = [str(name).lower() for name in names.values()]
    elif isinstance(names, list):
        class_names = [str(name).lower() for name in names]
    else:
        raise ValueError('O YAML precisa declarar names.')
    terms = (('fire', 'flame', 'fogo', 'chama') if kind == 'fire' else
             ('person', 'pessoa', 'human') if kind == 'person' else
             ('firefighter', 'fireman', 'bombeiro', 'helmet', 'capacete', 'turnout', 'protective', 'ppe', 'epi'))
    if not any(any(word in name for word in terms) for name in class_names):
        raise ValueError(f'O dataset {kind} precisa de uma classe com nome {terms}.')
    root = Path(specification.get('path') or path.parent)
    if not root.is_absolute():
        root = (path.parent / root).resolve()
    counts = {}
    for split in ('train', 'val'):
        location = specification.get(split)
        if not location or isinstance(location, list):
            raise ValueError(f'Informe um diretório {split}: images/{split} no YAML.')
        image_dir = Path(location)
        if not image_dir.is_absolute():
            image_dir = root / image_dir
        image_dir = image_dir.resolve()
        images = [p for p in image_dir.rglob('*') if p.suffix.lower() in
                  ('.jpg', '.jpeg', '.png', '.bmp', '.webp')]
        label_dir = Path(str(image_dir).replace(os.sep + 'images' + os.sep,
                                                 os.sep + 'labels' + os.sep))
        labels = [p for p in label_dir.rglob('*.txt')] if label_dir.is_dir() else []
        if not images or not labels:
            raise ValueError(f'{split}: faltam imagens ou arquivos YOLO .txt em {image_dir} / {label_dir}.')
        counts[split] = (len(images), len(labels))
    return counts

def load_model(path, kind):
    if path is None or not path.is_file():
        raise ValueError(f'Selecione um modelo .pt já treinado para {kind}.')
    from ultralytics import YOLO
    model = YOLO(str(path))
    names = getattr(model, 'names', {})
    names = list(names.values()) if isinstance(names, dict) else list(names)
    names = [str(name).lower() for name in names]
    terms = (('fire', 'flame', 'fogo', 'chama') if kind == 'fire' else
             ('person', 'pessoa', 'human') if kind == 'person' else
             ('firefighter', 'fireman', 'bombeiro', 'helmet', 'capacete', 'turnout', 'protective', 'ppe', 'epi'))
    if not any(any(term in name for term in terms) for name in names):
        raise ValueError(f'{path.name}: classes {names} sem classe {kind}.')
    return model

def detections(model, image, confidence, image_size, words):
    names = model.names
    names = dict(enumerate(names)) if isinstance(names, list) else names
    ids = [int(key) for key, name in names.items() if
           any(word in str(name).lower() for word in words)]
    result = model.predict(image, conf=float(confidence), imgsz=int(image_size),
                           classes=ids, max_det=100, verbose=False)[0]
    output = []
    if result.boxes is not None:
        for box, score in zip(result.boxes.xyxy.cpu().numpy(),
                              result.boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = map(int, box)
            if x2 > x1 and y2 > y1:
                output.append({'box': (x1, y1, x2, y2), 'conf': float(score),
                               'center': ((x1+x2)/2, (y1+y2)/2),
                               'area': (x2-x1)*(y2-y1)})
    return output

def merge_overlapping_boxes(boxes, iou_threshold):
    """Agrupa caixas da mesma classe por sobreposição, com união transitiva."""
    pending=sorted(boxes,key=lambda item:item['conf'],reverse=True)
    merged=[]
    while pending:
        group=[pending.pop(0)]
        changed=True
        while changed:
            changed=False
            remaining=[]
            for candidate in pending:
                if any(box_iou(candidate['box'],other['box']) >= iou_threshold for other in group):
                    group.append(candidate)
                    changed=True
                else:remaining.append(candidate)
            pending=remaining
        x1=min(item['box'][0] for item in group)
        y1=min(item['box'][1] for item in group)
        x2=max(item['box'][2] for item in group)
        y2=max(item['box'][3] for item in group)
        merged.append({'box':(x1,y1,x2,y2),'conf':max(item['conf'] for item in group),
                       'center':((x1+x2)/2,(y1+y2)/2),'area':(x2-x1)*(y2-y1)})
    return merged

def box_iou(a,b):
    overlap=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return overlap/max(1,area_a+area_b-overlap)

from runtime import pixel_to_servos as runtime_pixel_to_servos

def pixel_to_servos(point, shape, calibration, params):
    # A janela de teste não aciona servos; a calibração física por 12 pontos
    # pertence ao runtime e usa a posição comandada da plataforma.
    return runtime_pixel_to_servos(point, shape, calibration, params, 90.)

class VisionSession:
    def __init__(self, fire_model, person_model, params, calibration, mode, source,
                 firefighter_model=None):
        self.fire_model, self.person_model = fire_model, person_model
        self.firefighter_model = firefighter_model
        self.params, self.calibration, self.mode, self.source = params, calibration, mode, source
        self.journal_path = EXPORT_DIR / f'trajetoria_{_vision_datetime.now():%Y%m%d_%H%M%S}.jsonl'
        self.journal = self.journal_path.open('w', encoding='utf-8')
        self.reset()

    def close(self):
        self.journal.close()

    def reset(self):
        self.fires, self.people, self.firefighters = [], [], []
        self.regions, self.route = [], []
        self.impact, self.last_step = None, None
        self.last_person, self.last_detection = -1e10, -1e10
        self.trail, self.visited_cells = [], set()
        self.target = None

    def _painted_frame(self, frame, now):
        canvas=frame.copy()
        if self.mode != 'simulation':return canvas
        lifetime=max(.1,float(self.params['restore_seconds']))
        radius=int(self.params['erase_radius_px'])
        self.trail=[point for point in self.trail if now-point[0]<lifetime]
        for instant,px,py in self.trail:
            alpha=max(0.,min(1.,1.-(now-instant)/lifetime))
            cx,cy=int(round(px)),int(round(py))
            x0,x1=max(0,cx-radius),min(canvas.shape[1],cx+radius+1)
            y0,y1=max(0,cy-radius),min(canvas.shape[0],cy+radius+1)
            if x0>=x1 or y0>=y1:continue
            roi=canvas[y0:y1,x0:x1]
            mask=np.zeros(roi.shape[:2],dtype=np.uint8)
            cv2.circle(mask,(cx-x0,cy-y0),radius,255,-1)
            roi[mask>0]=(roi[mask>0].astype(np.float32)*(1.-alpha)).astype(np.uint8)
        return canvas

    def _fire_mask(self, frame, raw, cell):
        """Refina as caixas YOLO com pixels quentes, mantendo fallback para chama atípica."""
        height,width=frame.shape[:2]
        mask=np.zeros((height,width),dtype=np.uint8)
        for fire in raw:
            x1,y1,x2,y2=fire['box']
            x1,y1=max(0,x1),max(0,y1)
            x2,y2=min(width,x2),min(height,y2)
            if x2<=x1 or y2<=y1:continue
            roi=frame[y1:y2,x1:x2]
            hsv=cv2.cvtColor(roi,cv2.COLOR_BGR2HSV)
            hue,sat,val=cv2.split(hsv)
            warm=(((hue<=42)|(hue>=170)) & (sat>=65) & (val>=115)).astype(np.uint8)*255
            # O branco muito luminoso só é aceito perto de pixels quentes.
            bright=((val>=205)&(sat<=95)).astype(np.uint8)*255
            nearby=cv2.dilate(warm,np.ones((max(3,cell//4),)*2,np.uint8))
            support=cv2.bitwise_or(warm,cv2.bitwise_and(bright,nearby))
            coverage=cv2.countNonZero(support)
            if coverage>=max(12,int(.008*roi.shape[0]*roi.shape[1])):
                kernel=np.ones((max(3,cell//6),)*2,np.uint8)
                support=cv2.morphologyEx(support,cv2.MORPH_CLOSE,kernel)
                support=cv2.dilate(support,np.ones((max(3,cell//5),)*2,np.uint8))
                mask[y1:y2,x1:x2]=cv2.bitwise_or(mask[y1:y2,x1:x2],support)
            else:
                # Mesmo no fallback, pixels ainda escuros pelo jato ficam de fora.
                visible=(np.max(roi,axis=2)>35).astype(np.uint8)*255
                mask[y1:y2,x1:x2]=cv2.bitwise_or(mask[y1:y2,x1:x2],visible)
        return mask

    def _small_boxes_and_regions(self, frame, raw):
        height,width=frame.shape[:2]
        cell=max(4,int(math.floor(math.sqrt(float(self.params['max_fire_box_area_px'])))))
        mask=self._fire_mask(frame,raw,cell)
        boxes=[]
        coverage=np.zeros((height,width),dtype=np.uint8)
        # Cada caixa fica contida em uma célula fixa: nunca há sobreposição.
        for y0 in range(0,height,cell):
            for x0 in range(0,width,cell):
                cropped=mask[y0:min(height,y0+cell),x0:min(width,x0+cell)]
                if not cv2.countNonZero(cropped):continue
                ys,xs=np.nonzero(cropped)
                x1,x2=x0+int(xs.min()),x0+int(xs.max())+1
                y1,y2=y0+int(ys.min()),y0+int(ys.max())+1
                if (x2-x1)*(y2-y1)>self.params['max_fire_box_area_px']:
                    # O ajuste abaixo também cobre limites de área não quadrados.
                    x2=min(x2,x1+cell)
                    y2=min(y2,y1+max(1,int(self.params['max_fire_box_area_px']//(x2-x1))))
                overlaps=[f['conf'] for f in raw if box_iou((x1,y1,x2,y2),f['box'])>0]
                conf=max(overlaps,default=max((f['conf'] for f in raw),default=0.))
                index=len(boxes)
                boxes.append({'box':(x1,y1,x2,y2),'conf':conf,
                              'center':((x1+x2)/2.,(y1+y2)/2.),
                              'area':(x2-x1)*(y2-y1),
                              'cell':(x0//cell,y0//cell),'region':None})
                coverage[y1:y2,x1:x2]=255
        if not boxes:return [],[]
        count,labels,stats,_=cv2.connectedComponentsWithStats(coverage,connectivity=8)
        regions=[]
        for component in range(1,count):
            if stats[component,cv2.CC_STAT_AREA]<2:continue
            ox,oy,ow,oh=map(int,stats[component,:4])
            part=np.where(labels[oy:oy+oh,ox:ox+ow]==component,255,0).astype(np.uint8)
            contours,_=cv2.findContours(part,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            if not contours:continue
            contour=max(contours,key=cv2.contourArea).copy()
            contour[:,0,0]+=ox
            contour[:,0,1]+=oy
            reduced=cv2.approxPolyDP(contour,max(1.,cv2.arcLength(contour,True)*.002),True)
            regions.append({'contour':contour,'export':reduced.reshape(-1,2).tolist(),
                            'component':component,'box_indices':[]})
        component_to_region={region['component']:i for i,region in enumerate(regions)}
        for i,box in enumerate(boxes):
            cx=min(width-1,int(box['center'][0]))
            cy=min(height-1,int(box['center'][1]))
            component=int(labels[cy,cx])
            region_index=component_to_region.get(component)
            if region_index is not None:
                box['region']=region_index
                regions[region_index]['box_indices'].append(i)
        return boxes,regions

    def _detect(self, painted, now, force=False):
        p=self.params
        fps=max(1.,float(p.get('input_fps',25)))
        if not force and now-self.last_detection<max(.02,p['detect_every_n_frames']/fps):
            return
        self.people=merge_overlapping_boxes(
            detections(self.person_model,painted,p['person_conf'],p['imgsz_infer'],
                       ('person','pessoa','human')),p['merge_iou'])
        self.firefighters=(merge_overlapping_boxes(
            detections(self.firefighter_model,painted,p['firefighter_conf'],p['imgsz_infer'],
                       ('firefighter','fireman','bombeiro','helmet','capacete','turnout',
                        'protective','ppe','epi')),p['merge_iou'])
            if self.firefighter_model is not None else [])
        raw=detections(self.fire_model,painted,p['fire_conf'],p['imgsz_infer'],
                       ('fire','flame','fogo','chama'))
        self.fires,self.regions=self._small_boxes_and_regions(painted,raw)
        self.last_detection=time.monotonic()
        if self.people or self.firefighters:self.last_person=self.last_detection
        if not self.fires:self.visited_cells.clear()

    def process(self, frame, engaged=True):
        now=time.monotonic()
        p=self.params
        height,width=frame.shape[:2]
        painted=self._painted_frame(frame,now)
        # Em simulação a inferência recebe o vídeo já pintado, sem desenhos de interface.
        self._detect(painted,now)
        blocked=now-self.last_person<=p['person_memory_seconds']
        if self.impact is None:self.impact=(width/2.,height/2.)
        self.route=[tuple(box['center']) for box in self.fires]
        self.target=None
        if engaged and not blocked and self.fires:
            candidates=[box for box in self.fires if box['cell'] not in self.visited_cells]
            if not candidates:
                self.visited_cells.clear()
                candidates=self.fires
            self.target=min(candidates,key=lambda box:math.dist(self.impact,box['center']))
        if blocked or not self.fires:
            self.impact=None
            self.last_step=None
            if blocked:self.trail=[]
        fps=max(1.,float(p.get('input_fps',25)))
        dt=min(.25,max(1./fps,now-(self.last_step if self.last_step is not None else now)))
        self.last_step=now if self.target is not None else None
        arrived=False
        if self.target is not None:
            waypoint=self.target['center']
            distance=math.dist(self.impact,waypoint)
            step=float(p['route_speed_px_frame'])*fps*dt
            if distance<=step:
                self.impact=tuple(map(float,waypoint))
                arrived=True
                self.visited_cells.add(self.target['cell'])
                # A próxima imagem exige inferência nova antes de escolher outro centro.
                self.last_detection=-1e10
            elif distance:
                self.impact=(self.impact[0]+(waypoint[0]-self.impact[0])*step/distance,
                             self.impact[1]+(waypoint[1]-self.impact[1])*step/distance)
        active_region=None
        active_region_index=None
        if self.impact is not None and not blocked:
            for index,region in enumerate(self.regions):
                if cv2.pointPolygonTest(region['contour'],self.impact,False)>=0:
                    active_region=region
                    active_region_index=index
                    break
        enabled=active_region is not None and engaged
        if enabled and self.mode=='simulation':
            self.trail.append((now,*self.impact))
            self.trail=self.trail[-max(100,int(p['restore_seconds']*fps*2)):]
            painted=self._painted_frame(frame,now)
        aim=(self.impact[0],max(0.,self.impact[1]-p['jet_drop_px'])) if self.target else None
        pan,tilt=pixel_to_servos(aim,frame.shape,self.calibration,p) if aim else (None,None)
        record={'t_unix':time.time(),'mode':self.mode,'camera':str(self.source),
                'frame_width':width,'frame_height':height,'strategy':'centroides_em_regioes',
                'route_index':self.fires.index(self.target) if self.target else None,
                'route_pixels':[list(point) for point in self.route],
                'small_boxes':[list(box['box']) for box in self.fires],
                'regions':[region['export'] for region in self.regions],
                'impact_px':list(self.impact) if self.impact else None,
                'aim_px':list(aim) if aim else None,'pan_deg':pan,'tilt_deg':tilt,
                'target_valid':bool(self.target),'centroid_reached':bool(arrived),
                'person_detected':bool(self.people),'firefighter_detected':bool(self.firefighters),
                'blocked':bool(blocked),'jet_requested':bool(self.target and not blocked),
                'jet_permitted':bool(enabled),'region_index':active_region_index}
        if self.mode=='real' or enabled or blocked:
            self.journal.write(json.dumps(record,ensure_ascii=False)+'\n')
            self.journal.flush()
        canvas=painted
        label_scale=max(.45,min(.9,width/1500.))
        label_stroke=max(1,int(round(label_scale*2)))
        for box in self.fires:
            bx1,by1,bx2,by2=box['box']
            cv2.rectangle(canvas,(bx1,by1),(bx2,by2),(0,150,255),1)
            cv2.circle(canvas,tuple(map(int,box['center'])),3,(0,200,255),-1)
        for region in self.regions:
            cv2.polylines(canvas,[region['contour']],True,(220,40,220),2,cv2.LINE_AA)
            indices=region['box_indices']
            if indices:
                mean_conf=sum(self.fires[index]['conf'] for index in indices)/len(indices)
                label=f'FOGO {mean_conf:.0%}'
                points=region['contour'].reshape(-1,2)
                x=int(points[:,0].min())
                y=max(16,int(points[:,1].min())-4)
                small_scale=max(.35,min(.6,width/2200.))
                cv2.putText(canvas,label,(x,y),cv2.FONT_HERSHEY_SIMPLEX,
                            small_scale,(0,0,0),3)
                cv2.putText(canvas,label,(x,y),cv2.FONT_HERSHEY_SIMPLEX,
                            small_scale,(220,40,220),1)
        for kind,items,color in (('PESSOA',self.people,(0,0,255)),
                                 ('BOMBEIRO/EPI',self.firefighters,(255,200,0))):
            for found in items:
                x1,y1,x2,y2=found['box']
                margin=int(p['human_safety_radius_px'])
                cv2.rectangle(canvas,(max(0,x1-margin),max(0,y1-margin)),
                              (min(width-1,x2+margin),min(height-1,y2+margin)),color,1)
                cv2.rectangle(canvas,(x1,y1),(x2,y2),color,2)
                label=f'{kind} {found["conf"]:.0%}'
                y=max(18,y1-3)
                cv2.putText(canvas,label,(x1,y),cv2.FONT_HERSHEY_SIMPLEX,
                            label_scale,(0,0,0),label_stroke+2)
                cv2.putText(canvas,label,(x1,y),cv2.FONT_HERSHEY_SIMPLEX,
                            label_scale,color,label_stroke)
        if blocked:
            cv2.rectangle(canvas,(0,0),(width,max(55,int(width*.085))),(0,0,170),-1)
            cv2.putText(canvas,'PESSOA/BOMBEIRO: JATO BLOQUEADO',
                        (10,max(40,int(width*.06))),cv2.FONT_HERSHEY_SIMPLEX,
                        max(1.,width/640.),(255,255,255),3)
        if self.target is not None and not blocked and self.impact is not None:
            hit=(int(round(self.impact[0])),int(round(self.impact[1])))
            aim_pt=(int(round(aim[0])),int(round(aim[1])))
            cv2.circle(canvas,aim_pt,9,(0,200,0),-1)
            cv2.circle(canvas,hit,7,(0,255,255),2)
            if enabled:
                drop=float(p['jet_drop_px'])
                start=(width//2,height-1)
                path=[]
                for t in np.linspace(0.,1.,40):
                    x=start[0]+(hit[0]-start[0])*t
                    y=start[1]+(hit[1]-start[1]-drop)*t+drop*t*t
                    path.append((int(round(x)),int(round(y))))
                cv2.polylines(canvas,[np.array(path,dtype=np.int32)],
                              False,(255,110,0),max(5,width//120),cv2.LINE_AA)
                cv2.circle(canvas,aim_pt,9,(0,200,0),-1)
                cv2.circle(canvas,hit,9,(255,255,255),2)
        return canvas,record

# Encerra a sessão anterior ao reexecutar a célula.
if 'vision_stop' in globals(): vision_stop.set()
if 'vision_thread' in globals() and vision_thread is not None: vision_thread.join(timeout=5)
vision_stop = threading.Event()
vision_thread = None
vision_session = None
train_process = None
if 'camera_test_stop' in globals():camera_test_stop.set()
if 'camera_test_thread' in globals() and camera_test_thread is not None:
    camera_test_thread.join(timeout=4)
camera_test_stop=threading.Event()
camera_test_thread=None
vision_status = widgets.HTML('Configure as fontes e modelos antes de iniciar.')
training_status = widgets.HTML('Nenhum treinamento iniciado.')
setup_status = widgets.HTML('')
vision_display = widgets.Image(format='jpeg',layout=widgets.Layout(width='850px'))

mode_select = widgets.ToggleButtons(options=[('Simulação','simulation'),
                                             ('Combate real','real')],
                                     description='Modo:',style={'description_width':'initial'},
                                     layout=widgets.Layout(width='620px'))
saved_camera=cfg.camera_source()
initial_input='local' if isinstance(saved_camera,int) else 'video'
import runtime as network_runtime
input_mode = widgets.Dropdown(options=[('Câmera USB/local','local'),
                                       ('Arquivo de vídeo','video')],value=initial_input,
                              description='Entrada:',layout=widgets.Layout(width='390px'))
local_indices=sorted(set(range(6)) | ({saved_camera} if isinstance(saved_camera,int) else set()))
camera_list = widgets.Dropdown(options=[(f'Câmera local {i}',i) for i in local_indices],
                               value=saved_camera if isinstance(saved_camera,int) else 0,
                               description='Índice:',layout=widgets.Layout(width='340px'))
video_list = widgets.Dropdown(description='Vídeo:',layout=widgets.Layout(width='580px'))
input_fps=widgets.BoundedIntText(value=25,min=1,max=120,
    description='FPS desejado:',style={'description_width':'125px'},
    layout=widgets.Layout(width='240px'))
vision_playback_state={'seek':None,'internal':False}
vision_playback_lock=threading.Lock()
video_timeline=widgets.FloatSlider(value=0,min=0,max=1,step=.1,
    description='Tempo:',continuous_update=False,readout=False,
    layout=widgets.Layout(width='520px'))
video_timeline.disabled=True
video_time_label=widgets.HTML('00:00 / 00:00')
video_back=widgets.Button(description='Voltar 5 s',layout=widgets.Layout(width='110px'))
video_forward=widgets.Button(description='Avançar 5 s',layout=widgets.Layout(width='120px'))
video_pause=widgets.ToggleButton(value=False,description='Pausar vídeo',
    button_style='warning',layout=widgets.Layout(width='140px'))
video_controls=widgets.VBox([widgets.HBox([video_back,video_pause,video_forward]),
                              widgets.HBox([video_timeline,video_time_label])])
test_local_button=widgets.Button(description='Testar câmera',button_style='info',
                                 layout=widgets.Layout(width='155px'))
stop_camera_test_button=widgets.Button(description='Parar teste',button_style='warning',
                                       layout=widgets.Layout(width='130px'))
camera_test_image=widgets.Image(format='jpeg',layout=widgets.Layout(width='320px',height='240px'))
camera_test_status=widgets.HTML('Prévia parada.')
local_input_panel=widgets.HBox([camera_list,test_local_button])
video_input_panel=widgets.HBox([video_list])
camera_input_controls=widgets.VBox([local_input_panel,video_input_panel,
                                    note(input_fps,'No arquivo, define a velocidade de reprodução; na câmera, solicita esta taxa à captura. Padrão: 25 FPS.'),
                                    stop_camera_test_button,camera_test_status])
camera_test_preview=widgets.VBox([camera_test_image],layout=widgets.Layout(width='330px'))

def update_input_panel(_=None):
    local_input_panel.layout.display='' if input_mode.value=='local' else 'none'
    video_input_panel.layout.display='' if input_mode.value=='video' else 'none'
    video_controls.layout.display='' if input_mode.value=='video' else 'none'
    camera_test_preview.layout.display='' if input_mode.value!='video' else 'none'
    stop_camera_test_button.layout.display='' if input_mode.value!='video' else 'none'
    camera_test_status.layout.display='' if input_mode.value!='video' else 'none'
    if input_mode.value=='video':camera_test_stop.set()
    if 'update_calibration_panel' in globals():update_calibration_panel()

input_mode.observe(update_input_panel,names='value')
update_input_panel()

def format_video_time(seconds):
    seconds=max(0,int(seconds))
    return f'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}' if seconds>=3600 else f'{seconds//60:02d}:{seconds%60:02d}'

def request_video_seek(seconds):
    if video_timeline.disabled:return
    with vision_playback_lock:vision_playback_state['seek']=max(0,min(float(video_timeline.max),float(seconds)))

def on_video_timeline(change):
    if change['name']=='value' and not vision_playback_state['internal']:
        request_video_seek(change['new'])

def on_video_pause(change):
    video_pause.description='Retomar vídeo' if change['new'] else 'Pausar vídeo'
    video_pause.button_style='success' if change['new'] else 'warning'

video_timeline.observe(on_video_timeline,names='value')
video_pause.observe(on_video_pause,names='value')
video_back.on_click(lambda _:request_video_seek(video_timeline.value-5))
video_forward.on_click(lambda _:request_video_seek(video_timeline.value+5))
fire_list = widgets.Dropdown(description='Modelo fogo:',layout=widgets.Layout(width='680px'))
person_list = widgets.Dropdown(description='Modelo pessoa:',layout=widgets.Layout(width='680px'))
firefighter_list=widgets.Dropdown(description='Bombeiro/EPI:',layout=widgets.Layout(width='680px'))
fire_strategy = widgets.RadioButtons(options=[('Usar modelo já treinado','existing'),
                                              ('Treinar novo modelo com dataset','train')],
                                     value='existing',description='Fogo:',
                                     style={'description_width':'80px'},
                                     layout=widgets.Layout(width='440px'))
person_strategy = widgets.RadioButtons(options=[('Usar modelo já treinado','existing'),
                                                ('Treinar novo modelo com dataset','train')],
                                       value='existing',description='Pessoas:',
                                       style={'description_width':'80px'},
                                       layout=widgets.Layout(width='440px'))
firefighter_strategy=widgets.RadioButtons(options=[('Sem modelo (opcional)','none'),
    ('Usar modelo já treinado','existing'),('Treinar novo modelo com dataset','train')],
    value='none',description='Bombeiro:',style={'description_width':'80px'},
    layout=widgets.Layout(width='460px'))
trained_models={'fire':None,'person':None,'firefighter':None}
fire_trained_info=widgets.HTML('')
person_trained_info=widgets.HTML('')
firefighter_trained_info=widgets.HTML('')
fire_dataset = widgets.Dropdown(description='Dataset fogo:',layout=widgets.Layout(width='680px'))
person_dataset = widgets.Dropdown(description='Dataset pessoa:',layout=widgets.Layout(width='680px'))
firefighter_dataset=widgets.Dropdown(description='Dataset bombeiro:',layout=widgets.Layout(width='680px'))
seed_list = widgets.Dropdown(description='Peso inicial:',layout=widgets.Layout(width='680px'))
rgb_entries = widgets.Dropdown(description='Setup RGB:',layout=widgets.Layout(width='680px'))
load_rgb_button=widgets.Button(description='Importar parâmetros RGB',
                              layout=widgets.Layout(width='220px'))
refresh_button = widgets.Button(description='Atualizar listas')
train_fire = widgets.Button(description='Treinar fogo',button_style='info')
train_person = widgets.Button(description='Treinar pessoas',button_style='info')
train_firefighter=widgets.Button(description='Treinar bombeiros/EPI',button_style='info')
stop_training = widgets.Button(description='Interromper treino',button_style='warning')
train_epochs = widgets.BoundedIntText(value=50,min=1,max=1000,description='Épocas:')
train_size = widgets.Dropdown(options=[320,640,960,1280],value=640,description='Imagem:')
train_batch = widgets.BoundedIntText(value=8,min=1,max=128,description='Lote:')
train_name = widgets.Text(value='ensaio',description='Nome:')
for button,width in ((refresh_button,'150px'),(train_fire,'160px'),
                     (train_person,'175px'),(train_firefighter,'195px'),(stop_training,'185px')):
    button.layout=widgets.Layout(width=width)
fire_existing_panel=widgets.VBox([note(fire_list,'Escolha um arquivo .pt com classe fire, flame, fogo ou chama.')])
person_existing_panel=widgets.VBox([note(person_list,'Escolha um arquivo .pt com classe person, pessoa ou human.')])
firefighter_existing_panel=widgets.VBox([note(firefighter_list,'Classes bombeiro/firefighter ou vestimentas como helmet, capacete, protective_clothing ou EPI.')])
fire_training_panel=widgets.VBox([
    note(fire_dataset,'Dataset YOLO com imagens e rótulos em train e val.'),
    widgets.HBox([train_fire,fire_trained_info])])
person_training_panel=widgets.VBox([
    note(person_dataset,'Dataset YOLO com imagens e rótulos em train e val.'),
    widgets.HBox([train_person,person_trained_info])])
firefighter_training_panel=widgets.VBox([
    note(firefighter_dataset,'Dataset YOLO anotado de bombeiros e/ou vestimentas, com imagens e rótulos train/val.'),
    widgets.HBox([train_firefighter,firefighter_trained_info])])
shared_training_panel=widgets.VBox([
    note(seed_list,'Peso inicial; um modelo COCO é ajustado ao dataset selecionado.'),
    widgets.HBox([train_epochs,train_size,train_batch,train_name]),
    stop_training,training_status])

def update_model_panels(_=None):
    fire_existing_panel.layout.display='' if fire_strategy.value=='existing' else 'none'
    fire_training_panel.layout.display='' if fire_strategy.value=='train' else 'none'
    person_existing_panel.layout.display='' if person_strategy.value=='existing' else 'none'
    person_training_panel.layout.display='' if person_strategy.value=='train' else 'none'
    firefighter_existing_panel.layout.display='' if firefighter_strategy.value=='existing' else 'none'
    firefighter_training_panel.layout.display='' if firefighter_strategy.value=='train' else 'none'
    shared_training_panel.layout.display=('' if 'train' in (fire_strategy.value,person_strategy.value,
                                                           firefighter_strategy.value)
                                          else 'none')

fire_strategy.observe(update_model_panels,names='value')
person_strategy.observe(update_model_panels,names='value')
firefighter_strategy.observe(update_model_panels,names='value')
update_model_panels()

param_definitions = [
 ('fire_conf','Conf. chama',.01,.95,.01,.30,'Confiança mínima da chama; 0,30 significa aceitar resultados a partir de 30%.'),
 ('person_conf','Conf. pessoa',.05,.95,.01,.70,'Confiança mínima de pessoa; uma detecção aceita impede o acionamento do jato.'),
 ('firefighter_conf','Conf. bombeiro',.05,.95,.01,.70,'Confiança mínima de bombeiro ou EPI; usada somente se o modelo opcional for selecionado.'),
 ('imgsz_infer','Tamanho YOLO',320,1280,32,640,'Resolução da inferência; 640 equilibra detalhes e velocidade.'),
 ('detect_every_n_frames','Ciclo YOLO',1,30,1,15,'Intervalo-alvo em quadros entre inferências; equivale a 0,6 s a 25 FPS, quando o processamento acompanha a captura.'),
 ('merge_iou','IoU fusão',.05,.95,.05,.45,'Sobreposição mínima para fundir caixas de pessoa/bombeiro; caixas de fogo são recortadas em células sem sobreposição.'),
 ('max_fire_box_area_px','Área máxima caixa',100,10000,100,8000,'Área máxima de cada caixa de fogo em pixels quadrados; chamas grandes são divididas em caixas pequenas, sem sobreposição.'),
 ('route_speed_px_frame','Vel. mira',1,120,1,30,'Velocidade máxima em pixels por quadro do FPS configurado para perseguir o centróide ainda não visitado mais próximo.'),
 ('jet_drop_px','Queda do jato',0,160,1,90,'Curvatura ilustrativa do jato que sai do centro inferior e distância vertical da mira verde acima do impacto.'),
 ('erase_radius_px','Raio virtual',2,120,1,80,'Raio da área preta em pixels quando o jato virtual está acionado, apenas na simulação.'),
 ('restore_seconds','Restituição (s)',.5,20.,.5,20.,'Tempo para a cauda preta desaparecer gradualmente e a imagem original voltar à inferência; padrão da simulação: 20 s.'),
 ('human_safety_radius_px','Dist. pessoa',1,300,1,90,'Margem visual em pixels ao redor da caixa de pessoa; qualquer pessoa detectada bloqueia o jato em toda a imagem.'),
 ('person_memory_seconds','Memória pessoa',.1,10.,.1,2.,'Mantém o bloqueio durante este tempo após a última detecção de pessoa, evitando religamento imediato.'),
]
params_widgets = {}; param_rows=[]
for key,label,low,high,step,default,help_text in param_definitions:
    control_type = widgets.FloatSlider if isinstance(default,float) else widgets.IntSlider
    saved = default
    control = control_type(value=max(low,min(high,saved)),min=low,max=high,step=step,
        description=label,style={'description_width':'140px'},layout=widgets.Layout(width='470px'))
    params_widgets[key]=control
    param_rows.append(note(control,help_text))

simulation_defaults={key:default for key,_,_,_,_,default,_ in param_definitions}
real_defaults={**simulation_defaults,'route_speed_px_frame':5}
mode_defaults={'simulation':simulation_defaults,'real':real_defaults}
mode_param_cache={}
active_param_mode={'value':mode_select.value}

def switch_param_mode(change):
    previous=active_param_mode['value']
    mode_param_cache[previous]={key:control.value for key,control in params_widgets.items()}
    selected=mode_param_cache.get(change['new'],mode_defaults[change['new']])
    for key,control in params_widgets.items():
        control.value=selected[key]
    active_param_mode['value']=change['new']

mode_select.observe(switch_param_mode,names='value')

calibration_panel=widgets.HTML(
    '<div style="padding:12px;border:1px solid #94a3b8;border-radius:7px">'
    '<b>Calibração física da plataforma e da mira</b><br>'
    'Após gravar o firmware de combate real e conectar a câmera e os três servos, '
    'abra o runtime e clique <b>Calibrar câmera e canhão</b>. Registre os quatro '
    'cantos da imagem nas posições 0°, 90° e 180°. Nenhum slider desta seção '
    'move servos físicos.</div>')
calibration_width,calibration_height=640,480

setup_name=widgets.Text(value='RGB-CAM',description='Nome do setup:',layout=widgets.Layout(width='330px'))
setup_pick=widgets.Dropdown(description='Salvos:')
setup_save_button=widgets.Button(description='Salvar/atualizar',button_style='success')
setup_load_button=widgets.Button(description='Carregar para editar',button_style='info')
setup_delete_button=widgets.Button(description='Excluir',button_style='danger')
vision_start=widgets.Button(description='Iniciar visão',button_style='success')
vision_stop_button=widgets.Button(description='Parar visão',button_style='warning')
vision_reset=widgets.Button(description='Reiniciar trajetória')
for button,width in ((setup_save_button,'165px'),(setup_load_button,'190px'),
                     (setup_delete_button,'100px'),(vision_start,'145px'),
                     (vision_stop_button,'135px'),
                     (vision_reset,'190px')):
    button.layout=widgets.Layout(width=width)

def setup_key(mode, label):
    return ('S-' if mode=='simulation' else 'R-')+label.strip()

def refresh_setup_options(preferred=None):
    names=[]
    for name in setup_names('vision'):
        try:
            mode=setup_get('vision',name)['vision'].get('combat_mode','simulation')
            if mode==mode_select.value:names.append(name)
        except (KeyError,ValueError,TypeError):continue
    refresh_dropdown(setup_pick,[(name,name) for name in names],preferred)

mode_select.observe(lambda change:refresh_setup_options(),names='value')

def refresh_lists(_=None, preferred=None):
    models=choices(model_paths(),'Modelo')
    fire_guess=preferred or next((str(p.resolve()) for p in model_paths() if 'fire' in p.stem.lower()),'')
    person_guess=next((str(p.resolve()) for p in model_paths() if
                       'person' in p.stem.lower() or p.stem.lower()=='yolo11n'),'')
    refresh_dropdown(fire_list,models,preferred or fire_list.value or fire_guess)
    refresh_dropdown(person_list,models,person_list.value or person_guess)
    fighter_guess=next((str(p.resolve()) for p in model_paths() if
        'firefighter' in p.stem.lower() or 'bombeiro' in p.stem.lower()),'')
    refresh_dropdown(firefighter_list,models,firefighter_list.value or fighter_guess)
    refresh_dropdown(seed_list,models+[('YOLO11n inicial (baixa no treino)','yolo11n.pt')],
                     seed_list.value or 'yolo11n.pt')
    datasets=choices(data_paths(),'Dataset')
    refresh_dropdown(fire_dataset,datasets,fire_dataset.value or str((DATA_DIR/'fire'/'dataset.yaml').resolve()))
    refresh_dropdown(person_dataset,datasets,person_dataset.value or str((DATA_DIR/'person'/'dataset.yaml').resolve()))
    refresh_dropdown(firefighter_dataset,datasets,firefighter_dataset.value or
                     str((DATA_DIR/'firefighter'/'dataset.yaml').resolve()))
    videos=[p for p in VIDEO_DIR.iterdir() if p.suffix.lower() in ('.mp4','.avi','.mov','.mkv')]
    refresh_dropdown(video_list,choices(videos,'Vídeo'))
    legacy=read_legacy()
    names=sorted(legacy,key=lambda name:str(legacy[name].get('saved_at','')),reverse=True)
    refresh_dropdown(rgb_entries,[(name,name) for name in names],rgb_entries.value or (names[0] if names else ''))
    refresh_setup_options()

def apply_legacy(name):
    entry=read_legacy().get(name,{})
    values=entry.get('config',entry)
    if isinstance(values,dict):
        for key,control in params_widgets.items():
            if key in values:
                try: control.value=values[key]
                except Exception: pass

def selected_input():
    if input_mode.value=='video':
        path=resolved_path(video_list)
        if path is None or not path.is_file():
            raise ValueError(f'Coloque um vídeo em {VIDEO_DIR} e selecione-o.')
        return str(path), 'video'
    if input_mode.value=='local':
        return int(camera_list.value),'camera'
    raise ValueError('Selecione uma câmera USB ou um vídeo salvo.')

def stop_setup_camera_preview():
    callback=globals().get('stop_camera_preview')
    if not callable(callback):return
    try:
        callback()
    except AttributeError as exc:
        # Uma execução anterior da seção 4 sobrescreveu o widget da célula 2.
        # O callback já interrompeu a captura antes de atualizar esse indicador.
        if not (isinstance(globals().get('video_state'),dict) and
                "'dict' object has no attribute 'value'" in str(exc)):
            raise

def stop_camera_test(_=None):
    camera_test_stop.set()
    camera_test_image.value=b''
    camera_test_status.value=status('Teste da câmera','PENDENTE','Encerrando prévia...')
    def finish(thread=camera_test_thread, stopped=camera_test_stop):
        if thread is not None and thread is not threading.current_thread():
            thread.join(timeout=7)
        if stopped is not camera_test_stop:
            return
        if thread is None or not thread.is_alive():
            camera_test_status.value=status('Teste da câmera','OK','Teste encerrado; câmera liberada.')
        else:
            camera_test_status.value=status('Teste da câmera','FALHA',
                'A captura ainda está fechando; tente novamente após alguns segundos.')
    threading.Thread(target=finish,daemon=True).start()

def test_selected_camera(_):
    global camera_test_stop,camera_test_thread
    camera_test_stop.set()
    if camera_test_thread is not None:
        camera_test_thread.join(timeout=4)
        if camera_test_thread.is_alive():
            camera_test_status.value=status('Teste da câmera','FALHA','A captura anterior ainda está encerrando.')
            return
    if 'vision_stop' in globals():
        vision_stop.set()
    if 'vision_thread' in globals() and vision_thread is not None:
        vision_thread.join(timeout=4)
        if vision_thread.is_alive():
            camera_test_status.value=status('Teste da câmera','FALHA','Pare a visão antes de testar outra câmera.')
            return
    stop_setup_camera_preview()
    try:
        source,kind=selected_input()
        if kind!='camera':raise ValueError('Selecione uma câmera USB.')
        camera_test_stop=threading.Event()
        this_stop=camera_test_stop
        camera_test_image.value=b''
        camera_test_status.value=status('Teste da câmera','PENDENTE',f'Abrindo {network_runtime.redacted_source(source)}...')
        def preview_loop():
            cap=None
            try:
                cap,first,backend,errors=open_capture(source,this_stop,'camera',input_fps.value)
                if this_stop.is_set():return
                if cap is None:
                    camera_test_status.value=status('Teste da câmera','FALHA',
                                                     f'{network_runtime.redacted_source(source)} não forneceu vídeo: {errors}')
                    return
                pending=first
                failures=0
                while not this_stop.is_set():
                    if pending is not None:
                        frame,pending=pending,None
                    else:
                        ok,frame=cap.read()
                        if not ok or frame is None or not frame.size:
                            failures+=1
                            if failures>=10:raise RuntimeError('O fluxo de vídeo parou.')
                            this_stop.wait(.1)
                            continue
                    failures=0
                    h,w=frame.shape[:2]
                    shown=cv2.resize(frame,(320,max(1,round(h*320/w))))
                    ok,jpg=cv2.imencode('.jpg',shown,[cv2.IMWRITE_JPEG_QUALITY,75])
                    if ok:camera_test_image.value=jpg.tobytes()
                    camera_test_status.value=status('Teste da câmera','OK',
                        f'{network_runtime.redacted_source(source)} · {backend} · {w} × {h} pixels')
                    this_stop.wait(.12)
            except Exception as exc:
                camera_test_status.value=status('Teste da câmera','FALHA',
                    f'{type(exc).__name__}: {exc}')
            finally:
                if cap is not None:cap.release()
                if this_stop.is_set():
                    camera_test_image.value=b''
                    camera_test_status.value=status('Teste da câmera','OK','Prévia encerrada; câmera liberada.')
        camera_test_thread=threading.Thread(target=preview_loop,daemon=True)
        camera_test_thread.start()
    except Exception as exc:
        camera_test_status.value=status('Teste da câmera','FALHA',str(exc))

def calibration_fields():
    # A calibração física é salva pelo runtime para uma combinação completa.
    return {'enabled':False}

def section_fields():
    source,source_type=selected_input()
    fire=(resolved_path(fire_list) if fire_strategy.value=='existing' else trained_models['fire'])
    person=(resolved_path(person_list) if person_strategy.value=='existing' else trained_models['person'])
    if fire is None or not Path(fire).is_file():
        raise ValueError('Fogo: selecione um modelo treinado ou conclua o treinamento.')
    if person is None or not Path(person).is_file():
        raise ValueError('Pessoas: selecione um modelo treinado ou conclua o treinamento.')
    firefighter=(None if firefighter_strategy.value=='none' else
                 resolved_path(firefighter_list) if firefighter_strategy.value=='existing' else
                 trained_models['firefighter'])
    if firefighter_strategy.value!='none' and (firefighter is None or not Path(firefighter).is_file()):
        raise ValueError('Bombeiro/EPI: selecione um modelo ou conclua o treinamento opcional.')
    return {**cfg.data['vision'], 'combat_mode':mode_select.value,
      'camera_source':source, 'source_type':input_mode.value,
      'input_fps':input_fps.value,
      'model_strategy':{'fire':fire_strategy.value,'person':person_strategy.value,
                        'firefighter':firefighter_strategy.value},
      'fire_model':within_package(fire) if fire else '',
      'person_model':within_package(person) if person else '',
      'firefighter_model':within_package(firefighter) if firefighter else '',
      'legacy_setup_file':within_package(VISION_FILE),
      'legacy_setup_name':rgb_entries.value or '',
      'params_override':{**{k:v.value for k,v in params_widgets.items()},'input_fps':input_fps.value},
      'calibration_override':calibration_fields(), 'setup_label':setup_name.value.strip(),
      'trajectory_export_dir':str(EXPORT_DIR.resolve())}

def save_setup(_):
    global cfg
    try:
        if not setup_name.value.strip():raise ValueError('Informe um nome para este grupo.')
        section=section_fields()
        name=setup_save('vision',setup_key(mode_select.value,setup_name.value),{'vision':section})
        current=json.loads(config_path.read_text(encoding='utf-8')) if config_path.is_file() else {}
        current['vision']=section
        _json_atomic(config_path,current)
        cfg=AppConfig(config_path)
        refresh_setup_options(name)
        setup_status.value=status('Setup de visão','OK',f'{name} salvo. Selecione-o na combinação oficial ao final do notebook.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

def load_setup(_):
    try:
        if not setup_pick.value:raise ValueError('Selecione um setup salvo.')
        section=setup_get('vision',setup_pick.value)['vision']
        setup_name.value=re.sub(r'^(?:SIM - |REAL - |[SR]-)', '', setup_pick.value)
        input_fps.value=int(section.get('input_fps',section.get('params_override',{}).get('input_fps',25)))
        mode_select.value=section.get('combat_mode','simulation')
        saved_kind=section.get('source_type','camera')
        source=section.get('camera_source',cfg.camera_source())
        if saved_kind=='video':
            input_mode.value='video'
            refresh_dropdown(video_list,choices([resolved_path(video_list,str(source))],'Vídeo'),str(resolved_path(video_list,str(source))))
        elif isinstance(source, str) and source.lower().startswith(('rtsp://','rtsps://','http://','https://')):
            raise ValueError('Setup antigo de câmera IP: selecione uma câmera USB ou um vídeo salvo.')
        else:
            input_mode.value='local'
            number=int(source)
            if number not in [v for _,v in camera_list.options]:
                camera_list.options=list(camera_list.options)+[(f'Câmera local {number}',number)]
            camera_list.value=number
        for control,key in ((fire_list,'fire_model'),(person_list,'person_model'),
                            (firefighter_list,'firefighter_model')):
            path=section.get(key,'')
            if path:
                resolved=resolved_path(control,path)
                opts=list(control.options)
                if str(resolved) not in [v for _,v in opts]: opts.append((within_package(resolved),str(resolved)))
                control.options=opts;control.value=str(resolved)
        strategy=section.get('model_strategy',{})
        for kind,selector,control in (('fire',fire_strategy,fire_list),
                                      ('person',person_strategy,person_list),
                                      ('firefighter',firefighter_strategy,firefighter_list)):
            if kind=='firefighter' and strategy.get(kind,'none')=='none':
                selector.value='none'
                continue
            saved_path=resolved_path(control)
            if strategy.get(kind)=='train' and saved_path is not None and saved_path.is_file():
                trained_models[kind]=saved_path
                label=(fire_trained_info if kind=='fire' else
                       person_trained_info if kind=='person' else firefighter_trained_info)
                label.value='Modelo salvo: '+html.escape(str(saved_path))
                selector.value='train'
            else:
                selector.value='existing'
        entry=section.get('legacy_setup_name','')
        if entry in [v for _,v in rgb_entries.options]:rgb_entries.value=entry
        for key,control in params_widgets.items():
            if key in section.get('params_override',{}):control.value=section['params_override'][key]
        setup_status.value=status('Setup de visão','OK','Carregado para edição; salve novamente para atualizar.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

def delete_setup(_):
    try:
        name=setup_pick.value
        setup_delete('vision',name)
        refresh_setup_options()
        setup_status.value=status('Setup de visão','OK',f'{name} excluído.')
    except Exception as exc:setup_status.value=status('Setup de visão','FALHA',str(exc))

TRAIN_SCRIPT = r'''
import json, sys
from pathlib import Path
from ultralytics import YOLO
request=json.loads(sys.argv[1])
model=YOLO(request['seed'])
model.train(data=request['dataset'],epochs=request['epochs'],imgsz=request['imgsz'],
            batch=request['batch'],workers=0,project=request['project'],
            name=request['name'],exist_ok=False,plots=True)
print('TREINO_CONCLUIDO',flush=True)
'''

def start_training(kind):
    global train_process
    if train_process is not None and train_process.poll() is None:
        training_status.value=status('Treino','PENDENTE','Há outro treino em andamento.')
        return
    try:
        dataset=resolved_path({'fire':fire_dataset,'person':person_dataset,
                               'firefighter':firefighter_dataset}[kind])
        counts=dataset_report(dataset,kind)
        seed=seed_list.value
        if not seed:raise ValueError('Selecione um peso inicial YOLO.')
        import ultralytics
        run_name=''.join(c if c.isalnum() or c in '_-' else '_' for c in train_name.value.strip()) or 'ensaio'
        run_name=f'{kind}_{run_name}_{_vision_datetime.now():%Y%m%d_%H%M%S}'
        # Ultralytics resolve caminhos relativos de acordo com sua pasta de datasets.
        # Crie um YAML de execução com raiz absoluta para treinar a pasta selecionada.
        import yaml
        resolved_data=yaml.safe_load(dataset.read_text(encoding='utf-8'))
        data_root=Path(resolved_data.get('path') or dataset.parent)
        resolved_data['path']=str((dataset.parent/data_root).resolve() if not data_root.is_absolute() else data_root)
        normalized_yaml=EXPORT_DIR/f'{run_name}_dataset.yaml'
        normalized_yaml.write_text(yaml.safe_dump(resolved_data,allow_unicode=True),encoding='utf-8')
        request={'seed':seed,'dataset':str(normalized_yaml),'epochs':int(train_epochs.value),
                 'imgsz':int(train_size.value),'batch':int(train_batch.value),
                 'project':str(RUNS_DIR.resolve()),'name':run_name}
        log=EXPORT_DIR / f'{run_name}.log'
        log_handle=log.open('w',encoding='utf-8')
        try:
            train_process=subprocess.Popen([sys.executable,'-u','-c',TRAIN_SCRIPT,json.dumps(request)],
                cwd=str(PC_DIR),stdout=log_handle,stderr=subprocess.STDOUT)
        finally:
            log_handle.close()
        training_status.value=status('Treino','PENDENTE',
           f'{kind}: {counts}; execução {run_name}; log {log}. O detector de bombeiro é opcional.')
        def monitor(process):
            global train_process
            code=process.wait()
            if code==0:
                best=RUNS_DIR/run_name/'weights'/'best.pt'
                if best.is_file():
                    target=MODELS_DIR/f'{run_name}_best.pt'
                    shutil.copy2(best,target)
                    trained_models[kind]=target.resolve()
                    refresh_lists(preferred=str(target) if kind=='fire' else None)
                    if kind=='person':person_list.value=str(target)
                    if kind=='firefighter':firefighter_list.value=str(target)
                    label={'fire':fire_trained_info,'person':person_trained_info,
                           'firefighter':firefighter_trained_info}[kind]
                    label.value='<b>Modelo treinado:</b> '+html.escape(str(target))
                    training_status.value=status('Treino','OK',f'{kind} salvo em {target}.')
                else:training_status.value=status('Treino','FALHA',f'Pesos best.pt não encontrados; veja {log}.')
            else:
                tail='\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-8:])
                training_status.value=status('Treino','FALHA',f'Código {code}; {log}; {tail}')
            train_process=None
        threading.Thread(target=monitor,args=(train_process,),daemon=True).start()
    except Exception as exc:
        training_status.value=status('Treino','FALHA',f'{type(exc).__name__}: {exc}')

def stop_train(_):
    if train_process is not None and train_process.poll() is None:
        train_process.terminate()
        training_status.value=status('Treino','PENDENTE','Interrupção solicitada; confira o log.')

def run_vision(session,source,source_type,stop):
    global calibration_width,calibration_height
    cap=None
    try:
        cap,first,backend,errors=open_capture(source,stop,source_type,input_fps.value)
        if cap is None:
            vision_status.value=status('Câmera','FALHA',f'{network_runtime.redacted_source(source)}: {errors}. Selecione outra fonte.')
            return
        calibration_height,calibration_width=first.shape[:2]
        pending=first
        failures=0
        # Arquivos têm relógio próprio. Se a inferência levar mais que 1/FPS,
        # avance o decodificador até o quadro correspondente ao tempo real.
        file_fps=float(input_fps.value) if source_type=='video' else 0.0
        file_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if source_type=='video' else 0
        video_started=time.monotonic()
        next_frame_index=1  # o primeiro quadro já foi lido por open_capture
        skipped_frames=0
        paused_at=None
        if source_type=='video':
            duration=(file_frames/file_fps) if file_frames>0 else 0.0
            video_timeline.max=max(.1,duration)
            video_timeline.disabled=file_frames<=0
            vision_playback_state['internal']=True
            video_timeline.value=0
            vision_playback_state['internal']=False
            video_time_label.value=f'00:00 / {format_video_time(duration)}'
        while not stop.is_set():
            force_seek_frame=False
            if source_type=='video':
                with vision_playback_lock:
                    requested=vision_playback_state['seek']
                    vision_playback_state['seek']=None
                if requested is not None:
                    wanted=max(0,int(requested*file_fps))
                    if file_frames>0:wanted=min(wanted,file_frames-1)
                    if not cap.set(cv2.CAP_PROP_POS_FRAMES,wanted):
                        raise RuntimeError('Este arquivo não permite avançar para o tempo selecionado.')
                    video_started=time.monotonic()-wanted/file_fps
                    next_frame_index=wanted
                    pending=None
                    paused_at=time.monotonic() if video_pause.value else None
                    force_seek_frame=True
                    session.reset()
                if video_pause.value and not force_seek_frame and pending is None:
                    if paused_at is None:paused_at=time.monotonic()
                    stop.wait(.04)
                    continue
                if not video_pause.value and paused_at is not None:
                    video_started+=time.monotonic()-paused_at
                    paused_at=None
            if pending is not None:
                image,pending=pending,None
            else:
                if source_type=='video':
                    elapsed=time.monotonic()-video_started
                    wanted=next_frame_index if force_seek_frame else int(elapsed*file_fps)
                    if file_frames>0 and wanted>=file_frames:
                        video_started=time.monotonic()
                        wanted=0
                        next_frame_index=0
                        cap.set(cv2.CAP_PROP_POS_FRAMES,0)
                        session.reset()
                    if wanted<next_frame_index and not force_seek_frame:
                        stop.wait(max(0,min(.05,next_frame_index/file_fps-elapsed)))
                        continue
                    to_skip=max(0,wanted-next_frame_index)
                    if to_skip>int(file_fps*2) and cap.set(cv2.CAP_PROP_POS_FRAMES,wanted):
                        skipped_frames+=to_skip
                    else:
                        for _ in range(to_skip):
                            if stop.is_set() or not cap.grab():break
                            skipped_frames+=1
                ok,image=cap.read()
                if not ok or image is None:
                    if source_type=='video':
                        cap.set(cv2.CAP_PROP_POS_FRAMES,0)
                        video_started=time.monotonic()
                        next_frame_index=0
                        paused_at=time.monotonic() if video_pause.value else None
                        session.reset()
                        stop.wait(.02)
                        continue
                    failures+=1
                    if failures>=10:raise RuntimeError('Câmera parou de fornecer quadros.')
                    stop.wait(.1);continue
                if source_type=='video':next_frame_index=wanted+1
            failures=0
            if source_type=='video':
                position=max(0,(next_frame_index-1)/file_fps)
                vision_playback_state['internal']=True
                video_timeline.value=min(video_timeline.max,position)
                vision_playback_state['internal']=False
                video_time_label.value=(f'{format_video_time(position)} / '
                                        f'{format_video_time(duration)}')
            shown,record=session.process(image)
            h,w=shown.shape[:2]
            scaled=cv2.resize(shown,(850,max(1,int(h*850/w)))) if w>850 else shown
            ok,jpg=cv2.imencode('.jpg',scaled)
            if ok:vision_display.value=jpg.tobytes()
            state='FALHA' if record['blocked'] else 'OK' if record['target_valid'] else 'PENDENTE'
            vision_status.value=status('Visão',state,
               f'{session.mode} · {backend} · {record["strategy"]} · centros {len(session.route)} · '
               f'pessoa {record["person_detected"]} · bombeiro/EPI {record["firefighter_detected"]} · '
               f'regiões {len(session.regions)} · jato automático {record["jet_permitted"]} · '
               f'pan/tilt {record["pan_deg"]}/{record["tilt_deg"]} · '
               + (f'vídeo {file_fps:.1f} FPS; {skipped_frames} quadros pulados · ' if source_type=='video' else '')
               + str(session.journal_path))
            results['visao']={'camera':network_runtime.redacted_source(source),'modo':session.mode,'alvo':record['target_valid'],
                              'pessoa':record['person_detected'],'bloqueio':record['blocked'],
                              'bombeiro':record['firefighter_detected'],
                              'jato':record['jet_permitted'],'trajetoria':str(session.journal_path),
                              'fps_configurado':input_fps.value,
                              'quadros_pulados':skipped_frames if source_type=='video' else None}
            if source_type!='video':stop.wait(.03)
    except Exception as exc:
        vision_status.value=status('Visão','FALHA',f'{type(exc).__name__}: {exc}')
    finally:
        if cap is not None:cap.release()
        if source_type=='video':video_timeline.disabled=True
        session.close()

def start_vision(_):
    global vision_thread,vision_stop,vision_session
    vision_stop.set()
    vision_display.value=b''  # evita mostrar último quadro após falha de reconexão
    if vision_thread is not None:
        vision_thread.join(timeout=5)
        if vision_thread.is_alive():
            vision_status.value=status('Visão','FALHA','A câmera anterior ainda está fechando. Aguarde.')
            return
    camera_test_stop.set()
    if camera_test_thread is not None:
        camera_test_thread.join(timeout=4)
        if camera_test_thread.is_alive():
            vision_status.value=status('Câmera','FALHA','A prévia ainda está encerrando. Aguarde.')
            return
    try:
        stop_setup_camera_preview()
    except Exception as exc:
        vision_status.value=status('Prévia anterior','FALHA',str(exc))
        return
    try:
        source,source_type=selected_input()
        fire_path=(resolved_path(fire_list) if fire_strategy.value=='existing' else trained_models['fire'])
        person_path=(resolved_path(person_list) if person_strategy.value=='existing' else trained_models['person'])
        fire=load_model(fire_path,'fire')
        person=load_model(person_path,'person')
        fighter_path=(None if firefighter_strategy.value=='none' else
                      resolved_path(firefighter_list) if firefighter_strategy.value=='existing' else
                      trained_models['firefighter'])
        firefighter=load_model(fighter_path,'firefighter') if fighter_path else None
        if firefighter_strategy.value!='none' and firefighter is None:
            raise ValueError('Selecione ou treine o detector opcional de bombeiro/EPI.')
        params={k:v.value for k,v in params_widgets.items()}
        params['input_fps']=int(input_fps.value)
        vision_session=VisionSession(fire,person,params,calibration_fields(),mode_select.value,
                                     source,firefighter_model=firefighter)
        vision_stop=threading.Event()
        vision_thread=threading.Thread(target=run_vision,
          args=(vision_session,source,source_type,vision_stop),daemon=True)
        vision_status.value=status('Visão','PENDENTE',f'Abrindo {network_runtime.redacted_source(source)}; modelos de fogo e pessoa verificados.')
        vision_thread.start()
    except Exception as exc:
        vision_status.value=status('Pré-condições','FALHA',
          f'{type(exc).__name__}: {exc}. Treine ou selecione os dois modelos antes de iniciar.')

def stop_vision(_):
    vision_stop.set()
    vision_status.value=status('Visão','PENDENTE','Parando e liberando câmera...')

def reset_route(_):
    if vision_session is not None:vision_session.reset()

refresh_button.on_click(refresh_lists)
test_local_button.on_click(test_selected_camera)
stop_camera_test_button.on_click(stop_camera_test)
load_rgb_button.on_click(lambda _:apply_legacy(rgb_entries.value) if rgb_entries.value else None)
train_fire.on_click(lambda _:start_training('fire'))
train_person.on_click(lambda _:start_training('person'))
train_firefighter.on_click(lambda _:start_training('firefighter'))
stop_training.on_click(stop_train)
setup_save_button.on_click(save_setup)
setup_load_button.on_click(load_setup)
setup_delete_button.on_click(delete_setup)
vision_start.on_click(start_vision)
vision_stop_button.on_click(stop_vision)
vision_reset.on_click(reset_route)
refresh_lists()

display(widgets.VBox([
 widgets.HTML('<h3>5. Visão RGB e combate</h3><p>Escolha o modo antes de iniciar. '
  '<b>Simulação:</b> o vídeo continua sem congelar; as áreas atingidas escurecem temporariamente, '
  'e as próximas inferências analisam o quadro pintado. '
  '<b>Combate real:</b> a tela e a inferência seguem a câmera sem regiões pretas. '
  'Em ambos, pequenas caixas de fogo formam conjuntos delimitados por contornos roxos. '
  'Pessoa ou bombeiro/EPI detectado bloqueia o jato. '
  'Esta célula não abre a válvula física.</p>'),
 note(mode_select,'Modo salvo no setup; o runtime pode ler combat_mode e o arquivo de trajetória exportado.'),
 widgets.HTML('<h4>Entrada de vídeo e arquivos do pacote</h4>'),
 note(input_mode,'Escolha primeiro o tipo de entrada. Apenas o campo correspondente será mostrado.'),
 widgets.HBox([camera_input_controls,camera_test_preview]),
  widgets.HTML('<p>A webcam USB é aberta pelo PC. Vídeos salvos são lidos do diretório de vídeos do pacote e podem ser pausados e posicionados durante a simulação.</p>'),
 widgets.HBox([refresh_button]),
 widgets.HTML('<h4>Detector de fogo</h4><p>Escolha se usará um peso .pt existente ou se treinará um novo modelo. '
              'Aparecem apenas os campos da opção escolhida.</p>'),
 fire_strategy,fire_existing_panel,fire_training_panel,
 widgets.HTML('<h4>Detector de pessoas</h4><p>O peso yolo11n.pt já inclui a classe person. '
              'Você também pode ajustar um modelo com um dataset próprio.</p>'),
 person_strategy,person_existing_panel,person_training_panel,
 widgets.HTML('<h4>Detector opcional de bombeiros e vestimentas</h4><p>Escolha nenhum, um peso já treinado ou um dataset para treinar. '
              'As classes bombeiro, capacete e vestimenta de proteção são aceitas; detecções também bloqueiam o jato. '
              'Fogo e pessoa continuam obrigatórios.</p>'),
 firefighter_strategy,firefighter_existing_panel,firefighter_training_panel,
 shared_training_panel,
 widgets.HTML('<p>Para treinar, coloque imagens e rótulos YOLO nas pastas data/fire, data/person ou data/firefighter '
              '(train e val), ou selecione outro YAML. Os resultados são salvos em runs e models. '
              'A visão exige pesos .pt válidos de fogo e pessoa. O detector de bombeiros é opcional.</p>'),
 widgets.HTML('<h4>Parâmetros da visão e trajetória</h4><p>Estes controles definem os limiares dos detectores, a fusão de caixas sobrepostas e a velocidade da mira. '
              'Chamas detectadas são divididas em caixas pequenas sem sobreposição, com área limitada pelo controle abaixo. '
              'A mira visita o centróide não visitado mais próximo; ao alcançá-lo, força uma nova inferência antes de escolher outro. '
              'O jato liga quando a posição do impacto entra no contorno do conjunto de caixas e desliga ao sair. '
              'A parábola azul parte do centro inferior da imagem; a marca verde mostra a mira acima do impacto. '
              'Na simulação, o rastro escuro desaparece pelo tempo de restituição configurado, inicialmente 20 segundos. '
              'A análise seguinte ocorre sobre esse vídeo ainda pintado. A detecção de pessoa ou bombeiro bloqueia o jato.</p>'),
 widgets.HBox([rgb_entries,load_rgb_button]),
 widgets.HTML('<p>Os valores iniciais abaixo correspondem ao ensaio mostrado na imagem. '
              'Ao alternar para combate real, a velocidade inicial é 5 pixels por quadro; ajustes de cada modo são lembrados separadamente nesta execução. '
              'Os parâmetros de visão são independentes da calibração física de 12 pontos, realizada no runtime. '
              'Ajustes desta seção podem ser salvos em um setup e usados depois na combinação oficial.</p>'),
 *param_rows,
 calibration_panel,
 widgets.HBox([vision_start,vision_stop_button,vision_reset]),vision_status,
 vision_display,
 widgets.HTML('<p><b>Reprodução:</b> para arquivos de vídeo, pause, avance ou volte abaixo da tela única. '
              'Ao saltar no vídeo, os centróides visitados e o rastro são reiniciados. '
              'A reprodução e a detecção continuam enquanto a mira percorre os alvos.</p>'),
 video_controls,
 widgets.HTML('<h4>Salvar parâmetros</h4><p>A lista mostra somente setups do modo atual. '
  'Simulação e combate real têm nomes separados e permanecem selecionáveis na combinação oficial ao final do notebook.</p>'),
 widgets.HBox([setup_pick,setup_name]),
 widgets.HBox([setup_load_button,setup_save_button,setup_delete_button]),setup_status,
]))


## 5. Controle de velocidade, posição e plataforma da câmera

Teste os comandos do Xbox e salve um setup de controle. O stick esquerdo com LT movimenta o robô. No modo manual, o stick direito gira o canhão e as setas ←/→ giram a plataforma da câmera. O cone branco transparente mostra seu campo horizontal de 120°. Dois objetos sorteados para o ensaio ficam fixos no mapa e aparecem na janela menor apenas quando entram nesse cone. Girar a câmera desloca a posição aparente dos objetos; mover ou girar o robô também altera sua projeção.

Quando o retículo chega à margem lateral, o movimento do canhão pede um giro de 20° da câmera (ou o passo salvo) para recentralizar; ao girar a câmera pelas setas, o canhão acompanha apenas se o retículo estiver prestes a sair da imagem. O tilt é independente porque a câmera tem apenas um eixo. A escala horizontal da janela menor é relativa ao eixo da câmera (−60° a +60°); os valores absolutos de pan do canhão e da plataforma aparecem no painel. No automático, a plataforma varre 0°–180° na ilustração. Segure seta ↓ por 1 segundo para simular a parada de emergência. Esta cena é geométrica e não altera quadros de um vídeo gravado.


In [ ]:
# Célula completa: controle Xbox, setups, botões de função e simulação visual.
# Pré-requisitos das células anteriores: cfg, controller, widgets, results,
# setup_names/get/save/delete, config_path, _json_atomic, AppConfig e status_box.
import cv2, numpy as np, math, time, json, html, threading, subprocess, sys, random
from runtime import coupled_manual_pose

# Executar novamente esta célula encerra a leitura anterior sem deixar o joystick aberto.
for _event_name in ('control_stop', 'xbox_watch_stop'):
    if _event_name in globals(): globals()[_event_name].set()
for _thread_name in ('control_thread', 'xbox_watch_thread'):
    _old = globals().get(_thread_name)
    if _old is not None and _old.is_alive(): _old.join(timeout=2)
if 'xbox_read_stop' in globals(): xbox_read_stop.set()
if 'xbox_process' in globals() and xbox_process is not None and xbox_process.poll() is None:
    xbox_process.terminate()
if 'xbox_read_thread' in globals() and xbox_read_thread is not None and xbox_read_thread.is_alive():
    xbox_read_thread.join(timeout=2)

control_stop, xbox_watch_stop = threading.Event(), threading.Event()
control_thread = xbox_watch_thread = None
xbox_pygame_lock = threading.RLock()
bindings_lock = threading.RLock()
xbox_read_stop=threading.Event()
xbox_read_thread=None
xbox_process=None
latest_controls=None
reader_error=''
button_test_active=False
connection_snapshot = {'at':0., 'connected':False, 'name':'', 'index':-1,
                       'axes':0, 'buttons':0, 'hats':0}
awaiting_input = None
capture_baseline = None
suppress_capture = None

# A função de cada comando é fixa; 'Configurar' corrige apenas o índice físico
# indicado pelo driver do controle e registra um nome próprio para o comando.
INPUTS = {
 'lt':          ('LT',       'Aceleração proporcional das esteiras', 'axis',4),
 'drive_x':     ('Stick E X','Direção/curva das esteiras',           'axis',0),
 'drive_y':     ('Stick E Y','Avanço e ré das esteiras',             'axis',1),
 'lb':          ('LB',       'Freio das esteiras enquanto pressionado','button',4),
 'rt':          ('RT',       'Pressão proporcional da água',          'axis',5),
 'rb':          ('RB',       'Avançar atuador; focalizar jato',        'button',5),
 'r3':          ('R3',       'Clique do stick direito: recuar atuador', 'button',9),
 'cannon_x':    ('Stick D X','Mira horizontal; pan acompanha borda', 'axis',2),
 'cannon_y':    ('Stick D Y','Mira vertical; tilt acompanha borda',  'axis',3),
 'mode':        ('X',        'Alternar canhão manual/autônomo',       'button',2),
 'cool_internal':('A',      'Ligar/desligar arrefecimento interno', 'button',0),
 'cool_external':('B',      'Ligar/desligar arrefecimento externo', 'button',1),
 'leds':        ('Y',        'Ligar/desligar LEDs',                   'button',3),
 'camera_left': ('D-pad ←','Girar plataforma para 0° no manual',  'hat',0),
 'camera_right':('D-pad →','Girar plataforma para 180° no manual','hat',0),
 'overlay':     ('D-pad ↑', 'Mostrar/ocultar informações',           'hat',0),
 'link_hold':   ('Start',    'Conectar/desconectar: segurar 2 s',    'button',7),
 'stop':        ('D-pad ↓','Parada de emergência: segurar 1 s',     'hat',0),
 'water':       ('View',     'Ligar/desligar fornecimento de água',  'button',6),
}
DEFAULT_BINDINGS = {k:{'kind':kind,'index':i,**({'direction':[-1,0]} if k=='camera_left' else
                     {'direction':[1,0]} if k=='camera_right' else
                    {'direction':[0,1]} if k=='overlay' else
                    {'direction':[0,-1]} if k=='stop' else {})}
                    for k,(_,_,kind,i) in INPUTS.items()}
_saved_control = cfg.data.get('control', {})
bindings = {k:dict(_saved_control.get('input_bindings',{}).get(k,DEFAULT_BINDINGS[k])) for k in INPUTS}
if int(_saved_control.get('schema_version',0))<3:
    bindings['r3']=dict(DEFAULT_BINDINGS['r3'])
if int(_saved_control.get('schema_version',0))<4:
    for _key in ('camera_left','camera_right','stop'):
        bindings[_key]=dict(DEFAULT_BINDINGS[_key])
for _key,_legacy in [('drive_x','drive_x'),('drive_y','drive_y'),
                     ('cannon_x','cannon_x'),('cannon_y','cannon_y')]:
    if _key not in _saved_control.get('input_bindings',{}) and _legacy in cfg.data.get('xbox_axes',{}):
        bindings[_key]={'kind':'axis','index':int(cfg.data['xbox_axes'][_legacy])}
# O esquema 4 reserva ←/→ para a câmera e ↓ para emergência.
binding_warning=('Direções da plataforma e parada foram atualizadas; teste e salve.'
                 if int(_saved_control.get('schema_version',0))<4 else '')
input_names={k:_saved_control.get('button_names',{}).get(k, label) for k,(label,*_) in INPUTS.items()}
if int(_saved_control.get('schema_version',0))<4:
    for _key in ('stop','camera_left','camera_right'):input_names[_key]=INPUTS[_key][0]
if int(_saved_control.get('schema_version',0))<3:input_names['r3']=INPUTS['r3'][0]

connection_status=widgets.HTML(status_box('Controle Xbox','PENDENTE',
 'A leitura começa ao iniciar o teste de botões.').value,
 layout=widgets.Layout(width='100%'))
control_status=widgets.HTML(status_box('Simulação','PENDENTE',
 'Conecte um Xbox real e clique Iniciar simulação.').value,
 layout=widgets.Layout(width='100%'))
map_status=widgets.HTML(status_box('Mapeamento','PENDENTE',binding_warning).value if binding_warning else '')
speed_limit=widgets.IntSlider(value=max(10,min(80,int(_saved_control.get('speed_limit_rpm',80)))),
 min=10,max=80,step=1,description='Teto esteiras:',
 style={'description_width':'135px'},layout=widgets.Layout(width='565px'))
camera_scan_speed=widgets.IntSlider(value=int(_saved_control.get('camera_scan_speed_deg_s',12)),
 min=2,max=45,description='Varredura auto (°/s):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_manual_speed=widgets.IntSlider(value=int(_saved_control.get('camera_manual_speed_deg_s',45)),
 min=5,max=90,description='Giro manual (°/s):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_edge_offset=widgets.IntSlider(value=int(_saved_control.get('camera_edge_offset_px',8)),
 min=2,max=30,description='Margem da borda (px):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
camera_recenter_step=widgets.IntSlider(value=int(_saved_control.get('camera_recenter_step_deg',20)),
 min=5,max=50,description='Passo recentrar (°):',style={'description_width':'155px'},layout=widgets.Layout(width='540px'))
control_name=widgets.Text(value='XBOX360',description='Nome do setup:',layout=widgets.Layout(width='330px'))
control_pick=widgets.Dropdown(options=[('Novo setup','')]+[(n,n) for n in setup_names('control')],
 description='Salvos:',layout=widgets.Layout(width='420px'))
control_save=widgets.Button(description='Salvar/atualizar',button_style='success',layout=widgets.Layout(width='155px'))
control_load=widgets.Button(description='Carregar para editar',button_style='info',layout=widgets.Layout(width='170px'))
control_delete=widgets.Button(description='Excluir',button_style='danger',layout=widgets.Layout(width='95px'))
control_start=widgets.Button(description='Iniciar simulação',button_style='success',layout=widgets.Layout(width='165px'))
control_pause=widgets.Button(description='Parar simulação',button_style='warning',layout=widgets.Layout(width='150px'))
scene_image=widgets.Image(format='jpeg',layout=widgets.Layout(width='100%',max_width='1500px'))
reference_panel=widgets.HTML('',layout=widgets.Layout(width='100%'))
state_panel=widgets.HTML('',layout=widgets.Layout(width='100%'))
diagnostic=widgets.HTML('',layout=widgets.Layout(width='100%',height='480px',overflow='auto'))
diagnostic_reset=widgets.Button(description='Limpar leituras',layout=widgets.Layout(width='145px'))
diagnostic_start=widgets.Button(description='Testar todos os botões',button_style='info',layout=widgets.Layout(width='195px'))
diagnostic_stop=widgets.Button(description='Parar teste',button_style='warning',layout=widgets.Layout(width='115px'))
diagnostic_status=widgets.HTML(status_box('Leitura dos botões','PENDENTE',
 'Inicie o teste; cada entrada recebida ficará verde.').value)
diagnostic_seen={key:False for key in INPUTS}
raw_seen={'buttons':set(),'hats':set(),'axes':set()}
raw_snapshot={'buttons':[],'hats':[],'axes':[],'rest_axes':[]}
last_raw_input='Nenhum sinal bruto recebido ainda.'
message=widgets.HTML('<span style="color:#8b5e00">Aguardando controle.</span>',
 layout=widgets.Layout(width='100%'))
input_rows={}

def physical_label(binding):
    kind=binding.get('kind','?'); index=binding.get('index','?')
    if kind=='hat': return f'D-pad {index} {tuple(binding.get("direction",[]))}'
    return f'{"eixo" if kind=="axis" else "botão"} {index}'

def reset_diagnostic(_=None):
    global last_raw_input
    for key in diagnostic_seen:diagnostic_seen[key]=False
    for seen in raw_seen.values():seen.clear()
    last_raw_input='Nenhum sinal bruto recebido ainda.'
    update_diagnostic(zero_controls(),time.monotonic(),bool(connection_snapshot['connected']))

def update_diagnostic(controls,now,available):
    # Verde permanece depois de um recebimento; Limpar leituras reinicia o teste.
    rows=[]
    for key,(label,_,kind,_) in INPUTS.items():
        value=controls.get(key,False)
        active=(abs(float(value))>.13 if kind=='axis' and key not in ('lt','rt') else
                float(value)>.08 if kind=='axis' else bool(value))
        if active:diagnostic_seen[key]=True
        received=diagnostic_seen[key]
        color='#15803d' if received else '#9ca3af'
        state_label='Recebido' if received else 'Aguardando'
        amount=f' · {float(value):+.2f}' if kind=='axis' and active else ''
        rows.append('<div style="padding:5px 3px;border:1px solid #e5e7eb;border-radius:5px;font-size:12px">'
          '<span style="display:inline-block;width:11px;height:11px;border-radius:50%;'
          f'background:{color};margin-right:7px"></span><b>{html.escape(label)}</b> '
          f'<span style="color:{color}">{state_label}{amount}</span>'
          f'<br><small style="margin-left:19px;color:#6b7280">{html.escape(physical_label(bindings[key]))}</small></div>')
    raw_rows=[]
    def raw_chip(label,received,active=False):
        color='#15803d' if received else '#9ca3af'
        background='#dcfce7' if active else '#f9fafb'
        return ('<div style="font-size:11px;padding:4px;border:1px solid #e5e7eb;'
          f'border-radius:4px;background:{background}">'
          f'<span style="color:{color}">●</span> {html.escape(label)} '
          f'<small>{"recebido" if received else "aguardando"}</small></div>')
    for i,v in enumerate(raw_snapshot['buttons']):
        raw_rows.append(raw_chip(f'Botão {i}',i in raw_seen['buttons'],bool(v)))
    for i,hat in enumerate(raw_snapshot['hats']):
        for direction,label in (((-1,0),'←'),((1,0),'→'),((0,1),'↑'),((0,-1),'↓')):
            active=(direction[0]!=0 and hat[0]==direction[0] or
                    direction[1]!=0 and hat[1]==direction[1])
            raw_rows.append(raw_chip(f'D-pad {i} {label}',(i,direction) in raw_seen['hats'],active))
    for i,v in enumerate(raw_snapshot['axes']):
        active=(i<len(raw_snapshot['rest_axes']) and abs(v-raw_snapshot['rest_axes'][i])>.14)
        raw_rows.append(raw_chip(f'Eixo {i}: {v:+.2f}',i in raw_seen['axes'],active))
    hold=state['link_hold_since']
    progress=(min(2.,max(0.,now-hold)) if hold is not None and controls.get('link_hold') else 0.)
    simulation_running=control_thread is not None and control_thread.is_alive()
    link=('Conexão simulada ativa' if state['connected'] else
          f'Start detectado: {progress:.1f}/2,0 s' if controls.get('link_hold') and simulation_running else
          'Start recebido; contagem de 2 s ocorre na simulação' if controls.get('link_hold') else
          'Aguardando Start')
    if controls.get('stop') and state.get('stop_hold_since') is not None:
        link+=f' · Parada: {min(1.,max(0.,now-state["stop_hold_since"])):.1f}/1,0 s'
    info='OK: Xbox disponível' if available else ('Falha: '+reader_error if reader_error else 'Aguardando leitura do Xbox.')
    diagnostic.value=('<div style="background:#f8fafc;border:1px solid #cbd5e1;'
      'border-radius:8px;padding:12px;font-family:Arial,sans-serif">'
      '<b>Teste de todos os comandos</b><br><small>'+html.escape(info)+'</small><br>'
      '<span style="color:#155e75;font-size:12px">'+html.escape(link)+'</span><br>'
      '<span style="color:#92400e;font-size:11px">'+html.escape(last_raw_input)+'</span><hr>'+
      '<b style="font-size:12px">Funções configuradas</b>'
      '<div style="display:grid;grid-template-columns:repeat(3,minmax(170px,1fr));gap:5px">'+
      ''.join(rows)+'</div><br><b style="font-size:12px">Todas as entradas físicas detectadas</b>'
      '<div style="display:grid;grid-template-columns:repeat(6,minmax(95px,1fr));gap:4px">'+
      (''.join(raw_rows) if raw_rows else '<small>Aguardando contagem de botões do pygame.</small>')+
      '</div></div>')

def mark_diagnostic(controls):
    for key,(_,_,kind,_) in INPUTS.items():
        value=controls.get(key,False)
        if (abs(float(value))>.13 if kind=='axis' and key not in ('lt','rt') else
                float(value)>.08 if kind=='axis' else bool(value)):
            diagnostic_seen[key]=True

diagnostic_reset.on_click(reset_diagnostic)

def update_input_row(key):
    row=input_rows[key]
    b=bindings[key]
    row['physical'].value=f'<span style="color:#255da4">{html.escape(physical_label(b))}</span>'

def start_capture(key):
    global awaiting_input,capture_baseline
    with bindings_lock:
        awaiting_input=key
        capture_baseline=None
    start_button_test(None)
    map_status.value=status_box('Captura','TESTANDO',
       f'Mova/aperte {INPUTS[key][0]} até o fim do curso; aguarde a leitura.').value

for _key,(_label,_meaning,_kind,_index) in INPUTS.items():
    _color='#dbeafe' if _kind=='axis' else '#fee2e2' if _key in ('stop','link_hold') else '#e9f7ed'
    _name=widgets.Text(value=str(input_names[_key]),placeholder='Nome deste comando',
                       layout=widgets.Layout(width='42%'))
    _physical=widgets.HTML('',layout=widgets.Layout(width='105px'))
    _capture=widgets.Button(description='Configurar',layout=widgets.Layout(width='95px',height='27px'))
    _capture.on_click(lambda _,key=_key:start_capture(key))
    _meaning_html=widgets.HTML('<span style="font-size:11px">'+html.escape(_meaning)+
       '</span><br><small style="color:#375273;font-size:10px">Comando pré-definido</small>',
       layout=widgets.Layout(width='100%'))
    _badge=widgets.HTML('<div style="background:'+_color+';border-radius:5px;padding:3px 6px;'
      'font-size:12px;font-weight:bold">'+html.escape(_label)+'</div>',
      layout=widgets.Layout(width='83px'))
    input_rows[_key]={'name':_name,'physical':_physical,'configure':_capture,
                      'widget':widgets.VBox([
                        widgets.HBox([_badge,_meaning_html],layout=widgets.Layout(width='100%')),
                        widgets.HBox([_name,_physical,_capture],
                                     layout=widgets.Layout(width='100%',align_items='center'))],
                        layout=widgets.Layout(width='100%',border='1px solid #cbd5e1',
                                              padding='4px',min_height='73px'))}
    update_input_row(_key)

# Mesmo método de aquisição isolada usado no teste da seção 2: SDL vive em
# outro processo, enquanto este kernel só interpreta as leituras serializadas.
XBOX_READER_SCRIPT = r'''
import json, pygame, sys, time
pygame.init()
pygame.joystick.init()
index=int(sys.argv[1])
count=pygame.joystick.get_count()
if not 0 <= index < count and count==1:
    index=0  # Um único controle presente pode ser selecionado automaticamente.
if not 0 <= index < count:
    print('ERR:' + f'pygame detectou {count} controle(s); índice selecionado {index} indisponível. Repita Buscar conexões na seção 2.',flush=True)
    sys.exit(2)
joy=pygame.joystick.Joystick(index)
joy.init()
print('INFO:' + json.dumps({'name':joy.get_name(),'index':index,'axes':joy.get_numaxes(),
 'buttons':joy.get_numbuttons(),'hats':joy.get_numhats()}),flush=True)
try:
    while True:
        pygame.event.pump()
        packet={'axes':[round(float(joy.get_axis(i)),4) for i in range(joy.get_numaxes())],
          'buttons':[int(joy.get_button(i)) for i in range(joy.get_numbuttons())],
          'hats':[list(joy.get_hat(i)) for i in range(joy.get_numhats())]}
        print('DATA:'+json.dumps(packet,separators=(',',':')),flush=True)
        time.sleep(.05)
finally:
    joy.quit()
    pygame.joystick.quit()
    pygame.quit()
'''

def probe_xbox_connection():
    with xbox_pygame_lock:
        snap=connection_snapshot.copy()
    good=snap['connected'] and time.monotonic()-snap['at']<.8
    if good:
        label=(f'{snap["name"]} · índice {snap["index"]} · {snap["axes"]} eixos · '
               f'{snap["buttons"]} botões · {snap["hats"]} D-pads. Dados recebidos às '
               +time.strftime('%H:%M:%S')+'.')
    else:label=reader_error or 'Aguardando dados do controle selecionado na seção 2.'
    connection_status.value=status_box('Controle Xbox','OK' if good else 'FALHA' if reader_error else 'TESTANDO',label).value

def ensure_xbox_reader():
    global xbox_read_thread,xbox_read_stop,reader_error
    if xbox_read_thread is not None and xbox_read_thread.is_alive():return
    if 'xbox_monitor' in globals() and xbox_monitor.get('process') is not None and \
            'stop_xbox_preview' in globals():
        stop_xbox_preview()  # A seção 2 libera a leitura antes da nova aquisição.
    xbox_read_stop=threading.Event();reader_error=''
    xbox_read_thread=threading.Thread(target=xbox_read_loop,args=(xbox_read_stop,),daemon=True)
    xbox_read_thread.start()

def start_button_test(_):
    global button_test_active
    button_test_active=True
    diagnostic_status.value=status_box('Leitura dos botões','TESTANDO',
        'Pressione os botões e mova os eixos; os sinais recebidos ficam verdes.').value
    ensure_xbox_reader()

def stop_button_test(_):
    global button_test_active
    button_test_active=False
    if control_thread is None or not control_thread.is_alive():
        xbox_read_stop.set()
        if xbox_process is not None and xbox_process.poll() is None:xbox_process.terminate()
    diagnostic_status.value=status_box('Leitura dos botões','PENDENTE','Teste interrompido.').value

# A captura é feita na mesma thread de leitura, impedindo duas consultas concorrentes ao SDL.
def capture_input(axes,pressed,hats,previous_pressed,previous_hats):
    global awaiting_input,capture_baseline,suppress_capture
    with bindings_lock:
        key=awaiting_input
        if key is None:return False
        if capture_baseline is None:
            capture_baseline=list(axes)
            return True
        candidates=[]
        for i in sorted(pressed-previous_pressed):candidates.append({'kind':'button','index':i})
        for i,hat in enumerate(hats):
            before=previous_hats[i] if i<len(previous_hats) else (0,0)
            if hat!=(0,0) and hat!=before:
                candidates.append({'kind':'hat','index':i,'direction':list(hat)})
        for i,value in enumerate(axes):
            base=capture_baseline[i] if i<len(capture_baseline) else 0.
            if abs(value-base)>.48:
                candidates.append({'kind':'axis','index':i,
                                   **({'rest':base,'active':value} if key in ('lt','rt') else {})})
        desired=INPUTS[key][2]
        allowed=('hat','button') if key in ('stop','camera_left','camera_right','overlay') else (desired,)
        candidates=[c for c in candidates if c['kind'] in allowed]
        if not candidates:return True
        candidate=candidates[0]
        suppress_capture=dict(candidate)
        physical=(candidate['kind'],candidate['index'],tuple(candidate.get('direction',[])))
        duplicate=[k for k,b in bindings.items() if k!=key and
          (b.get('kind'),b.get('index'),tuple(b.get('direction',[])))==physical]
        if duplicate:
            # Uma troca entre dois eixos/botões ocupados não exige uma entrada livre.
            other=duplicate[0]
            bindings[other]=dict(bindings[key])
            bindings[key]=candidate
            update_input_row(key);update_input_row(other)
            awaiting_input=None;capture_baseline=None
            map_status.value=status_box('Entradas trocadas','OK',
               f'{INPUTS[key][0]} recebeu {physical_label(candidate)}; '
               f'{INPUTS[other][0]} recebeu {physical_label(bindings[other])}. '
               'Confira as duas linhas antes de salvar.').value
        else:
            bindings[key]=candidate
            update_input_row(key)
            awaiting_input=None;capture_baseline=None
            map_status.value=status_box('Entrada capturada','OK',
               f'{INPUTS[key][0]} = {physical_label(candidate)}. Você pode editar o nome e salvar.').value
        return True

def _analog(key,axes):
    b=bindings[key];i=int(b.get('index',-1))
    return float(axes[i]) if b.get('kind')=='axis' and 0<=i<len(axes) else 0.

def _trigger(key,axes):
    b=bindings[key];i=int(b.get('index',-1))
    if b.get('kind')!='axis' or not 0<=i<len(axes):return 0.
    if b.get('rest_pending'):return 0.
    raw=float(axes[i]);rest=float(b.get('rest',-1.))
    active=float(b.get('active',1.))
    if abs(active-rest)<.1:return 0.
    return max(0.,min(1.,(raw-rest)/(active-rest)))

def _pressed(key,pressed,hats):
    b=bindings[key];i=int(b.get('index',-1))
    if b.get('kind')=='button':return i in pressed
    if b.get('kind')=='hat' and 0<=i<len(hats):
        direction=b.get('direction',[0,0]);return tuple(direction)==hats[i]
    return False

def xbox_read_loop(stop):
    global xbox_process,latest_controls,last_raw_input,reader_error,suppress_capture
    previous_pressed=set();previous_hats=[];previous_axes=[];info={};had_data=False;last_probe=0.
    try:
        index=int(controller.value) if controller.value is not None else -1
        process=subprocess.Popen([sys.executable,'-u','-c',XBOX_READER_SCRIPT,str(index)],
            stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,bufsize=1)
        xbox_process=process
        for line in process.stdout:
            if stop.is_set():break
            line=line.strip()
            if line.startswith('ERR:'):
                raise RuntimeError(line[4:])
            if line.startswith('INFO:'):
                info=json.loads(line[5:])
                actual=int(info.get('index',index))
                if actual!=index:
                    options=list(controller.options)
                    if not any((entry[1] if isinstance(entry,tuple) else entry)==actual for entry in options):
                        options.append((f'{info["name"]} (detectado automaticamente)',actual))
                        controller.options=options
                    controller.value=actual
                connection_status.value=status_box('Controle Xbox','OK',
                    f'{info["name"]}: dispositivo aberto; aguardando leituras.').value
                continue
            if not line.startswith('DATA:'):continue
            data=json.loads(line[5:]);now=time.monotonic();had_data=True
            axes=[float(x) for x in data.get('axes',[])]
            pressed={i for i,b in enumerate(data.get('buttons',[])) if b}
            hats=[tuple(h) for h in data.get('hats',[])]
            if len(raw_snapshot['rest_axes'])!=len(axes):
                raw_snapshot['rest_axes']=list(axes)
            raw_snapshot.update(axes=list(axes),buttons=list(data.get('buttons',[])),hats=list(hats))
            raw_seen['buttons'].update(pressed)
            for i,h in enumerate(hats):
                if h[0]:raw_seen['hats'].add((i,(int(math.copysign(1,h[0])),0)))
                if h[1]:raw_seen['hats'].add((i,(0,int(math.copysign(1,h[1])))))
            raw_seen['axes'].update(i for i,v in enumerate(axes)
                if abs(v-raw_snapshot['rest_axes'][i])>.14)
            newly_pressed=sorted(pressed-previous_pressed)
            changed_hats=[(i,h) for i,h in enumerate(hats) if h!=(0,0) and
                          (i>=len(previous_hats) or h!=previous_hats[i])]
            changed_axes=[i for i,v in enumerate(axes) if i<len(previous_axes) and
                          abs(v-previous_axes[i])>.35]
            if newly_pressed:
                i=newly_pressed[0]
                related=[INPUTS[k][0] for k,b in bindings.items() if b.get('kind')=='button'
                         and b.get('index')==i]
                last_raw_input=f'Último sinal físico: botão {i} → '+(', '.join(related) if related else 'não configurado')
            elif changed_hats:
                last_raw_input=f'Último sinal físico: D-pad {changed_hats[0][0]} {changed_hats[0][1]}'
            elif changed_axes:
                last_raw_input=f'Último sinal físico: eixo {changed_axes[0]} = {axes[changed_axes[0]]:+.2f}'
            captured=capture_input(axes,pressed,hats,previous_pressed,previous_hats)
            with bindings_lock:
                for key in ('lt','rt'):
                    b=bindings[key];idx=int(b.get('index',-1))
                    if b.get('kind')=='axis' and 'rest' not in b and 0<=idx<len(axes):
                        base=axes[idx];b.update(rest=base,active=1.,rest_pending=base>.6)
                    if b.get('rest_pending') and 0<=idx<len(axes) and axes[idx]<.5:
                        b['rest']=axes[idx];b['rest_pending']=False
                controls={key:(_analog(key,axes) if key in ('drive_x','drive_y','cannon_x','cannon_y')
                    else _trigger(key,axes) if key in ('lt','rt') else _pressed(key,pressed,hats))
                    for key in INPUTS}
            diagnostic_controls=dict(controls)
            if button_test_active:mark_diagnostic(diagnostic_controls)
            previous_pressed=set(pressed);previous_hats=list(hats);previous_axes=list(axes)
            if suppress_capture is not None:
                b=suppress_capture;i=int(b['index'])
                still=(i in pressed if b['kind']=='button' else
                       i<len(hats) and hats[i]==tuple(b.get('direction',[])) if b['kind']=='hat' else
                       i<len(axes) and abs(axes[i]-float(b.get('rest',0)))>.12)
                if not still:suppress_capture=None
            if captured or suppress_capture is not None:controls=zero_controls()
            with xbox_pygame_lock:
                latest_controls=dict(controls)
                connection_snapshot.update(at=now,connected=True,name=info.get('name','Xbox'),
                    index=info.get('index',index),axes=len(axes),buttons=len(data.get('buttons',[])),hats=len(hats))
            if now-last_probe>1.:
                probe_xbox_connection()
                last_probe=now
            if button_test_active and (newly_pressed or changed_hats or changed_axes or
                 not hasattr(xbox_read_loop,'last_display') or now-xbox_read_loop.last_display>.1):
                update_diagnostic(diagnostic_controls,now,True)
                xbox_read_loop.last_display=now
                diagnostic_status.value=status_box('Leitura dos botões','OK',
                    'Dados chegando do pygame; pressione cada entrada para marcar Recebido.').value
        if not stop.is_set():
            details=(process.stderr.read()[-300:] if process.stderr else '').strip()
            raise RuntimeError(reader_error or details or
                'O processo de leitura terminou sem dados; verifique o controle/driver.')
    except Exception as exc:
        if not stop.is_set():
            reader_error=f'{type(exc).__name__}: {exc}'
            connection_status.value=status_box('Controle Xbox','FALHA',reader_error).value
            diagnostic_status.value=status_box('Leitura dos botões','FALHA',reader_error).value
            update_diagnostic(zero_controls(),time.monotonic(),False)
    finally:
        with xbox_pygame_lock:
            latest_controls=None
            connection_snapshot.update(at=time.monotonic(),connected=False,name='',index=-1,
                                      axes=0,buttons=0,hats=0)
        process=xbox_process;xbox_process=None
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=2)
            except subprocess.TimeoutExpired:process.kill()

def zero_controls():
    return {k:False for k in INPUTS if INPUTS[k][2]!='axis'} | {
     'drive_x':0.,'drive_y':0.,'cannon_x':0.,'cannon_y':0.,'lt':0.,'rt':0.}

def random_field_objects():
    """Dois marcos fixos por ensaio, gerados fora do chassi e do painel da mira."""
    rng=random.SystemRandom()
    first=(rng.randint(300,790),rng.randint(100,310))
    while True:
        second=(rng.randint(135,950),rng.randint(110,700))
        if math.dist(first,second)>330 and math.dist(second,(750,470))>185:
            break
    return [{'name':name,'x':xy[0],'y':xy[1],'color':color}
            for name,xy,color in [('A',first,(90,218,251)),
                                  ('B',second,(208,132,245))]]

def new_control_state():
    return {'objects':random_field_objects(),'connected':False,'emergency':False,'mode':'manual',
      'cool_internal':False,'cool_external':False,'leds':False,'water':False,
      'overlay':False,'camera_pan_deg':90.,'scan_direction':1,
      'last_recenter':0.,'focus':0.,'pan':0.,'tilt':45.,
      'aim_x':0.,'aim_y':0.,'x':750.,'y':470.,'heading':0.,
      'left_rpm':0.,'right_rpm':0.,'v_mps':0.,'omega':0.,
      'throttle':0.,'pressure':0.,'link_hold_since':None,'hold_latched':False,
      'stop_hold_since':None,'stop_latched':False,
      'previous':{},'hint':'Segure Start por 2 s para conectar o robô (simulado).'}
state=new_control_state()

def _deadzone(value,threshold=.08):
    value=float(value)
    return 0. if abs(value)<threshold else max(-1.,min(1.,value))

def apply_control(s, c, now, dt, rpm_limit, geometry):
    """Simula a máquina de estados; não envia sinais ao ESP32."""
    dt=max(0.,min(.12,float(dt)))
    prev=s['previous'];rising=lambda key: bool(c.get(key)) and not prev.get(key,False)
    if c.get('stop'):
        if s['stop_hold_since'] is None:s['stop_hold_since']=now
        if not s['stop_latched'] and now-s['stop_hold_since']>=1.:
            s.update(connected=False,emergency=True,water=False,leds=False,
                     cool_internal=False,cool_external=False,throttle=0.,pressure=0.,
                     left_rpm=0.,right_rpm=0.)
            s['hint']='PARADA DE EMERGÊNCIA: solte a seta para baixo e segure Start 2 s para rearmar.'
            s['hold_latched']=True
            s['stop_latched']=True
    else:
        s['stop_hold_since']=None;s['stop_latched']=False
    start=bool(c.get('link_hold'))
    if start:
        if s['link_hold_since'] is None:s['link_hold_since']=now
        if not s['hold_latched'] and now-s['link_hold_since']>=2.:
            neutral=(abs(float(c.get('drive_x',0)))<.08 and abs(float(c.get('drive_y',0)))<.08
                      and float(c.get('lt',0))<.08 and float(c.get('rt',0))<.08
                      and not c.get('stop'))
            if not neutral:
                s['hint']='Solte LT/RT e a seta para baixo; centralize os sticks para conectar/rearmar.'
            elif s['connected']:
                s.update(connected=False,water=False,pressure=0.,throttle=0.,
                         cool_internal=False,cool_external=False,leds=False)
                s['hint']='Conexão cortada. Segure Start por 2 s para reconectar.'
                s['hold_latched']=True
            else:
                s['connected']=True;s['emergency']=False
                s['hint']='Conectado (simulado). View libera água; LT acelera; LB freia.'
                s['hold_latched']=True
    else:s['link_hold_since']=None;s['hold_latched']=False
    if not s['connected']:
        s['throttle']=s['pressure']=s['v_mps']=s['omega']=0.
        s['left_rpm']=s['right_rpm']=0.
        if not s['emergency'] and not start and (float(c.get('lt',0))>.08 or
             float(c.get('rt',0))>.08 or any(rising(k) for k in ('water','mode','rb','r3'))):
            s['hint']='Primeiro segure Start por 2 s para conectar.'
    else:
        for key,field in (('mode','mode'),('water','water'),('cool_internal','cool_internal'),
                          ('cool_external','cool_external'),('leds','leds'),('overlay','overlay')):
            if not rising(key):continue
            if key=='mode':s['mode']='auto' if s['mode']=='manual' else 'manual'
            else:s[field]=not s[field]
            s['hint']=f'{INPUTS[key][0]}: {INPUTS[key][1]}.'
        if c.get('rb') and c.get('r3'):
            s['hint']='Solte RB ou R3: focalizar e dispersar são comandos opostos.'
        elif c.get('rb'):s['focus']=min(1.,s['focus']+.35*dt)
        elif c.get('r3'):s['focus']=max(0.,s['focus']-.35*dt)
        throttle=max(0.,min(1.,float(c.get('lt',0))))
        if c.get('lb'):
            if throttle>.08:s['hint']='LB acionado: solte o freio para acelerar.'
            throttle=0.
        x=_deadzone(c.get('drive_x',0));y=-_deadzone(c.get('drive_y',0))
        s['throttle']=throttle
        left=throttle*max(-1.,min(1.,y+x))
        right=throttle*max(-1.,min(1.,y-x))
        s['left_rpm']=rpm_limit*left;s['right_rpm']=rpm_limit*right
        pressure=max(0.,min(1.,float(c.get('rt',0))))
        s['pressure']=pressure if s['water'] else 0.
        if pressure>.08 and not s['water']:
            s['hint']='Aperte View para liberar água antes de pressionar RT.'
        radius=float(geometry.get('sprocket_radius_m') or 0)
        width=float(geometry.get('track_width_m') or 0)
        if radius>0 and width>0:
            vl=left*rpm_limit*2*math.pi*radius/60
            vr=right*rpm_limit*2*math.pi*radius/60
            s['v_mps']=(vl+vr)/2;s['omega']=(vr-vl)/width
            # Ângulo positivo na tela aponta à direita; yaw cinemático positivo é anti-horário.
            s['heading']-=s['omega']*dt
            travel=s['v_mps']*115*dt
        else:
            s['v_mps']=s['omega']=0.
            s['heading']-=(right-left)*1.2*dt
            travel=(left+right)*42*dt
        s['x']=(s['x']+math.sin(s['heading'])*travel-110)%1280+110
        s['y']=(s['y']-math.cos(s['heading'])*travel-90)%650+90
        if s['mode']=='manual':
            direction=int(c.get('camera_right',False))-int(c.get('camera_left',False))
            (s['pan'],s['tilt'],s['camera_pan_deg'],s['aim_x'],s['aim_y']) = coupled_manual_pose(
                s['pan'],s['tilt'],s['camera_pan_deg'],
                _deadzone(c.get('cannon_x',0)),_deadzone(c.get('cannon_y',0)),
                direction,dt,camera_manual_speed.value,camera_edge_offset.value,370.,
                camera_recenter_step.value)
        if s['mode']=='auto' and not c.get('stop'):
            s['camera_pan_deg']+=s['scan_direction']*camera_scan_speed.value*dt
            if s['camera_pan_deg']>=180:s['camera_pan_deg']=180.;s['scan_direction']=-1
            elif s['camera_pan_deg']<=0:s['camera_pan_deg']=0.;s['scan_direction']=1
    s['previous']={k:bool(c.get(k)) for k in INPUTS if INPUTS[k][2]!='axis'}
    return s

# Composição em pixels: figura única para usar toda a largura disponível.
def _line(img,a,b,color,thick=1):
    cv2.line(img,tuple(map(int,a)),tuple(map(int,b)),color,thick,cv2.LINE_AA)
def _text(img,value,point,scale=.5,color=(220,230,238),thick=1):
    cv2.putText(img,str(value),tuple(map(int,point)),cv2.FONT_HERSHEY_SIMPLEX,
                scale,color,thick,cv2.LINE_AA)
def _ray(origin,heading,distance):
    return (origin[0]+math.sin(heading)*distance,origin[1]-math.cos(heading)*distance)
def _cone(overlay,origin,angle,length,half_angle,color):
    points=[(int(origin[0]),int(origin[1]))]
    for a in np.linspace(angle-half_angle,angle+half_angle,17):
        points.append(tuple(map(int,_ray(origin,float(a),length))))
    cv2.fillPoly(overlay,[np.asarray(points,np.int32)],color)

def project_field_objects(s, sx0, sx1, sy0, sy1):
    """Projeta objetos fixos do mapa no quadro de 120° da câmera."""
    camera_angle=float(s['heading'])+math.radians(float(s['camera_pan_deg'])-90.)
    half_fov=math.radians(60.)
    projected=[]
    for obj in s['objects']:
        dx=float(obj['x'])-float(s['x']);dy=float(obj['y'])-float(s['y'])
        bearing=math.atan2(dx,-dy)
        relative=(bearing-camera_angle+math.pi)%(2*math.pi)-math.pi
        if abs(relative)>half_fov:continue
        distance=math.hypot(dx,dy)
        px=int((sx0+sx1)/2+relative/half_fov*(sx1-sx0)/2)
        py=int(sy1-18-min(1.,distance/1050.)*(sy1-sy0-37))
        projected.append((obj,px,py))
    return projected

def draw_control_scene(s,c):
    image=np.full((790,1500,3),(29,35,45),dtype=np.uint8)
    # O grid começa logo abaixo do título, sem a faixa vazia vista na imagem anterior.
    for x in range(30,1500,80):_line(image,(x,37),(x,790),(42,51,62))
    for y in range(37,790,80):_line(image,(0,y),(1500,y),(42,51,62))
    _text(image,'SIMULACAO DO ROBO E DO CANHAO',(18,26),.62,(235,244,250),2)
    pos=(float(s['x']),float(s['y']));h=float(s['heading']);front=_ray(pos,h,60)
    camera_angle=h+math.radians(float(s['camera_pan_deg'])-90.)
    half_fov=math.radians(60.)
    field=image.copy()
    _cone(field,pos,camera_angle,1900.,half_fov,(218,225,231))
    image=cv2.addWeighted(field,.11,image,.89,0)
    for edge in (camera_angle-half_fov,camera_angle+half_fov):
        _line(image,pos,_ray(pos,edge,1900),(230,238,245),1)
    _line(image,pos,_ray(pos,camera_angle,135),(235,245,253),1)
    for obj in s['objects']:
        point=(int(obj['x']),int(obj['y']))
        cv2.circle(image,point,17,obj['color'],-1,cv2.LINE_AA)
        cv2.circle(image,point,20,(240,245,251),1,cv2.LINE_AA)
        _text(image,'OBJETO '+obj['name'],(point[0]+22,point[1]-9),.44,obj['color'])
    # Cones transparentes: três faróis frontais e dois laterais.
    lights=image.copy()
    if s['leds']:
        for shift in (-.2,0,.2):_cone(lights,front,h+shift,190,.15,(30,225,250))
        for side in (-1,1):_cone(lights,pos,h+side*math.pi/2,135,.23,(30,225,250))
    image=cv2.addWeighted(lights,.24,image,.76,0)
    # Cabos de comunicação e água seguem a traseira, conforme os dois estados.
    rear=_ray(pos,h+math.pi,58)
    lateral=np.array([math.cos(h),math.sin(h)])
    for active,shift,source,color in (
        (s['connected'],-10,(0,690),(65,222,105)),
        (s['water'] and s['connected'],10,(0,718),(245,166,48))):
        if not active:continue
        target=np.array(rear)+lateral*shift
        bend1=(max(45,int(target[0]*.36)),int(source[1]*.75+target[1]*.25))
        bend2=(max(60,int(target[0]*.75)),int(source[1]*.20+target[1]*.80))
        path=np.asarray([source,bend1,bend2,tuple(target.astype(int))],np.int32)
        cv2.polylines(image,[path],False,(18,29,35),8,cv2.LINE_AA)
        cv2.polylines(image,[path],False,color,4,cv2.LINE_AA)
        cv2.circle(image,tuple(target.astype(int)),5,color,-1,cv2.LINE_AA)
    # Carrinho de esteiras rotacionado pelo rumo integrado.
    ux=np.array([math.sin(h),-math.cos(h)]);side=np.array([math.cos(h),math.sin(h)])
    center=np.array(pos)
    def footprint(half_w,half_l,offset=0):
        p=center+side*offset
        return np.array([p-side*half_w-ux*half_l,p+side*half_w-ux*half_l,
                         p+side*half_w+ux*half_l,p-side*half_w+ux*half_l],np.int32)
    cv2.fillPoly(image,[footprint(34,55,-35),footprint(34,55,35)],(72,84,95))
    cv2.fillPoly(image,[footprint(42,43)],(116,142,159))
    cv2.polylines(image,[footprint(42,43)],True,(212,228,236),2,cv2.LINE_AA)
    _line(image,pos,front,(238,245,250),3)
    cv2.circle(image,tuple(map(int,pos)),8,(241,249,253),-1,cv2.LINE_AA)
    # Arrefecimento: partículas azuis ao redor (externo) ou sobre o chassi (interno).
    particles=image.copy()
    if s['cool_external']:
        for i in range(110):
            angle=i*2.39996
            radius=69+39*(.5+.5*math.sin(i*5.71))
            point=_ray(pos,angle,radius)
            cv2.circle(particles,tuple(map(int,point)),2+i%3,(255,178,52),-1,cv2.LINE_AA)
    if s['cool_internal']:
        for i in range(58):
            local_x=32*math.sin(i*2.39996)
            local_y=35*math.sin(i*3.828)
            point=center+side*local_x+ux*local_y
            cv2.circle(particles,tuple(point.astype(int)),2+i%3,(255,205,95),-1,cv2.LINE_AA)
    if s['cool_external'] or s['cool_internal']:
        image=cv2.addWeighted(particles,.72,image,.28,0)
    # Vetor laranja: aceleração e direção; cone azul: jato e focalização.
    drive_amount=float(s['throttle'])*min(1.,math.hypot(
        float(c.get('drive_x',0)),float(c.get('drive_y',0))))
    heading_vector=h+math.atan2(float(c.get('drive_x',0)),
                           max(.1,abs(float(c.get('drive_y',0)))))
    if float(c.get('drive_y',0))>0:heading_vector+=math.pi
    if drive_amount>.01:
        tip=_ray(pos,heading_vector,170*drive_amount)
        cv2.arrowedLine(image,tuple(map(int,pos)),tuple(map(int,tip)),(40,170,255),
                        3,cv2.LINE_AA,tipLength=.16)
    if s['water'] and s['connected']:
        pressure=float(s['pressure'])
        blue_angle=h+math.radians(float(s['pan']))
        spread=.035+.38*(1.-float(s['focus']))
        length=115+220*pressure
        jet=image.copy()
        _cone(jet,front,blue_angle,length,spread,(245,167,48))
        image=cv2.addWeighted(jet,.48 if pressure>.02 else .15,
                              image,.52 if pressure>.02 else .85,0)
        for edge in (blue_angle-spread,blue_angle+spread):
            _line(image,front,_ray(front,edge,length),(255,193,92),2 if pressure>.02 else 1)
    # Estados e sensores aparecem sempre no painel lateral, fora da imagem.
    # Janela de mira, escala de panorama e inclinação; retículo contido em ±62%.
    bx0,by0,bx1,by1=1040,205,1480,500
    cv2.rectangle(image,(bx0,by0),(bx1,by1),(16,28,43),-1)
    cv2.rectangle(image,(bx0,by0),(bx1,by1),(170,188,210),2)
    horizon=int(350+(float(s['tilt'])-45.)*1.4)
    _line(image,(bx0+25,horizon),(bx1-25,horizon),(140,170,195),2)
    _text(image,'CAMERA 120° / MIRA',(bx0+12,by0+24),.48,(223,236,250))
    sx0,sx1=bx0+47,bx1-23
    sy0,sy1=by0+44,by1-57
    for deg in range(-60,61,10):
        tick=int(sx0+(deg+60)*(sx1-sx0)/120)
        major=deg%30==0
        _line(image,(tick,sy1),(tick,sy1+(9 if major else 4)),
              (152,177,201) if major else (82,105,130),1)
        if major:_text(image,f'{deg:+d}',(tick-14,by1-30),.43,(186,202,222))
    # Projeção angular: girar a câmera desloca os objetos no quadro menor;
    # deslocar ou girar o robô muda a direção e a distância relativas.
    for obj,px,py in project_field_objects(s,sx0,sx1,sy0,sy1):
        cv2.circle(image,(px,py),9,obj['color'],-1,cv2.LINE_AA)
        cv2.circle(image,(px,py),11,(245,250,255),1,cv2.LINE_AA)
        _text(image,obj['name'],(px+13,py+5),.43,(244,247,252))
    for deg in range(0,91,10):
        tick=int(sy1-deg*(sy1-sy0)/90)
        _line(image,(sx0-(9 if deg%30==0 else 4),tick),(sx0,tick),
              (152,177,201) if deg%30==0 else (82,105,130),1)
        _text(image,str(deg),(bx0+9,tick+4),.44,(186,202,222))
    pan_tick=int((sx0+sx1)/2+s['aim_x']*(sx1-sx0)/2)
    tilt_tick=int(sy1-s['tilt']*(sy1-sy0)/90)
    cv2.circle(image,(pan_tick,sy1+2),4,(45,211,245),-1,cv2.LINE_AA)
    cv2.circle(image,(sx0-3,tilt_tick),4,(45,211,245),-1,cv2.LINE_AA)
    _text(image,f'CANHAO {s["pan"]:+.0f}  CAMERA {s["camera_pan_deg"]:.0f}',(bx0+115,by1-9),.42,(244,196,95))
    cv2.rectangle(image,(bx0+44,by0+38),(bx1-19,by1-54),(96,122,149),1)
    mx=int((sx0+sx1)/2+s['aim_x']*(sx1-sx0)/2)
    my=int((sy0+sy1)/2+s['aim_y']*(sy1-sy0)/2)
    cv2.circle(image,(mx,my),10,(50,230,120),2,cv2.LINE_AA)
    _line(image,(mx-23,my),(mx+23,my),(70,235,125),2)
    _line(image,(mx,my-23),(mx,my+23),(70,235,125),2)
    _text(image,f'TILT {s["tilt"]:.0f} graus',(bx0+308,by0+55),.42,(216,226,241))
    if s['overlay']:
        _text(image,'Vetor laranja: tracao   Cone azul: jato   Cabos: verde=comunicacao, azul=agua',
              (20,765),.48)
        _text(image,'Pontos azuis: arrefecimento',(1110,765),.42)
    image=cv2.resize(image,(1080,569),interpolation=cv2.INTER_AREA)
    ok,encoded=cv2.imencode('.jpg',image,[cv2.IMWRITE_JPEG_QUALITY,85])
    return encoded.tobytes() if ok else b''

def update_sim_panels(s,c,now):
    refs=[]
    for key,(label,meaning,kind,_) in INPUTS.items():
        custom=input_rows[key]['name'].value.strip() or label
        # Função em destaque; índice físico fica somente no teste de botões.
        active=(abs(float(c.get(key,0)))>.08 if kind=='axis' else bool(c.get(key)))
        bg='#e7f7e9' if active else '#f8fafc'
        refs.append('<div style="padding:4px 6px;background:'+bg+';border:1px solid #e5e7eb;'
                    'border-radius:5px;font-size:11px">'
                    f'<b>{html.escape(meaning)}</b><br>'
                    f'<small style="color:#3f5568">{html.escape(custom)}</small></div>')
    reference_panel.value=('<div style="font-family:Arial,sans-serif;background:#f8fafc;'
       'border:1px solid #cbd5e1;padding:7px;border-radius:6px">'
       '<b style="font-size:14px">Referência dos comandos</b>'
       '<div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:5px">'+
       ''.join(refs)+'</div></div>')
    telemetry=results.get('enlace',{}) if isinstance(results,dict) else {}
    if not isinstance(telemetry,dict):telemetry={}
    last=telemetry.get('latest',telemetry)
    if not isinstance(last,dict):last={}
    temperature=last.get('temperature_c',last.get('temperature','--'))
    humidity=last.get('humidity_pct',last.get('humidity','--'))
    link_elapsed=(f'{min(2.,now-s["link_hold_since"]):.1f}/2,0 s'
                  if s['link_hold_since'] is not None else '0/2,0 s')
    stop_elapsed=(f'{min(1.,now-s["stop_hold_since"]):.1f}/1,0 s'
                  if s['stop_hold_since'] is not None else '0/1,0 s')
    on=lambda value:'Ligado' if value else 'Desligado'
    toggle=lambda value:'green' if value else 'red'
    moving=bool(s['throttle']>.02 and
                (abs(float(c.get('drive_x',0)))>.08 or abs(float(c.get('drive_y',0)))>.08))
    aim_active=bool(s['mode']=='manual' and
        (abs(float(c.get('cannon_x',0)))>.08 or abs(float(c.get('cannon_y',0)))>.08))
    rows=[
      ('Conexão simulada',on(s['connected']),toggle(s['connected'])),
      ('Start pressionado',link_elapsed,'green' if c.get('link_hold') else ''),
      ('Parada de emergência',on(s['emergency']),'red' if s['emergency'] else 'green'),
      ('Seta para baixo pressionada',stop_elapsed,'red' if c.get('stop') else ''),
      ('Modo do canhão','Automático' if s['mode']=='auto' else 'Manual',
                         'green' if s['mode']=='auto' else 'red'),
      ('Fornecimento de água',on(s['water']),toggle(s['water'])),
      ('Pressão comandada',f'{100*s["pressure"]:.0f}%',
                           'green' if s['pressure']>.02 else 'red' if c.get('rt',0)>.08 else ''),
      ('Aceleração LT',f'{100*s["throttle"]:.0f}%',
                       'green' if s['throttle']>.02 else 'red' if c.get('lb') else ''),
      ('Freio LB',on(c.get('lb')),'red' if c.get('lb') else ''),
      ('Limite das esteiras',f'{speed_limit.value} RPM',''),
      ('Esteira esquerda',f'{s["left_rpm"]:+.1f} RPM','green' if moving else ''),
      ('Esteira direita',f'{s["right_rpm"]:+.1f} RPM','green' if moving else ''),
      ('Velocidade linear',f'{s["v_mps"]:+.2f} m/s','green' if moving else ''),
      ('Giro do robô',f'{s["omega"]:+.2f} rad/s','green' if moving else ''),
      ('Posição no quadro',f'({s["x"]:.0f}, {s["y"]:.0f}) px',''),
      ('Rumo',f'{math.degrees(s["heading"])%360:.0f}°',''),
      ('Stick esquerdo X/Y',f'{float(c.get("drive_x",0)):+.2f} / {float(c.get("drive_y",0)):+.2f}',
                              'green' if moving else ''),
      ('Stick direito X/Y',f'{float(c.get("cannon_x",0)):+.2f} / {float(c.get("cannon_y",0)):+.2f}',
                              'green' if aim_active else ''),
      ('Pan do canhão',f'{s["pan"]:+.1f}°','green' if aim_active else ''),
      ('Tilt do canhão',f'{s["tilt"]:.1f}°','green' if aim_active else ''),
      ('Mira na janela',f'{s["aim_x"]:+.2f} / {s["aim_y"]:+.2f}',
                         'green' if aim_active else ''),
      ('Focalização do jato',f'{100*s["focus"]:.0f}%',
                              'green' if c.get('rb') else 'red' if c.get('r3') else ''),
      ('Avanço RB',on(c.get('rb')),'green' if c.get('rb') else ''),
      ('Recuo R3',on(c.get('r3')),'red' if c.get('r3') else ''),
      ('Arrefecimento interno',on(s['cool_internal']),toggle(s['cool_internal'])),
      ('Arrefecimento externo',on(s['cool_external']),toggle(s['cool_external'])),
      ('LEDs',on(s['leds']),toggle(s['leds'])),
      ('Plataforma da câmera',f'{s["camera_pan_deg"]:.1f}° (0–180)',
         'green' if c.get('camera_left') or c.get('camera_right') else ''),
      ('Legendas na imagem',on(s['overlay']),toggle(s['overlay'])),
      ('Temperatura medida',str(temperature),''),('Umidade medida',str(humidity),'')]
    colors={'green':('#e8f6e9','#12652d'),'red':('#fdeaea','#a02b2b'),
            '':('#ffffff','#273444')}
    lines=[]
    for label,value,tone in rows:
        bg,fg=colors[tone]
        lines.append('<div style="display:flex;justify-content:space-between;gap:6px;'
          f'background:{bg};color:{fg};border-bottom:1px solid #e5e7eb;'
          'padding:2px 5px;min-height:17px">'
          f'<span>{html.escape(label)}</span><b style="text-align:right">{html.escape(value)}</b></div>')
    state_panel.value=('<div style="font:12px Arial,sans-serif;background:#f8fafc;'
        'border:1px solid #cbd5e1;padding:6px;border-radius:6px">'
        '<b style="font-size:14px">Estados e parâmetros de controle</b>'+''.join(lines)+'</div>')

def control_loop(stop):
    global state
    last=time.monotonic();last_status=0.;last_panel=0.
    try:
        while not stop.is_set():
            now=time.monotonic();dt=min(.12,now-last);last=now
            with xbox_pygame_lock:
                snap=connection_snapshot.copy()
                fresh=snap['connected'] and now-snap['at']<.35
                controls=dict(latest_controls) if fresh and latest_controls is not None else zero_controls()
            if not fresh and state['connected']:
                state=new_control_state()
                state['hint']='Leitura do Xbox interrompida: comandos neutralizados. Reconecte e segure Start.'
            apply_control(state,controls,now,dt,int(speed_limit.value),cfg.data.get('geometry',{}))
            if now-last_panel>.15:
                update_sim_panels(state,controls,now)
                last_panel=now
            if now-last_status>.8:
                control_status.value=status_box('Simulação','OK' if fresh else 'FALHA' if reader_error else 'TESTANDO',
                    f'{snap["name"]}: dados recebidos; segure Start por 2 s.' if fresh else
                    reader_error or 'Aguardando dados do controle; veja o teste acima.').value
                last_status=now
            message.value='<div style="background:#fff8dd;padding:8px;border-radius:5px">'+html.escape(state['hint'])+'</div>'
            scene_image.value=draw_control_scene(state,controls)
            results['controle']={'fonte':'xbox' if fresh else 'indisponível',
                'conexao_simulada':state['connected'],'parada':state['emergency'],
                'modo':state['mode'],'rpm_estimadas':[round(state['left_rpm'],2),round(state['right_rpm'],2)],
                'pan_tilt_comandados':[round(state['pan'],1),round(state['tilt'],1)],
                'pressao_virtual':round(state['pressure'],3),'limite_rpm':int(speed_limit.value)}
            stop.wait(.05)
    except Exception as exc:
        control_status.value=status_box('Simulação','FALHA',f'{type(exc).__name__}: {exc}').value

def start_control(_):
    global control_stop,control_thread,state
    control_stop.set()
    if control_thread is not None and control_thread.is_alive():
        control_thread.join(timeout=2)
        if control_thread.is_alive():
            control_status.value=status_box('Simulação','FALHA','A leitura anterior ainda está encerrando.').value
            return
    state=new_control_state()
    ensure_xbox_reader()
    control_stop=threading.Event()
    control_thread=threading.Thread(target=control_loop,args=(control_stop,),daemon=True)
    control_thread.start()

def stop_control(_):
    control_stop.set()
    control_status.value=status_box('Simulação','PENDENTE','Leitura interrompida.').value
    if not button_test_active:
        xbox_read_stop.set()
        if xbox_process is not None and xbox_process.poll() is None:xbox_process.terminate()

def refresh_control_names(selected=''):
    names=setup_names('control')
    control_pick.options=[('Novo setup','')]+[(n,n) for n in names]
    control_pick.value=selected if selected in names else ''

def validate_bindings():
    with bindings_lock:
        signatures=[]
        for key,(label,_,kind,_) in INPUTS.items():
            b=bindings[key]
            allowed=('hat','button') if key in ('stop','camera_left','camera_right','overlay') else (kind,)
            if b.get('kind') not in allowed or int(b.get('index',-1))<0:
                raise ValueError(f'Configure a entrada física de {label}.')
            signatures.append((kind,int(b['index']),tuple(b.get('direction',[]))))
        if len(signatures)!=len(set(signatures)):
            raise ValueError('Há índices físicos duplicados. Configure uma entrada diferente em cada linha.')
        return {key:dict(bindings[key]) for key in INPUTS}

def save_control(_):
    global cfg
    try:
        if controller.value is None or int(controller.value)<0:
            raise ValueError('Selecione um controle Xbox real na seção 2.')
        mapped=validate_bindings()
        axes={k:int(mapped[k]['index']) for k in ('drive_x','drive_y','cannon_x','cannon_y')}
        legacy={k:int(mapped[k]['index']) for k in
                ('mode','cool_internal','cool_external','leds')}
        # O runtime legado só admite botões, não D-pad nem retenção por 1 s.
        # Evite associar por engano a emergência ao botão A (índice do hat 0).
        legacy.update(stop=65533,jet=65535,rearm=65534)
        control_settings={**cfg.data.get('control',{}),'schema_version':4,
            'speed_limit_rpm':int(speed_limit.value),'input_bindings':mapped,
            'camera_scan_speed_deg_s':camera_scan_speed.value,
            'camera_manual_speed_deg_s':camera_manual_speed.value,
            'camera_edge_offset_px':camera_edge_offset.value,
            'camera_recenter_step_deg':camera_recenter_step.value,
            'button_names':{k:input_rows[k]['name'].value.strip() or INPUTS[k][0] for k in INPUTS}}
        payload={'xbox_axes':axes,'xbox_buttons':legacy,'xbox_index':int(controller.value),
                 'control':control_settings}
        name=setup_save('control',control_name.value,payload)
        data=json.loads(config_path.read_text(encoding='utf-8'))
        data.update(payload);_json_atomic(config_path,data);cfg=AppConfig(config_path)
        refresh_control_names(name)
        map_status.value=status_box('Setup de controle','OK',
                 f'{name} salvo com nomes, entradas físicas e teto de {speed_limit.value} RPM.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

def load_control(_):
    try:
        name=control_pick.value
        if not name:raise ValueError('Selecione um setup salvo.')
        data=setup_get('control',name)
        ctl=data.get('control',{})
        old_axes=data.get('xbox_axes',{})
        with bindings_lock:
            for key in INPUTS:
                fallback=({'kind':'axis','index':int(old_axes[key])} if key in old_axes else
                          DEFAULT_BINDINGS[key])
                bindings[key]=dict(ctl.get('input_bindings',{}).get(key,fallback))
                if (int(ctl.get('schema_version',0))<3 and key=='r3') or (
                     int(ctl.get('schema_version',0))<4 and key in ('stop','camera_left','camera_right')):
                    bindings[key]=dict(DEFAULT_BINDINGS[key])
                input_rows[key]['name'].value=(INPUTS[key][0] if key in ('r3','stop') and
                    int(ctl.get('schema_version',0))<3 else
                    ctl.get('button_names',{}).get(key,INPUTS[key][0]))
                update_input_row(key)
        speed_limit.value=int(ctl.get('speed_limit_rpm',80))
        camera_scan_speed.value=int(ctl.get('camera_scan_speed_deg_s',12))
        camera_manual_speed.value=int(ctl.get('camera_manual_speed_deg_s',45))
        camera_edge_offset.value=int(ctl.get('camera_edge_offset_px',8))
        camera_recenter_step.value=int(ctl.get('camera_recenter_step_deg',20))
        control_name.value=name
        map_status.value=status_box('Setup de controle','OK',
                 f'{name} carregado; edite nomes, capture entradas e salve novamente.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

def delete_control(_):
    try:
        name=control_pick.value
        if not name:raise ValueError('Selecione um setup salvo.')
        setup_delete('control',name);refresh_control_names()
        map_status.value=status_box('Setup de controle','OK',f'{name} excluído.').value
    except Exception as exc:
        map_status.value=status_box('Setup de controle','FALHA',str(exc)).value

diagnostic_start.on_click(start_button_test);diagnostic_stop.on_click(stop_button_test)
control_start.on_click(start_control);control_pause.on_click(stop_control)
control_save.on_click(save_control);control_load.on_click(load_control);control_delete.on_click(delete_control)
scene_image.value=draw_control_scene(state,zero_controls())
update_diagnostic(zero_controls(),time.monotonic(),False)
update_sim_panels(state,zero_controls(),time.monotonic())

# Ocupa toda a largura disponível do notebook clássico e do JupyterLab.
display(widgets.HTML('<style>.container {width:98% !important; max-width:1600px !important;} '
                     '.jp-NotebookPanel-notebook {max-width: none !important;}</style>'))
display(widgets.VBox([
 widgets.HTML('<h3>1. Teste de todos os botões e eixos do Xbox</h3><p>O teste já verifica '
              'a conexão. Pressione cada botão e mova joysticks, gatilhos e D-pad. São listadas '
              'todas as entradas físicas que o pygame detectar, inclusive as não configuradas. '
              'Os sinais recebidos ficam verdes até clicar Limpar leituras. Use o índice mostrado '
              'para corrigir uma associação na seção 3.</p>'),
 widgets.HBox([diagnostic_start,diagnostic_stop,diagnostic_reset]),
 connection_status,diagnostic_status,diagnostic,
 widgets.HTML('<h3>2. Setups de controle</h3><p>Selecione para editar ou excluir. '
              'Salvar com o mesmo nome atualiza o setup; a última célula escolhe qual setup será oficial.</p>'),
 widgets.HBox([control_pick,control_name],layout=widgets.Layout(width='100%')),
 widgets.HBox([control_load,control_save,control_delete]),
 speed_limit,
 widgets.HTML('<h3>3. Botões de função e comandos pré-definidos</h3><p>Os comandos '
              'têm função fixa. Dê um nome e capture o índice físico em cada cartão. '
              'Azul: eixos; vermelho: Start e parada. D-pad para baixo por 1 s aciona a parada; '
              'R3 é o clique do stick direito. Entradas repetidas são recusadas.</p>'),
 widgets.GridBox([input_rows[k]['widget'] for k in INPUTS],
     layout=widgets.Layout(width='100%',grid_template_columns='repeat(3, minmax(0, 1fr))',
                           grid_gap='6px')),map_status,
 widgets.HTML('<h3>4. Simulação visual do robô e do canhão</h3><p>É necessário um Xbox conectado. '
              'Todos os atuadores começam desligados. Segure Start por 2 s; LT acelera, '
              'o stick esquerdo guia, LB freia; View libera água e RT regula pressão. '
              'O stick direito move a mira; ←/→ giram a plataforma no modo manual. '
              'Segurar D-pad para baixo por 1 s interrompe e exige rearme. '
              'A imagem é uma simulação; não envia comandos ao ESP32.</p>'),
 widgets.HBox([control_start,control_pause]),control_status,message,
 widgets.HBox([
   widgets.VBox([scene_image,reference_panel],layout=widgets.Layout(width='70%')),
   widgets.VBox([state_panel],layout=widgets.Layout(width='29%'))],
   layout=widgets.Layout(width='100%',align_items='flex-start'))
],layout=widgets.Layout(width='100%',max_width='1550px')))


## 6. ESP32: portas, rede local e setup de hardware

**Objetivo:** definir o conjunto de motores, os testes autorizados e o transporte USB ou rede local criada pelo ESP32. O firmware fornece endereço fixo `192.168.4.1:8765`; o notebook gera um SSID `BLAZE-XXXXXX` e uma senha únicos ao salvar esta seção. O guia mostra esses dados e eles ficam no JSON local `pc_operador/parameters_setups/user/network_credentials.json` e no cabeçalho que será compilado. Nenhum SSID ou senha de hotspot externo é necessário.

**Ordem:** conecte a placa por cabo de dados para identificar a COM; escolha o transporte; selecione motor e driver; deixe saídas físicas e testes de atuadores desligados no primeiro ensaio; salve o setup; crie a combinação na seção 7; compile e grave pela USB na seção 8; teste a conexão na seção 9. Após mudar as credenciais ou o setup de hardware, grave novamente. Os GPIOs constam em `esp32/include/pins.h`.


In [ ]:
# Ferramentas de ensaio deste notebook; código de produção PC em runtime.py.
import csv, getpass, os, re, shutil, socket, subprocess, threading, time, types
ESP_HERE = ROOT / 'esp32'
HOST_TEST_FILES = {'central_states.cpp': '// Compila apenas a máquina de estados em um PC; não simula Wi-Fi ou GPIO.\n#include <cassert>\n#include "central.h"\n\nint main() {\n  Central central;\n  Command command;\n  command.valid = true;\n  command.vision_valid = true;\n  command.stop = true;\n  SafetyReport safety;\n\n  auto state = central.update(command, true, safety);\n  assert(!state.safe && !state.allow_motion && !state.armed);\n  command.stop = false;\n  command.rearm = true;\n  state = central.update(command, true, safety);\n  assert(state.armed && state.allow_motion);\n\n  command.rearm = false;\n  command.person_detected = true;\n  state = central.update(command, true, safety);\n  assert(!state.armed && !state.allow_jet && !state.allow_motion);\n  command.person_detected = false;\n  state = central.update(command, true, safety);\n  assert(!state.armed);  // desaparecer da imagem não rearma.\n\n  command.rearm = true;\n  state = central.update(command, true, safety);\n  assert(state.armed);\n  state = central.update(command, false, safety);\n  assert(!state.connected && !state.armed && !state.allow_motion);\n  state = central.update(command, true, safety);\n  assert(!state.armed); // REARMAR ainda pressionado não vale após reconexão.\n  command.rearm = false;\n  state = central.update(command, true, safety);\n  assert(!state.armed);\n  command.rearm = true;\n  state = central.update(command, true, safety);\n  assert(state.armed);\n\n  command.auto_mode = true;\n  command.rearm = false;\n  command.target_valid = false;\n  state = central.update(command, true, safety);\n  assert(state.auto_mode && state.allow_motion && !state.allow_jet);\n  command.target_valid = true;\n  command.water_on = true;\n  state = central.update(command, true, safety);\n  assert(state.allow_jet);\n  command.calibrating = true;\n  state = central.update(command, true, safety);\n  assert(!state.armed && !state.allow_motion && !state.allow_jet);\n}\n', 'motion_math.cpp': '#include <cassert>\n#include "motion.h"\n\nint main() {\n  const auto forward = mixDrive(0, -1);\n  assert(forward.left == 1 && forward.right == 1);\n  const auto turn = mixDrive(1, 0);\n  assert(turn.left == 1 && turn.right == -1);\n  const auto bounded = mixDrive(1, -1);\n  assert(bounded.left == 1 && bounded.right == 0);\n}\n', 'stubs/Arduino.h': '#pragma once\n#include <cmath>\n#include <cstdint>\n'}
from runtime import AppConfig, ESPConnection, control_message

def check_structure():
    for path in (ESP_HERE / 'platformio.ini', ESP_HERE / 'src/main.cpp', ESP_HERE / 'include/pins.h'):
        if not path.is_file():
            raise FileNotFoundError(str(path))
    print('Estrutura do firmware ESP32: OK')


def answer(question, default='n'):
    if not sys.stdin.isatty():
        return default
    try:
        return input(question).strip().lower() or default
    except EOFError:
        return default


def stage_configuration(allow_outputs=False):
    pins = (ESP_HERE / 'include/pins.h').read_text(encoding='utf-8') + (ESP_HERE / 'include/setup_generated.h').read_text(encoding='utf-8')
    mapping = {name: int(value) for name, value in
               re.findall(r'constexpr int\s+(\w+)\s*=\s*(-?\d+)\s*;', pins)}
    required = ('left_rpwm', 'left_lpwm', 'left_ren', 'left_len',
                'right_rpwm', 'right_lpwm', 'right_ren', 'right_len',
                'servo_pan', 'servo_tilt', 'servo_camera_pan', 'dht_external', 'dht_internal',
                'relay_cool_external', 'relay_cool_internal', 'relay_jet')
    optional = {'left_ren', 'left_len', 'right_ren', 'right_len'}  # EN pode estar ligado a VCC
    if any(mapping.get(name, -1) < 0 for name in required if name not in optional):
        raise ValueError('Mapa GPIO incompleto; confira docs/conexoes_esp32.md')
    unique = {name: pin for name, pin in mapping.items()
              if name not in ('left_len', 'right_len') and pin >= 0}
    if len(set(unique.values())) != len(unique) or (mapping['left_ren'] != mapping['left_len'] or
                                                    mapping['right_ren'] != mapping['right_len']):
        raise ValueError('GPIO duplicado além do par EN de cada BTS7960.')
    print('LEDs:', 'GPIO '+str(mapping['led_status']) if mapping.get('led_status',-1)>=0 else 'sem GPIO definido; botão não acionará LEDs')
    print('Mapa de GPIO:')
    for name in required:
        print(' ', name, '→ GPIO', mapping[name])
    physical = re.search(r'^#define\s+BLAZE_ENABLE_PHYSICAL_OUTPUTS\s+([01])', pins, re.M)
    jet = re.search(r'^#define\s+BLAZE_ENABLE_JET_OUTPUT\s+([01])', pins, re.M)
    if not physical or not jet:
        raise ValueError('Flags de acionamento não encontradas em pins.h')
    print('Saídas físicas:', 'HABILITADAS' if physical.group(1) == '1' else 'desabilitadas')
    print('Válvula física do jato:', 'HABILITADA' if jet.group(1) == '1' else 'desabilitada')
    import profile_manager
    credentials = profile_manager.local_link_credentials(create=True)
    print('Rede local criada pelo ESP32:', credentials[0], '· IP fixo 192.168.4.1')
    if physical.group(1) == '1' and not allow_outputs and answer(
            'Saídas físicas estão habilitadas. Continuar com compilação/teste? [s/N] ') not in ('s', 'sim'):
        raise RuntimeError('Desabilite saídas físicas em pins.h para o primeiro ensaio.')
    if jet.group(1) == '1':
        raise RuntimeError('O ensaio de chama impressa exige BLAZE_ENABLE_JET_OUTPUT 0.')
    return {'physical_outputs': physical.group(1) == '1', 'jet_output': False}


def stage_host_test():
    compiler = shutil.which('g++')
    if not compiler:
        print('g++ host ausente: teste da máquina de estados será coberto no firmware.')
        return 'indisponível'
    output = ROOT / 'output' / 'testes_esp32'
    output.mkdir(parents=True, exist_ok=True)
    main = (ESP_HERE / 'src/main.cpp').read_text(encoding='utf-8')
    start = '// ===== SUBSISTEMA CENTRAL ====='
    end = '// ===== SUBSISTEMA MOTION ====='
    if main.count(start) != 1 or main.count(end) != 1:
        raise RuntimeError('Seção CENTRAL não foi encontrada no runtime ESP32.')
    central = output / 'central_host.cpp'
    central.write_text(main.split(start, 1)[1].split(end, 1)[0], encoding='utf-8')
    stub = output / 'stubs'
    stub.mkdir(exist_ok=True)
    for name, source in HOST_TEST_FILES.items():
        target = output / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(source, encoding='utf-8')
    for name, sources in (('central', [central, output / 'central_states.cpp']),
                          ('motion', [output / 'motion_math.cpp'])):
        executable = output / (name + ('_test.exe' if os.name == 'nt' else '_test'))
        command = [compiler, '-std=c++17', '-I' + str(stub),
                   '-I' + str(ESP_HERE / 'include'), '-I' + str(ESP_HERE / 'src')]
        command += [str(path) for path in sources] + ['-o', str(executable)]
        subprocess.run(command, check=True, cwd=str(ESP_HERE))
        subprocess.run([str(executable)], check=True, cwd=str(ESP_HERE))
        print('Teste isolado {}: aprovado'.format(name))
    return 'aprovado'


def pio_executable():
    path = Path(sys.executable).parent / ('pio.exe' if os.name == 'nt' else 'pio')
    if not path.is_file():
        raise RuntimeError('Instalação do PlatformIO não disponibilizou pio.')
    return str(path)


def stage_build():
    pio = pio_executable()
    subprocess.run([pio, 'run', '-e', 'esp32dev'], cwd=str(ESP_HERE), check=True)
    print('Firmware compilado para esp32dev. Nenhum upload foi realizado ainda.')
    return 'aprovado'


def stage_ports():
    from serial.tools import list_ports

    ports = list(list_ports.comports())
    if ports:
        print('Portas seriais encontradas:')
        for item in ports:
            print(' ', item.device, '-', item.description)
    else:
        print('Nenhuma porta serial visível. Conecte a placa/cabo de dados quando for gravar.')
    return [item.device for item in ports]


def stage_upload(port, automatic):
    if not port:
        print('Sem porta escolhida: gravação adiada.')
        return False
    if port not in stage_ports():
        print('Porta informada não está disponível:', port)
        return False
    if automatic or answer('Gravar firmware no ESP32 pela porta {}? [s/N] '.format(port)) in ('s', 'sim'):
        subprocess.run([pio_executable(), 'run', '-e', 'esp32dev', '-t', 'upload',
                        '--upload-port', port], cwd=str(ESP_HERE), check=True)
        print('Upload concluído.')
        return True
    print('Upload adiado pelo operador.')
    return False


def stage_serial_ip(port, seconds=15):
    if not port:
        return ''
    import serial

    deadline = time.monotonic() + seconds
    try:
        with serial.Serial(port, 115200, timeout=0.4) as monitor:
            print('Aguardando IP no serial por até {} s...'.format(seconds))
            while time.monotonic() < deadline:
                line = monitor.readline().decode('utf-8', 'replace').strip()
                if line:
                    print('SERIAL:', line)
                match = re.search(r'BLAZE ESP32 IP:\s*(\d+\.\d+\.\d+\.\d+)', line)
                if match:
                    return match.group(1)
    except (OSError, serial.SerialException) as exc:
        print('Monitor serial indisponível:', exc)
    print('IP não apareceu; informe --ip ou confira Wi-Fi em secrets.h.')
    return ''


def stage_telemetry(host='', usb_port='', transport='auto'):
    import threading
    sys.path.insert(0, str(ROOT / 'pc_operador'))
    from runtime import ESPConnection
    from runtime import control_message

    found = []
    ready = threading.Event()
    link = ESPConnection(host or 'auto', 8765,
                         lambda: control_message({'requested_mode': 'manual', 'stop': True}, {}),
                         lambda data: (found.append(data), ready.set()),
                         transport=transport, serial_port=usb_port or 'auto',
                         preserve_wifi=(isinstance(cfg.data.get('vision',{}).get('camera_source'),str) and cfg.data['vision']['camera_source'].lower().startswith(('rtsp://','rtsps://','http://','https://'))))
    link.start()
    try:
        ready.wait(9)
        if not found:
            print('Sem telemetria; USB requer porta reconhecida e firmware atualizado. Wi-Fi requer mesma rede.')
            return None
        data = dict(found[-1], transport=link.active_transport)
        print('Enlace:', data['transport'], '| modo:', data.get('mode'), '| motivo:', data.get('reason'))
        print('Sensores I/E válidos:', data.get('sensor_internal_valid'), data.get('sensor_external_valid'))
        return data
    finally:
        link.stop()


def stage_usb_channels(port, automatic=False, enabled_channels=None, settings=None):
    if not port or (automatic and enabled_channels is None):
        print('Canais USB pendentes: escolha uma porta e execute interativamente.')
        return {'estado': 'pendente'}
    import serial

    channels = [('MOTOR_L', .15, 600, 'Esteira esquerda, sem carga'),
                ('MOTOR_R', .15, 600, 'Esteira direita, sem carga'),
                ('SERVO_PAN', 110, 700, 'Servo pan'),
                ('SERVO_TILT', 110, 700, 'Servo tilt'),
                ('SERVO_CAMERA', 90, 700, 'MG90S da plataforma da câmera'),
                ('COOL_INT', 1, 300, 'Relé interno'),
                ('COOL_EXT', 1, 300, 'Relé externo')]
    settings = settings or {}
    result = {}
    try:
        with serial.Serial(port, 115200, timeout=.2) as monitor:
            time.sleep(2.5)  # algumas placas reiniciam ao abrir a porta
            monitor.reset_input_buffer()
            monitor.write(b'STATUS\n')
            for _ in range(12):
                line = monitor.readline().decode('utf-8', 'replace').strip()
                if line.startswith('SENSOR,'):
                    print('Leitura dos DHT (válido, temperatura, umidade):', line)
                    result['sensores'] = line
                    break
            for name, value, duration, title in channels:
                value, duration = settings.get(name, (value, duration))
                if (name not in enabled_channels if enabled_channels is not None else
                    answer('Componente {} conectado e área livre? Testar {}? [s/N] '.format(
                        name, title)) not in ('s', 'sim')):
                    result[name] = 'pendente'
                    continue
                monitor.reset_input_buffer()
                monitor.write('TEST {} {} {}\n'.format(name, value, duration).encode('ascii'))
                response = []
                deadline = time.monotonic() + 3.0
                while time.monotonic() < deadline:
                    line = monitor.readline().decode('utf-8', 'replace').strip()
                    if line.startswith('DIAG '):
                        response.append(line)
                        print(name + ':', line)
                        if line == 'DIAG DONE' or line.startswith('DIAG REFUSED'):
                            break
                done = any(line == 'DIAG DONE' for line in response)
                observed = (answer('Observou a resposta física esperada? [s/N] ')
                            if done and enabled_channels is None else '')
                result[name] = ('observado_manualmente' if observed in ('s', 'sim') else
                                'sem_comprovacao_fisica') if done else 'sem_resposta_firmware'
    except (OSError, serial.SerialException) as exc:
        print('USB indisponível:', exc)
        result['erro'] = str(exc)
    return result


def stage_integrated(cfg, seconds=45):
    from runtime import CameraSource
    from runtime import OperatorInputs
    from runtime import ESPConnection
    from runtime import control_message
    from runtime import VisionWorker

    inputs = OperatorInputs(cfg.data['xbox_axes'], cfg.data['xbox_buttons'],
                            preferred_index=int(cfg.data.get('xbox_index', 0)),
                            speed_limit_rpm=float(cfg.data.get('control', {}).get('speed_limit_rpm', 80)),
                            calibrated_max_rpm=float(cfg.data['geometry'].get('output_max_rpm', 0)),
                            bindings=cfg.data.get('control',{}).get('input_bindings',{}))
    cam = cfg.data['camera']
    camera = CameraSource(cfg.data['vision'].get('camera_source',cfg.camera_source()), cam['reconnect_seconds'], cam.get('mdns_name', ''),
                          cam.get('stream_path', '/live'), cam.get('fallback_source', ''))
    vision = VisionWorker(camera, cfg.vision, cfg.data['vision'], inputs)
    received = []
    sent = []

    def command():
        value = control_message(inputs.snapshot(), vision.snapshot().state())
        sent.append({'t': round(time.monotonic() - start, 3), 'data': value})
        return value

    def telemetry(message):
        received.append({'t': round(time.monotonic() - start, 3), 'data': message})
        inputs.acknowledge_mode(str(message.get('mode', '')), bool(message.get('jet_virtual', False)))

    esp = cfg.data['esp']
    link = ESPConnection(str(esp['host']), int(esp['port']), command, telemetry,
                         float(esp['reconnect_seconds']), str(esp.get('transport', 'auto')),
                         str(esp.get('serial_port', 'auto')), int(esp.get('baudrate', 115200)),
                         str(esp.get('bluetooth_port','auto')),
                         str(cfg.data.get('_official',{}).get('firmware_signature','')),
                         preserve_wifi=(isinstance(cfg.data.get('vision',{}).get('camera_source'),str) and cfg.data['vision']['camera_source'].lower().startswith(('rtsp://','rtsps://','http://','https://'))))
    print('Ensaio real: controle Xbox e vídeo da câmera devem estar ligados.')
    print('Siga as instruções por fase. O jato permanece virtual. Duração:', seconds, 's.')
    start = time.monotonic()
    camera.start()
    vision.start()
    link.start()
    last_print = -5.
    phase_index = -1
    phases = [(0, 'Aguarde câmera e telemetria, deixe os joysticks neutros.'),
              (5, 'Segure Start por 2 s para conectar/rearmar; mantenha os joysticks neutros.'),
              (10, 'Mova o joystick esquerdo: frente, trás e giros; depois centralize.'),
              (20, 'Mova o joystick direito: pan e tilt; depois use o botão de alternar modo.'),
              (30, 'Teste View para água, RT para pressão, arrefecimento e LEDs; confira a telemetria.'),
              (36, 'Deixe o robô parado e apresente uma pessoa à câmera; observe o bloqueio.'),
              (41, 'Segure D-pad para baixo por 1 s para emergência e mantenha comandos neutros.')]
    reconnect_ok = False
    disarmed_after_reconnect = False
    try:
        while time.monotonic() - start < seconds:
            inputs.poll()
            inputs.set_link(link.connected)
            elapsed = time.monotonic() - start
            while phase_index + 1 < len(phases) and elapsed >= phases[phase_index + 1][0]:
                phase_index += 1
                print('FASE {}: {}'.format(phase_index + 1, phases[phase_index][1]))
            if elapsed - last_print >= 5:
                recent = received[-1]['data'] if received else {}
                print('t={:.0f}s ESP={} câmera={} visão={} bloqueio={} E/D={}/{}'.format(
                    elapsed, link.connected, bool(vision.snapshot().frame_stamp),
                    vision.snapshot().valid, recent.get('reason', 'sem telemetria'),
                    recent.get('drive_left_commanded'), recent.get('drive_right_commanded')))
                last_print = elapsed
            time.sleep(.05)
        if received:
            print('Testando desconexão/reconexão TCP controlada; comandos locais serão zerados.')
            previous = len(received)
            link.stop()
            inputs.set_link(False)
            time.sleep(1.0)  # excede o timeout de 750 ms do firmware
            link.start()
            deadline = time.monotonic() + 5.0
            while time.monotonic() < deadline:
                inputs.poll()
                inputs.set_link(link.connected)
                if len(received) > previous:
                    reconnect_ok = True
                    disarmed_after_reconnect = not received[previous]['data'].get('armed', True)
                    break
                time.sleep(.05)
    finally:
        inputs.action('stop')
        link.stop()
        vision.stop()
        camera.stop()
    states = [row['data'] for row in received]
    commands = [row['data'] for row in sent]
    result = {
        'pacotes_enviados': len(sent), 'pacotes_recebidos': len(received),
        'enlace_real': bool(received),
        'camera_recente': any(cmd.get('vision_valid') for cmd in commands),
        'rearmado': any(msg.get('armed') for msg in states),
        'tracao_comandada': any(abs(float(msg.get('drive_left_commanded') or 0)) > .02 or
                                 abs(float(msg.get('drive_right_commanded') or 0)) > .02 for msg in states),
        'servo_comandado': any(abs(float(msg.get('servo_pan_commanded') or 90) - 90) > 2 or
                                abs(float(msg.get('servo_tilt_commanded') or 90) - 90) > 2 for msg in states),
        'modo_automatico': any(msg.get('mode') == 'auto' for msg in states),
        'sensor_interno_valido': any(msg.get('sensor_internal_valid') for msg in states),
        'sensor_externo_valido': any(msg.get('sensor_external_valid') for msg in states),
        'arrefecimento_comandado': any(msg.get('cool_internal_commanded') or
                                       msg.get('cool_external_commanded') for msg in states),
        'jato_virtual': any(msg.get('jet_virtual') for msg in states),
        'parada_operador': any(msg.get('reason') == 'operator_stop' for msg in states),
        'pessoa_bloqueada': any(msg.get('reason') == 'person_detected' for msg in states),
        'reconectou': reconnect_ok,
        'desarmado_apos_reconexao': disarmed_after_reconnect,
        'observacao': 'Atuação e velocidades físicas exigem confirmação visual/encoder; telemetria é comando.'}
    folder = ROOT / 'output' / 'testes_pc'
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'integrado_pacotes.json').write_text(
        json.dumps({'enviados': sent, 'recebidos': received}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Resumo do teste integrado:', json.dumps(result, ensure_ascii=False))
    return result



esp_test = types.SimpleNamespace(
    check_structure=check_structure, stage_configuration=stage_configuration,
    stage_host_test=stage_host_test, stage_build=stage_build,
    stage_ports=stage_ports, stage_upload=stage_upload,
    stage_serial_ip=stage_serial_ip, stage_telemetry=stage_telemetry,
    stage_usb_channels=stage_usb_channels, stage_integrated=stage_integrated)
esp_test.check_structure()


In [ ]:
# Seção 6 — conexão, firmware e diagnóstico do ESP32.
# Execute após a célula que define ROOT e as funções setup_names/get/save/delete.

import html
import json
import os
import re

import ipywidgets as widgets
from IPython.display import display
from serial.tools import list_ports


def esp_layout():
    # Cada widget recebe seu próprio Layout para que ocultar Wi-Fi
    # não oculte os demais campos.
    return widgets.Layout(width="100%")


ESP_STYLE = {"description_width": "160px"}

esp_ports = widgets.Dropdown(
    description="Porta USB do ESP32:",
    style=ESP_STYLE,
    layout=esp_layout(),
)
esp_manual = widgets.Text(
    description="Porta COM manual:",
    placeholder="Ex.: COM4 ou /dev/ttyUSB0",
    style=ESP_STYLE,
    layout=esp_layout(),
)
esp_transport = widgets.Dropdown(description="Comunicação:", options=[
    ("Rede local criada pelo ESP32 (192.168.4.1)", "wifi_local"),
    ("Somente USB", "usb"),
    ("Automática: USB, depois rede do ESP32", "auto")],
    value="wifi_local", style=ESP_STYLE, layout=esp_layout())
esp_host = widgets.Text(value="192.168.4.1", disabled=True)
esp_bt_port = widgets.Text(value="auto")
esp_ssid = widgets.Text(value="")
esp_password = widgets.Password(value="")

esp_outputs = widgets.Checkbox(
    value=False,
    description="Saídas físicas",
    indent=False,
    layout=widgets.Layout(width="155px"),
)
esp_jet_output = widgets.Checkbox(
    value=False,
    description="Válvula do jato real",
    indent=False,
    layout=widgets.Layout(width="210px"),
)
esp_flash = widgets.Checkbox(
    value=False,
    description="Gravar firmware",
    indent=False,
    layout=widgets.Layout(width="155px"),
)
esp_motors = widgets.Checkbox(
    value=False,
    description="Motores",
    indent=False,
    layout=widgets.Layout(width="155px"),
)
esp_servos = widgets.Checkbox(
    value=False,
    description="Servos",
    indent=False,
    layout=widgets.Layout(width="155px"),
)
esp_relays = widgets.Checkbox(
    value=False,
    description="Relés",
    indent=False,
    layout=widgets.Layout(width="155px"),
)

esp_name = widgets.Text(
    value="ESP32",
    description="Modelo ESP32:",
    style=ESP_STYLE,
    layout=esp_layout(),
)
esp_drive_kind = widgets.Dropdown(
    options=[('Motor DC 3–6 V · 2 × BTS7960 (fonte 5,5–6 V)','dc_3_6v'),
             ('Mabuchi 12 V · 2 × BTS7960','bts7960_12v'),
             ('Kalatec KTC-86BLS125-J · driver BLDC (planejamento)','bldc_ktc86')],
    value='bts7960_12v',description='Esteiras:',style=ESP_STYLE,layout=esp_layout())
esp_drive_model = widgets.Text(value='Mabuchi 12V',description='Modelo do motor:',style=ESP_STYLE,layout=esp_layout())
esp_drive_driver = widgets.Text(value='BTS7960',description='Modelo do driver:',style=ESP_STYLE,layout=esp_layout())
esp_cannon_kind = widgets.Dropdown(
    options=[('2 × MG996R · PWM 50 Hz','mg996r'),
             ('2 × HT23-EC1000 · stepservo (planejamento)','ht23_ec1000')],
    value='mg996r',description='Canhão:',style=ESP_STYLE,layout=esp_layout())
esp_cannon_driver = widgets.Text(value='MG996R-PWM',description='Driver do canhão:',style=ESP_STYLE,layout=esp_layout())
esp_gear_ratio = widgets.FloatText(value=1.,description='Redução esteiras:',style=ESP_STYLE,layout=esp_layout())
ESP_DRIVER_MODELS={'dc_3_6v':'BTS7960','bts7960_12v':'BTS7960',
                   'bldc_ktc86':'KTC-J-DR-BLDC750'}
def esp_update_motor_fields(change=None):
    if change and change['owner'] is esp_drive_kind:
        esp_drive_driver.value=ESP_DRIVER_MODELS[esp_drive_kind.value]
        esp_drive_model.value={'dc_3_6v':'DC 3-6V', 'bts7960_12v':'Mabuchi 12V',
                               'bldc_ktc86':'KTC-86BLS125-J'}[esp_drive_kind.value]
    if change and change['owner'] is esp_cannon_kind:
        esp_cannon_driver.value='MG996R-PWM' if esp_cannon_kind.value=='mg996r' else 'KTC-DR23EC-K'
    esp_update_guide()

esp_saved = widgets.Dropdown(
    description="Setup salvo:",
    style=ESP_STYLE,
    layout=esp_layout(),
)

esp_find = widgets.Button(
    description="Atualizar portas USB",
    button_style="info",
    layout=widgets.Layout(width="185px"),
)
esp_save = widgets.Button(
    description="Salvar ou atualizar",
    button_style="success",
    layout=widgets.Layout(width="185px"),
)
esp_load = widgets.Button(
    description="Carregar para editar",
    button_style="info",
    layout=widgets.Layout(width="185px"),
)
esp_delete = widgets.Button(
    description="Excluir setup",
    button_style="danger",
    layout=widgets.Layout(width="150px"),
)

esp_form_status = widgets.HTML(layout=esp_layout())
esp_guide = widgets.HTML(layout=esp_layout())
esp_wifi_group = widgets.VBox(layout=esp_layout())
esp_local_group = widgets.VBox(layout=esp_layout())
esp_bluetooth_group = widgets.VBox(layout=esp_layout())
esp_manual_group = widgets.VBox(layout=esp_layout())

ESP_FORM_SAVED = False


def esp_hint(message):
    return widgets.HTML(
        '<div style="font-size:12px;line-height:1.45;color:#475569;'
        'white-space:normal;overflow-wrap:anywhere;padding:2px 0 7px">'
        + message
        + "</div>",
        layout=esp_layout(),
    )


def esp_checkbox_row(control, explanation):
    return widgets.HBox(
        [
            control,
            widgets.HTML(
                '<div style="font-size:12px;line-height:1.4;'
                'white-space:normal;overflow-wrap:anywhere;padding-top:5px">'
                + explanation
                + "</div>",
                layout=widgets.Layout(width="auto", flex="1 1 320px"),
            ),
        ],
        layout=widgets.Layout(width="100%", align_items="flex-start"),
    )


def esp_notice(state, title, message):
    colors = {
        "OK": ("#e9f7ed", "#176c38"),
        "PENDENTE": ("#fff8e4", "#865e00"),
        "FALHA": ("#fff0ee", "#ad2921"),
    }
    background, foreground = colors.get(state, ("#eef2f7", "#334155"))
    esp_form_status.value = (
        f'<div style="background:{background};border:1px solid {foreground};'
        'border-radius:8px;padding:10px 12px;line-height:1.45;'
        'white-space:normal;overflow-wrap:anywhere">'
        f'<b style="color:{foreground}">{html.escape(state)} — '
        f"{html.escape(title)}</b><br>{html.escape(message)}</div>"
    )


def esp_selected_port():
    if esp_ports.value == "manual":
        return esp_manual.value.strip()
    return esp_ports.value or "auto"


def esp_refresh_saved(selected=""):
    names = setup_names("hardware")
    esp_saved.options = [("Novo setup", "")] + [(name, name) for name in names]
    esp_saved.value = selected if selected in names else ""


def esp_refresh_ports(_=None):
    previous = esp_ports.value
    found = list(list_ports.comports())

    options = [
        ("Selecione a porta após conectar o ESP32", "auto"),
        ("Informar porta manualmente", "manual"),
    ]

    for item in found:
        description = item.description or "dispositivo serial"

        if re.search(r"CP210|Silicon Labs|CH340|CH341", description, re.I):
            hint = " — provável ESP32"
        elif re.search(r"Nuvoton", description, re.I):
            hint = " — confirme desconectando/reconectando"
        else:
            hint = ""

        options.append(
            (f"{item.device} — {description}{hint}", item.device)
        )

    esp_ports.options = options
    valid_values = [value for _, value in options]
    esp_ports.value = previous if previous in valid_values else "auto"

    if found:
        labels = ", ".join(
            f"{item.device} ({item.description})" for item in found
        )
        esp_notice(
            "OK",
            "Portas encontradas",
            labels
            + ". Confirme a porta retirando e recolocando somente o cabo do ESP32.",
        )
    else:
        esp_notice(
            "PENDENTE",
            "Nenhuma porta USB encontrada",
            "Use um cabo USB de dados, confira o driver da ponte USB "
            "e clique em Atualizar portas USB.",
        )

    esp_update_guide()


def esp_update_fields(_=None):
    esp_manual_group.layout.display = "" if esp_ports.value == "manual" else "none"
    esp_wifi_group.layout.display = "none"
    esp_bluetooth_group.layout.display = "none"
    esp_local_group.layout.display = "" if esp_transport.value in ("auto", "wifi_local") else "none"
    esp_update_guide()


def esp_update_guide(_=None):
    import profile_manager
    port = esp_selected_port()
    credentials = profile_manager.local_link_credentials(create=False)
    if credentials:
        ssid, password, _, _ = credentials
        ap = (f"SSID: <b>{html.escape(ssid)}</b>; senha: <code>{html.escape(password)}</code>; "
              "endereço do ESP32: <code>192.168.4.1:8765</code>. "
              "A senha também está em parameters_setups/user/network_credentials.json.")
    else:
        ap = "A rede e a senha são geradas ao salvar o setup de hardware."
    steps = [
        ("1. Porta USB", "Conecte um cabo de dados e selecione a COM do ESP32 para a primeira gravação. "
         + (f"Porta selecionada: <b>{html.escape(port)}</b>." if port not in ('auto','manual','')
            else "Use Atualizar portas USB e identifique a COM que aparece com a placa.")),
        ("2. Comunicação", "Escolha USB, rede local criada pelo ESP32 ou automática (USB → rede local). "
         "Nenhum IP dinâmico ou hotspot do Windows precisa ser configurado."),
        ("3. Credenciais da rede local", ap + " Depois de gravar o firmware pela USB, "
         "o runtime conecta o Windows à rede BLAZE automaticamente quando escolher Wi-Fi."),
        ("4. Motores e proteção", "Selecione os motores/driver e confira a pinagem. "
         "Deixe as saídas físicas e os diagnósticos desmarcados no primeiro teste."),
        ("5. Salvar e gravar", "Salve este setup, selecione a combinação completa na seção 7, "
         "compile e grave por USB na seção 8. A seção 9 mede a comunicação configurada.")]
    esp_guide.value = ('<div style="border:1px solid #9cb7cf;border-radius:10px;'
        'background:#f7fbff;padding:14px;line-height:1.5;overflow-wrap:anywhere">'
        '<h3>Primeiro uso</h3>' + ''.join(f'<p><b>{title}</b><br>{body}</p>'
            for title,body in steps) + '</div>')

def esp_form_payload():
    name = esp_name.value.strip()
    if esp_drive_driver.value.strip().upper()!=ESP_DRIVER_MODELS[esp_drive_kind.value].upper():
        raise ValueError('Confirme o driver compatível com o conjunto de esteiras selecionado.')
    expected_cannon='MG996R-PWM' if esp_cannon_kind.value=='mg996r' else 'KTC-DR23EC-K'
    if esp_cannon_driver.value.strip().upper()!=expected_cannon:
        raise ValueError(f'O canhão selecionado exige o driver {expected_cannon}.')
    if not esp_drive_model.value.strip() or not 1<=esp_gear_ratio.value<=10000:
        raise ValueError('Preencha o modelo do motor e uma redução entre 1 e 10000.')

    if not name or '/' in name or '\\' in name or any(ord(ch)<32 for ch in name):
        raise ValueError(
            "Nome: use 1 a 60 caracteres (letras, números, "
            "espaço, ponto, _ ou -)."
        )

    port = esp_selected_port()

    if esp_ports.value == "manual" and not esp_manual.value.strip():
        raise ValueError("Preencha a porta manual selecionada.")

    if esp_flash.value and port == "auto":
        raise ValueError(
            "Para gravar, escolha uma porta USB real; "
            "a COM automática não basta."
        )

    # O AP local do ESP32 tem credenciais próprias, geradas e salvas em JSON.
    import profile_manager
    ap_ssid, ap_password, bt_name, bt_pin = profile_manager.local_link_credentials(create=True)
    if esp_transport.value in ('auto','wifi_local') and not (ap_ssid and len(ap_password)>=8):
        raise ValueError('Credenciais da rede local do ESP32 ausentes. Salve novamente.')

    if esp_jet_output.value and not esp_outputs.value:
        raise ValueError("Habilite as saídas físicas antes de marcar a válvula do jato real.")
    host = "192.168.4.1"

    return name, {
        "esp": {
            "transport": esp_transport.value,
            "serial_port": port,
            "bluetooth_port": esp_bt_port.value.strip() or "auto",
            "host": host,
        },
        "hardware": {
            "upload_port": port,
            "host": host,
            "transport": esp_transport.value,
            "physical_outputs": bool(esp_outputs.value),
            "drive_kind":esp_drive_kind.value,
            "drive_motor_model":esp_drive_model.value.strip(),
            "drive_driver_model":esp_drive_driver.value.strip(),
            "cannon_kind":esp_cannon_kind.value,
            "cannon_driver_model":esp_cannon_driver.value.strip(),
            "gear_ratio":float(esp_gear_ratio.value),
            "jet_output_enabled": bool(esp_jet_output.value),
            "ensaios": {
                "upload": bool(esp_flash.value),
                "motors": bool(esp_motors.value),
                "servos": bool(esp_servos.value),
                "relays": bool(esp_relays.value),
            },
        },
    }


def esp_apply(_):
    global ESP_FORM_SAVED

    try:
        name, payload = esp_form_payload()
        setup_save("hardware", name, payload)
        esp_refresh_saved(name)
        ESP_FORM_SAVED = True

        esp_notice(
            "OK",
            "Setup salvo",
            f"{name} foi salvo. Selecione-o na combinação oficial "
            "da seção 7; depois siga as seções 8, 10 e 11.",
        )
    except Exception as exc:
        esp_notice("FALHA", "Não foi possível salvar", str(exc))

    esp_update_guide()


def esp_load_saved(_):
    try:
        if not esp_saved.value:
            raise ValueError("Selecione um setup salvo na lista.")

        name = esp_saved.value
        payload = setup_get("hardware", name)
        data = payload.get("hardware", {})

        esp_name.value = name
        port = str(data.get("upload_port", "auto"))
        listed_ports = [value for _, value in esp_ports.options]

        esp_ports.value = port if port in listed_ports else "manual"
        esp_manual.value = port if esp_ports.value == "manual" else ""
        esp_transport.value = data.get('transport') if data.get('transport') in ('auto','usb','wifi_local') else 'wifi_local'
        esp_host.value = data.get("host", "auto")
        esp_bt_port.value = data.get("bluetooth_port", "auto")

        flags = data.get("ensaios", {})
        for control, key in (
            (esp_flash, "upload"),
            (esp_motors, "motors"),
            (esp_servos, "servos"),
            (esp_relays, "relays"),
        ):
            control.value = bool(flags.get(key, False))

        esp_drive_kind.value=data.get('drive_kind','bts7960_12v')
        esp_drive_model.value=data.get('drive_motor_model','Mabuchi 12V')
        esp_drive_driver.value=data.get('drive_driver_model',ESP_DRIVER_MODELS[esp_drive_kind.value])
        esp_cannon_kind.value=data.get('cannon_kind','mg996r')
        esp_cannon_driver.value=data.get('cannon_driver_model','MG996R-PWM')
        esp_gear_ratio.value=float(data.get('gear_ratio',1))
        esp_outputs.value = bool(data.get("physical_outputs", False))
        esp_jet_output.value = bool(data.get("jet_output_enabled", False))
        esp_ssid.value = ""
        esp_password.value = ""

        esp_notice(
            "OK",
            "Setup carregado",
            f"{name} está disponível para edição. Após salvar mudanças, "
            "reative a combinação oficial na seção 7.",
        )
    except Exception as exc:
        esp_notice("FALHA", "Não foi possível carregar", str(exc))

    esp_update_fields()


def esp_remove_saved(_):
    try:
        if not esp_saved.value:
            raise ValueError("Selecione um setup salvo para excluir.")

        name = esp_saved.value
        setup_delete("hardware", name)
        esp_refresh_saved()
        esp_notice("OK", "Setup excluído", f"{name} foi excluído.")
    except Exception as exc:
        esp_notice("FALHA", "Não foi possível excluir", str(exc))

    esp_update_guide()


esp_manual_group.children = (
    esp_manual,
    esp_hint(
        "Informe a porta como aparece no Gerenciador de Dispositivos. "
        "Este campo só substitui a lista quando você seleciona "
        "“Informar porta manualmente”."
    ),
)

esp_wifi_group.children = ()

esp_local_group.children = (esp_hint('O ESP32 cria a rede BLAZE com endereço fixo 192.168.4.1. O nome e a senha são gerados ao salvar e aparecem no guia.'),)
esp_bluetooth_group.children = ()

esp_left = widgets.VBox(
    [
        widgets.HTML(
            '<h3 style="margin:0">ESP32 — conexão e hardware</h3>'
            '<p style="margin:4px 0 12px">'
            "Preencha o formulário seguindo o guia à direita.</p>"
        ),
        esp_form_status,
        widgets.HBox([esp_find]),
        esp_ports,
        esp_hint(
            "No Windows, Silicon Labs CP210x costuma identificar "
            "a ponte USB do ESP32. Confirme desconectando apenas a placa."
        ),
        esp_manual_group,
        esp_transport,
        esp_wifi_group,
        esp_local_group,
        esp_bluetooth_group,
        widgets.HTML(
            '<h4 style="margin:12px 0 4px">Firmware e diagnóstico</h4>'
        ),
        esp_checkbox_row(
            esp_outputs,
            "Habilita saídas físicas no firmware gerado. Deixe desmarcada "
            "no primeiro teste; marque após conferir GPIOs e fiação.",
        ),
        esp_checkbox_row(
            esp_jet_output,
            "Apenas em combate real: habilita o relé da válvula do jato quando "
            "o circuito físico, a pinagem e os intertravamentos tiverem sido verificados. "
            "Fica desligado na simulação.",
        ),
        esp_checkbox_row(
            esp_flash,
            "Autoriza o upload na seção 8. Marque quando selecionar "
            "uma COM real e estiver pronto para gravar o ESP32.",
        ),
        esp_checkbox_row(
            esp_motors,
            "Autoriza o diagnóstico das esteiras na seção 10, após "
            "preparar ponte H, motores, alimentação e área livre.",
        ),
        esp_checkbox_row(
            esp_servos,
            "Autoriza o diagnóstico de pan e tilt na seção 10, "
            "após preparar alimentação, terra comum e curso livre.",
        ),
        esp_checkbox_row(
            esp_relays,
            "Autoriza o diagnóstico dos relés na seção 10, "
            "após conferir cargas e isolamento.",
        ),
        widgets.HTML('<h4>Conjunto de motores e drivers</h4><p>Os dois motores DC usam um BTS7960 por motor e PWM de sentido. A bateria alimenta o driver, nunca o GPIO. BLDC Kalatec requer driver KTC-J-DR-BLDC750 e referência 0–5 V; HT23 requer KTC-DR23EC-K com pulso/direção. Estes dois conjuntos futuros ficam selecionáveis e documentados, com acionamento físico bloqueado até validar suas interfaces. A redução é a razão motor/saída; calibração de RPM é na saída.</p>'),
        esp_drive_kind, esp_drive_model, esp_drive_driver,
        esp_cannon_kind, esp_cannon_driver, esp_gear_ratio,
        widgets.HTML('<h4 style="margin:12px 0 4px">Setups de hardware</h4>'),
        esp_saved,
        esp_name,
        esp_hint(
            "Salvar com o mesmo nome atualiza o setup. "
            "Carregar permite editá-lo antes de salvar novamente."
        ),
        widgets.Box(
            [esp_load, esp_save, esp_delete],
            layout=widgets.Layout(
                display="flex",
                flex_flow="row wrap",
                grid_gap="8px",
                width="100%",
            ),
        ),
    ],
    layout=widgets.Layout(
        width="100%",
        flex="1 1 650px",
        min_width="380px",
    ),
)

esp_right = widgets.VBox(
    [esp_guide],
    layout=widgets.Layout(
        width="100%",
        flex="1 1 390px",
        min_width="340px",
    ),
)

esp_find.on_click(esp_refresh_ports)
esp_save.on_click(esp_apply)
esp_load.on_click(esp_load_saved)
esp_delete.on_click(esp_remove_saved)

for field in (
    esp_ports,
    esp_manual,
    esp_transport,
    esp_outputs,
    esp_jet_output,
    esp_flash,
    esp_motors,
    esp_servos,
    esp_relays,
    esp_name,
    esp_drive_kind,esp_drive_model,esp_drive_driver,esp_cannon_kind,esp_cannon_driver,esp_gear_ratio,
):
    callback = (
        esp_update_fields
        if field in (esp_ports, esp_transport)
        else esp_update_guide
    )
    field.observe(callback, names="value")

esp_drive_kind.observe(esp_update_motor_fields,names='value')
esp_cannon_kind.observe(esp_update_motor_fields,names='value')
esp_refresh_saved()
esp_refresh_ports()
esp_update_fields()

display(
    widgets.HTML(
        "<style>"
        ".container{width:98% !important;max-width:1600px !important}"
        ".jp-NotebookPanel-notebook{max-width:none !important}"
        "</style>"
    )
)
display(
    widgets.Box(
        [esp_left, esp_right],
        layout=widgets.Layout(
            display="flex",
            flex_flow="row wrap",
            align_items="flex-start",
            grid_gap="18px",
            width="100%",
        ),
    )
)

## 7. Salvar uma combinação completa e definir a oficial

As listas mostram apenas os setups que você salvou. Deixe qualquer grupo em **Default** para usar os valores padrão, sem criar um item salvo. O nome automático reúne os quatro grupos separados por `*`; cada grupo personalizado indica informações como transporte do ESP, fonte da câmera, modo de visão, velocidade e tipo de motor/driver. A data e hora são registradas junto à combinação. Clique **Salvar combinação** e depois **Definir como setup oficial**. Se nenhum grupo for selecionado, o nome será `default*default*default*default`. Para editar um setup de sistema, retorne à seção correspondente; para atualizar o firmware e os runtimes, prossiga à seção 8.

A calibração física da câmera móvel e do canhão é salva separadamente com o nome da combinação completa pela janela inicial do runtime. Se você modificar a montagem ou os pontos de referência, repita os 12 registros; editar esses registros não exige recompilar o firmware.


In [ ]:
# Seção 7 — selecionar até quatro setups; ausência usa os defaults de fábrica.
import datetime as _final_dt
import ipywidgets as widgets
from IPython.display import display

FINAL_SECTIONS=(('connection','Conexão e câmera'),('vision','Visão e jato'),
                ('control','Xbox e velocidade'),('hardware','ESP32 e motores'))
final_section_pickers={}
for key,title in FINAL_SECTIONS:
    names=setup_names(key)
    final_section_pickers[key]=widgets.Dropdown(
        options=[('Default (sem setup salvo)','')]+[(n,n) for n in names],
        description=title+':',style={'description_width':'170px'},
        layout=widgets.Layout(width='100%'))
final_name=widgets.Text(description='Nome completo:',disabled=True,
    style={'description_width':'120px'},layout=widgets.Layout(width='100%'))
final_pick=widgets.Dropdown(options=[('Nova combinação','')],description='Salvas:',
    style={'description_width':'120px'},layout=widgets.Layout(width='100%'))
final_status=widgets.HTML(layout=widgets.Layout(width='100%'))
final_official=widgets.HTML(layout=widgets.Layout(width='100%'))
final_load=widgets.Button(description='Carregar para editar',button_style='info',
    layout=widgets.Layout(min_width='190px',height='38px'))
final_save=widgets.Button(description='Salvar combinação',button_style='success',
    layout=widgets.Layout(min_width='195px',height='38px'))
final_delete=widgets.Button(description='Excluir combinação',button_style='danger',
    layout=widgets.Layout(min_width='175px',height='38px'))
final_activate=widgets.Button(description='Definir como setup oficial',button_style='primary',
    layout=widgets.Layout(min_width='225px',height='38px'))

def final_message(title,state,message):
    final_status.value=status_box(title,state,message).value

def final_refresh_official():
    name=official_name()
    final_official.value=status_box('Atualmente oficial','OK' if name else 'PENDENTE',
        name or 'Nenhum. Salve e ative uma combinação para usar os programas independentes.').value

def final_selections():
    return {section:pick.value or '' for section,pick in final_section_pickers.items()}

def final_segment(section,choice):
    if not choice:
        return 'default'
    payload=setup_get(section,choice)
    if section=='connection':
        transport=payload.get('esp',{}).get('transport','auto')
        source=payload.get('camera',{}).get('source',0)
        camera=('videoSalvo' if isinstance(source,str) else 'cameraUSB'+str(source))
        return f'{choice}_ESP-{transport}_{camera}'
    if section=='vision':
        vision=payload.get('vision',{})
        source=vision.get('camera_source',0)
        medium=('videoSalvo' if vision.get('source_type')=='video' else 'cameraUSB')
        return f'{choice}_{"simulacao" if vision.get("combat_mode")=="simulation" else "combateReal"}_{medium}'
    if section=='control':
        rpm=payload.get('control',{}).get('speed_limit_rpm',80)
        return f'{choice}_Xbox_{rpm}RPM'
    hw=payload.get('hardware',{})
    drive=hw.get('drive_kind','dc_3_6v')
    model={'dc_3_6v':'MotorDC3-6V','bts7960_12v':'Mabuchi12V',
           'bldc_ktc86':'KTC-BLDC'}.get(drive,drive)
    return f'{choice}_{model}_{hw.get("drive_driver_model","BTS7960")}_ESP-{hw.get("transport","auto")}'

def final_auto_name(selections):
    # '*' separa as quatro seções. Cada seção não salva aparece como default.
    return '*'.join(final_segment(section,selections[section]) for section,_ in FINAL_SECTIONS)

def final_preview(_=None):
    try:final_name.value=final_auto_name(final_selections())
    except Exception as exc:final_name.value=f'Corrija o setup selecionado: {exc}'

for picker in final_section_pickers.values():
    picker.observe(final_preview,names='value')

def refresh_combinations(selected=''):
    names=setup_names('combinations')
    options=[('Nova combinação','')]
    for name in names:
        saved=setup_get('combinations',name).get('saved_at','')
        options.append((f'{name}  ·  {saved[:19]}' if saved else name,name))
    final_pick.options=options
    final_pick.value=selected if selected in names else ''
    final_preview()

def save_combination(_):
    try:
        selected=final_selections()
        assemble_combination(selected)  # Valida defaults e escolhas salvas.
        name=final_auto_name(selected)
        now=_final_dt.datetime.now().astimezone()
        setup_save('combinations',name,{'sections':selected,'saved_at':now.isoformat()})
        refresh_combinations(name)
        final_message('Combinação','OK',
            f'{name} salvo. Clique em Definir como setup oficial para preparar PC e ESP32.')
    except Exception as exc:
        final_message('Combinação','FALHA',str(exc))

def load_combination(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Selecione uma combinação salva.')
        selected=setup_get('combinations',name)['sections']
        for section,picker in final_section_pickers.items():
            value=selected.get(section,'')
            if value not in [item for _,item in picker.options]:
                raise ValueError(f'Setup {value!r} ausente em {section}.')
            picker.value=value
        final_name.value=name
        final_message('Combinação','OK',f'{name} carregado. Altere as seleções e salve.')
    except Exception as exc:final_message('Combinação','FALHA',str(exc))

def delete_combination(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Selecione a combinação a excluir.')
        setup_delete('combinations',name)
        refresh_combinations();final_refresh_official()
        final_message('Combinação','OK',f'{name} excluído.')
    except Exception as exc:final_message('Combinação','FALHA',str(exc))

def activate_official(_):
    try:
        name=final_pick.value
        if not name:raise ValueError('Salve e selecione uma combinação primeiro.')
        from profile_manager import prepare
        _,data=prepare(name)
        data['_official']['activated_at']=_final_dt.datetime.now(
            _final_dt.timezone.utc).isoformat()
        _json_atomic(SETUPS_DIR/'official'/'active.json',data)
        final_refresh_official()
        warning=(' Calibre a RPM máxima para comandar tração física.'
                 if float(data.get('geometry',{}).get('output_max_rpm',0))==0 else '')
        final_message('Setup oficial','OK',f'{name} ativado para os runtimes.{warning}')
    except Exception as exc:final_message('Setup oficial','FALHA',str(exc))

final_load.on_click(load_combination)
final_save.on_click(save_combination)
final_delete.on_click(delete_combination)
final_activate.on_click(activate_official)
refresh_combinations();final_refresh_official()
display(widgets.VBox([
    widgets.HTML('<h3>7. Combinação completa</h3><p>Selecione os setups salvos de cada seção. '
        'Se uma lista estiver vazia ou mantida em Default, o pacote aplicará os parâmetros padrão '
        'sem criar um setup salvo. Os quatro trechos do nome são separados por *; os trechos '
        'selecionados incluem conexão, fonte de vídeo, modo e motor/driver. Uma nova gravação '
        'com as mesmas seleções atualiza a data do registro.</p>'),
    final_official,*final_section_pickers.values(),final_name,final_pick,
    widgets.HBox([final_load,final_save,final_delete,final_activate],
        layout=widgets.Layout(flex_flow='row wrap',grid_gap='8px')),
    final_status],layout=widgets.Layout(width='100%',max_width='1400px')))


## 8. Preparar programas independentes e firmware

Selecione **uma combinação completa** já salva na seção 7. O notebook monta juntos câmera, visão, controle, conexão e hardware; não é necessário escolher novamente os setups de cada seção. **Preparar runtimes** gera os arquivos de configuração dos programas de simulação e combate real. **Compilar** cria um projeto de firmware separado para a combinação selecionada, sem substituir outro perfil. **Gravar** compila e envia esse firmware ao ESP32 pela porta USB indicada, com andamento e registro abaixo dos botões.

Na simulação, as saídas físicas são desabilitadas no firmware. O combate real somente aceita combinações com saídas físicas habilitadas. Para gravar, feche o monitor serial; segure BOOT, pressione e solte EN, clique Gravar e solte BOOT quando o ESP32 for reconhecido. A operação usa USB ou rede local criada pelo ESP32; a gravação usa somente USB. A primeira gravação é por USB. A primeira preparação gera credenciais exclusivas locais: após prepará-las, compile e grave o novo firmware.
A combinação preparada mantém a calibração física em um registro separado, atualizado pelo botão **Calibrar câmera e canhão** na janela inicial do runtime. Assim, repetir a calibração não exige recompilar o firmware.


In [ ]:
# Seção 8 — preparar runtimes e firmware com partição adequada para ESP32 de 4 MB.
import re
import threading
import html
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from profile_manager import names as profile_names, prepare_all, assemble, run_pio

fw_choice = widgets.Dropdown(description='Setup completo:',
    style={'description_width':'135px'}, layout=widgets.Layout(width='620px'))
fw_port = widgets.Text(description='Porta para gravar:',
    style={'description_width':'135px'}, layout=widgets.Layout(width='300px'),
    placeholder='COM4')
fw_port_row = widgets.HBox([fw_port])
fw_port_row.layout.display = 'none'
fw_refresh = widgets.Button(description='Atualizar combinações',
    layout=widgets.Layout(min_width='190px'))
fw_export = widgets.Button(description='Preparar runtimes', button_style='success',
    layout=widgets.Layout(min_width='190px'))
fw_build = widgets.Button(description='Compilar firmware selecionado', button_style='info',
    layout=widgets.Layout(min_width='255px'))
fw_upload = widgets.Button(description='Gravar firmware somente por cabo USB', button_style='warning',
    layout=widgets.Layout(min_width='290px'))
fw_status = widgets.HTML(layout=widgets.Layout(width='100%'))
fw_connection = widgets.HTML('<b>Gravação:</b> aguardando operação.')
fw_progress = widgets.IntProgress(value=0, min=0, max=100, description='Gravação:',
    layout=widgets.Layout(width='100%', display='none'))
fw_log = widgets.HTML(layout=widgets.Layout(width='100%', height='230px', overflow='auto'))
_fw_busy = threading.Lock()
_fw_lines = []

def fw_ensure_partition():
    """Usa uma partição real de 3 MB, sem OTA, na flash de 4 MB."""
    ini = Path(ROOT) / 'esp32' / 'platformio.ini'
    original = ini.read_text(encoding='utf-8')
    header = re.search(r'(?m)^\[env:esp32dev\]\s*$', original)
    if not header:
        raise ValueError(f'Não encontrei [env:esp32dev] em {ini}')
    start = header.end()
    next_section = re.search(r'(?m)^\s*\[[^\]]+\]', original[start:])
    end = start + next_section.start() if next_section else len(original)
    section = original[start:end]
    for key, value in (
        ('board_build.partitions', 'huge_app.csv'),
        ('board_upload.maximum_size', '3145728'),
    ):
        pattern = rf'(?m)^[ \t]*{re.escape(key)}[ \t]*=[^\r\n]*'
        replacement = f'{key} = {value}'
        if re.search(pattern, section):
            section = re.sub(pattern, lambda _: replacement, section, count=1)
        else:
            section = '\n' + replacement + section
    updated = original[:start] + section + original[end:]
    if updated != original:
        ini.write_text(updated, encoding='utf-8')
    return ini

def fw_refresh_choices(_=None):
    choices = profile_names()
    previous = fw_choice.value
    fw_choice.options = [(name, name) for name in choices]
    fw_choice.value = previous if previous in choices else choices[0] if choices else None
    if fw_choice.value:
        data = assemble(fw_choice.value)
        fw_port.value = str(data.get('hardware', {}).get('upload_port') or
                            data.get('esp', {}).get('serial_port') or '')
        fw_status.value = status_box('Combinação completa', 'OK',
            f"{fw_choice.value}: {data['_official']['profile']}.").value
    else:
        fw_status.value = status_box('Combinação completa', 'PENDENTE',
            'Salve uma combinação completa na seção 7.').value

def fw_print(line):
    _fw_lines.append(line)
    del _fw_lines[:-250]
    fw_log.value = ('<pre style="font-size:11px;white-space:pre-wrap;overflow-wrap:anywhere">'
                    + html.escape('\n'.join(_fw_lines[-150:])) + '</pre>')
    if 'Connecting' in line:
        fw_connection.value = '<b>Connecting...</b> aguardando resposta do ESP32.'
    if 'Chip is ESP32' in line:
        fw_connection.value = '<b>ESP32 reconhecido.</b> Solte BOOT.'
    match = re.search(r'Writing at[^\n]*\((\d+)\s*%\)', line)
    if match:
        fw_progress.value = min(100, int(match.group(1)))

def fw_start(action):
    if action == 'upload' and fw_port_row.layout.display == 'none':
        fw_port_row.layout.display = ''
        fw_status.value = status_box('Porta de gravação USB', 'PENDENTE',
            'A comunicação operacional pode ser Wi-Fi, mas a gravação atual do firmware '
            'usa cabo USB. Confira a COM do ESP32 e clique Gravar novamente.').value
        return
    if not _fw_busy.acquire(blocking=False):
        return
    fw_export.disabled = fw_build.disabled = fw_upload.disabled = True
    fw_progress.layout.display = '' if action == 'upload' else 'none'
    fw_progress.value = 0
    _fw_lines.clear()
    def worker():
        try:
            if action == 'export':
                report = prepare_all()
                result = (f"Simulação: {report['simulacao']}; combate real: "
                          f"{report['combate_real']}. Pendências: {report['erros'] or 'nenhuma'}")
            else:
                if not fw_choice.value:
                    raise ValueError('Escolha uma combinação completa.')
                ini = fw_ensure_partition()
                fw_print(f'Partição de 3 MB configurada em: {ini}')
                fw_print('Tabela huge_app.csv: 3 MB para firmware, sem atualização OTA.')
                binary = run_pio(fw_choice.value, action == 'upload', fw_port.value, fw_print)
                result = (f'Firmware {fw_choice.value} gravado em {fw_port.value}.'
                          if action == 'upload' else f'Firmware compilado: {binary}')
                if action == 'upload':
                    fw_progress.value = 100
            fw_status.value = status_box('Runtimes e firmware', 'OK', result).value
        except Exception as exc:
            fw_status.value = status_box('Runtimes e firmware', 'FALHA',
                                        f'{type(exc).__name__}: {exc}').value
        finally:
            fw_export.disabled = fw_build.disabled = fw_upload.disabled = False
            _fw_busy.release()
    threading.Thread(target=worker, daemon=True).start()

def fw_changed(change):
    if change['new']:
        try:
            data = assemble(change['new'])
            fw_port.value = str(data.get('hardware', {}).get('upload_port') or
                                data.get('esp', {}).get('serial_port') or '')
            fw_status.value = status_box('Combinação completa', 'OK',
                f"{change['new']} ({data['_official']['profile']}) selecionada.").value
        except Exception as exc:
            fw_status.value = status_box('Combinação completa', 'FALHA', str(exc)).value

fw_choice.observe(fw_changed, names='value')
fw_refresh.on_click(fw_refresh_choices)
fw_export.on_click(lambda _: fw_start('export'))
fw_build.on_click(lambda _: fw_start('build'))
fw_upload.on_click(lambda _: fw_start('upload'))
try:
    fw_refresh_choices()
except Exception as exc:
    fw_status.value = status_box('Configuração', 'FALHA', str(exc)).value

display(widgets.VBox([
    widgets.HTML('<b>Escolha uma combinação completa.</b> Preparar runtimes gera os '
                 'arquivos de configuração; Compilar cria o firmware com partição de 3 MB. '
                 'Ao clicar Gravar, a porta COM aparece para a gravação por cabo; a comunicação '
                 'operacional permanece conforme o setup Wi-Fi ou USB.'),
    widgets.HBox([fw_choice, fw_refresh]),
    widgets.HBox([fw_connection]), fw_port_row,
    widgets.HBox([fw_export, fw_build, fw_upload]),
    fw_status, fw_progress,
    widgets.HTML('Antes de gravar: feche o monitor serial, segure BOOT, pressione e solte EN; '
                 'mantendo BOOT, clique Gravar. Solte BOOT quando aparecer Chip is ESP32.'),
    fw_log
], layout=widgets.Layout(width='100%')))


## 9. Comunicação e latência PC ↔ ESP32

**Necessário:** ESP32 alimentado e com firmware da combinação oficial gravado pela USB. Para transporte USB, conecte o cabo e libere a COM. Para a rede local, ligue o Wi-Fi do PC e deixe a rede `BLAZE-XXXXXX` visível; o runtime tentará entrar nela e falar com `192.168.4.1:8765`. No modo automático, tenta USB antes da rede. Câmera, Xbox e motores não são necessários.

O teste transmite comandos neutros de parada por até 30 segundos pela rede local (10 segundos pela USB) e mede a resposta de telemetria. A barra mostra o tempo decorrido; se falhar, a mensagem inclui a última tentativa. Uma conexão ativa não comprova acionamento físico.


In [ ]:
import statistics
link_status=widgets.HTML()
link_button=widgets.Button(description='Medir comunicação',button_style='info',layout=widgets.Layout(width='260px'))
link_output=widgets.Output()
link_progress=widgets.IntProgress(value=0,min=0,max=10,description='Medição:',layout=widgets.Layout(width='420px'))

def measure_link(_):
    with link_output:
        link_output.clear_output()
        link=None
        try:
            official=ROOT/'pc_operador'/'parameters_setups'/'official'/'active.json'
            if not official.is_file():raise FileNotFoundError('Defina um setup oficial na seção 7.')
            config=AppConfig(official)
            esp=config.data['esp'];sent={};latencies=[];received=[]
            serial_port=str(esp.get('serial_port','auto'));host=str(esp.get('host','auto'))
            seq=[0]
            def command():
                seq[0]+=1;sent[seq[0]]=time.monotonic()
                return control_message({'requested_mode':'manual','stop':True,'seq':seq[0]}, {})
            def telemetry(data):
                now=time.monotonic();received.append(data)
                origin=sent.pop(data.get('seq'),None)
                if origin is not None:latencies.append((now-origin)*1000)
            link=ESPConnection(host,int(esp.get('port',8765)),command,telemetry,
                transport=str(esp.get('transport','auto')),serial_port=serial_port,
                bluetooth_port=str(esp.get('bluetooth_port','auto')),
                preserve_wifi=False)
            link_progress.value=0
            duration=30 if esp.get('transport') in ('wifi_local','auto') else 10
            link_progress.max=duration
            link.start();started=time.monotonic();deadline=started+duration
            while time.monotonic()<deadline:
                time.sleep(.1)
                link_progress.value=min(duration,int(time.monotonic()-started))
                if received and time.monotonic()-started>=3:break
            if not received:
                detail=link.last_error or link.attempting or 'nenhuma tentativa concluída'
                raise TimeoutError(f'Nenhuma telemetria. Transporte {esp.get("transport")}; última tentativa: {detail}. '
                    'Para rede local, grave o firmware com este setup e confira se a rede BLAZE aparece; '
                    'para USB, confira a COM e feche o monitor serial.')
            summary={'transporte':link.active_transport or 'desconectado',
                     'recebidos':len(received),'rtt_mediana_ms':round(statistics.median(latencies),1) if latencies else None,
                     'rtt_max_ms':round(max(latencies),1) if latencies else None,
                     'motivo':received[-1].get('reason')}
            results['comunicacao']=summary
            link_status.value=status_box('Comunicação PC–ESP32','OK',str(summary)).value
            print(json.dumps(summary,ensure_ascii=False,indent=2))
        except Exception as exc:
            results['comunicacao']={'estado':'falha','erro':str(exc)}
            link_status.value=status_box('Comunicação PC–ESP32','FALHA',str(exc)).value
        finally:
            if link is not None:link.stop()
            link_progress.value=link_progress.max if received else link_progress.value
link_button.on_click(measure_link)
display(widgets.VBox([link_status,link_button,link_progress,link_output]))


## 10. Sensores e diagnósticos físicos

**Necessário:** ESP32 com firmware gravado, porta USB selecionada. O botão de sensores pede `STATUS` e lê os dois DHT11 sem acionar saídas. Os botões dos atuadores executam pulsos curtos apenas nos canais autorizados no setup oficial, com alimentação, fiação e área livre verificadas fisicamente. A resposta `DIAG DONE` confirma o firmware, não a resposta mecânica; registre o que foi observado.

O diagnóstico de **SERVO_CAMERA** move apenas a plataforma MG90S no GPIO4 para uma posição breve. A calibração completa de 12 pontos acontece depois, no seletor inicial do runtime de combate real.

Para cada conjunto de tração, o notebook confere a versão e seleciona os testes compatíveis: os dois arranjos com BTS7960 recebem pulsos limitados (esquerda/direita), enquanto BLDC KTC depende da interface do driver antes de qualquer pulso. Para o canhão HT23, o teste PWM MG996R fica bloqueado. Os dois relés são testados separadamente. **DIAG START/DONE não mede a presença física de motor, driver ou relé.**


In [ ]:
# Seção 10 — sensores e diagnósticos físicos do ESP32.
# Substitua a célula inteira por este código. Requer a célula inicial (ROOT)
# e o setup oficial definido na seção 7.
import html
import json
import re
import threading
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from serial.tools import list_ports

_s10_root = Path(ROOT)
_s10_setups = Path(globals().get(
    'SETUPS_DIR', _s10_root / 'pc_operador' / 'parameters_setups'))
_s10_official = _s10_setups / 'official' / 'active.json'
_s10_pins = _s10_root / 'esp32' / 'include' / 'pins.h'
_s10_lock = threading.Lock()
_s10_results = globals().setdefault('results', {})


def _s10_card(title, description, requirements, outcome, color='#e2e8f0'):
    return widgets.HTML(
        '<div style="box-sizing:border-box;width:100%;padding:12px 14px;'
        f'border:1px solid {color};border-radius:9px;line-height:1.5;'
        'white-space:normal;overflow-wrap:anywhere">'
        f'<b>{title}</b><br><b>Função:</b> {description}<br>'
        f'<b>Requisitos:</b> {requirements}<br>'
        f'<b>Resultado esperado:</b> {outcome}</div>',
        layout=widgets.Layout(width='100%'))


def _s10_message(label, state, detail):
    colors = {'OK': ('#ecfdf3', '#166534'),
              'PENDENTE': ('#fff8e6', '#875d00'),
              'FALHA': ('#fef0ef', '#b42318'),
              'TESTANDO': ('#eff6ff', '#1d4ed8')}
    bg, fg = colors.get(state, ('#f1f5f9', '#334155'))
    return (f'<div style="background:{bg};color:{fg};padding:8px 11px;'
            'border-radius:7px;white-space:normal;overflow-wrap:anywhere">'
            f'<b>{html.escape(label)} — {state}</b><br>'
            f'{html.escape(str(detail))}</div>')


def _s10_config():
    if not _s10_official.is_file():
        raise RuntimeError('Defina uma combinação oficial na última célula.')
    data = json.loads(_s10_official.read_text(encoding='utf-8'))
    if not isinstance(data, dict) or not data.get('_official', {}).get('name'):
        raise RuntimeError('O active.json não contém um setup oficial válido.')
    hw = data.get('hardware') or {}
    if not isinstance(hw, dict):
        raise RuntimeError('O setup oficial contém hardware inválido.')
    return data, hw, (hw.get('ensaios') or {})


def _s10_port():
    _, hw, _ = _s10_config()
    configured = str(hw.get('upload_port') or '').strip()
    if configured.lower() not in ('', 'auto', 'manual', 'none'):
        return configured
    found = list(list_ports.comports())
    likely = [p.device for p in found if re.search(
        r'CP210|Silicon Labs|CH340|CH341', p.description or '', re.I)]
    if len(likely) == 1:
        return likely[0]
    raise RuntimeError('Selecione a COM real do ESP32 na seção 6, salve e '
                       'ative novamente o setup oficial na última célula.')


def _s10_output_macro():
    # Perfil efetivamente compilado, não o cabeçalho padrão da árvore-fonte.
    from profile_manager import FIRMWARE_BUILDS, _safe, _build_dir_name
    data = json.loads(_s10_official.read_text(encoding='utf-8'))
    meta = data.get('_official', {})
    if not meta.get('name') or not meta.get('firmware_signature'):
        return None
    project = FIRMWARE_BUILDS / meta.get('profile', '') / _build_dir_name(meta['name'])
    if not project.is_dir():
        project = FIRMWARE_BUILDS / meta.get('profile', '') / _safe(meta['name'])  # build legado
    header = project / 'include' / 'setup_generated.h'
    binary = project / '.pio' / 'build' / 'esp32dev' / 'firmware.bin'
    if not header.is_file() or not binary.is_file():
        return None
    text = header.read_text(encoding='utf-8')
    if f'BLAZE_SETUP_SIGNATURE "{meta["firmware_signature"]}"' not in text:
        return None
    match = re.search(r'^\s*#\s*define\s+BLAZE_ENABLE_PHYSICAL_OUTPUTS\s+([01])\b',
                      text, re.M)
    return int(match.group(1)) if match else None


def _s10_open():
    import serial
    port = _s10_port()
    connection = serial.Serial(port, baudrate=115200, timeout=0.25,
                               write_timeout=2)
    time.sleep(2.5)  # abrir a COM pode reiniciar o ESP32
    connection.reset_input_buffer()
    return connection, port


def _s10_version_line(connection, expected):
    """A saída de canal configurado não prova conexão física; assinatura evita ensaio com outro mapa."""
    connection.write(b'VERSION\n');connection.flush()
    deadline=time.monotonic()+4.
    while time.monotonic()<deadline:
        line=connection.readline().decode('utf-8','replace').strip()
        if not line.startswith('VERSION,'):continue
        fields=line.split(',')
        if len(fields)!=5:raise RuntimeError('Resposta VERSION incompleta: '+line)
        meta=expected.get('_official',{})
        drive=expected.get('hardware',{}).get('drive_kind','bts7960_12v')
        cannon=expected.get('hardware',{}).get('cannon_kind','mg996r')
        drive_id={'dc_3_6v':1,'bts7960_12v':2,'bldc_ktc86':3}[drive]
        cannon_id={'mg996r':1,'ht23_ec1000':2}[cannon]
        if (fields[1]!=meta.get('name') or fields[2]!=meta.get('firmware_signature')
                or int(fields[3])!=drive_id or int(fields[4])!=cannon_id):
            raise RuntimeError('Firmware gravado diferente do setup oficial/motores: '+line)
        return fields
    raise TimeoutError('Sem resposta VERSION: atualize o firmware antes de ensaiar saídas físicas.')

def _s10_sensor_line(connection):
    connection.write(b'STATUS\n')
    connection.flush()
    end = time.monotonic() + 5.0
    while time.monotonic() < end:
        line = connection.readline().decode('utf-8', 'replace').strip()
        if line.startswith('SENSOR,'):
            fields = line.split(',')
            if len(fields) != 7 or fields[1] not in ('0', '1') or fields[4] not in ('0', '1'):
                raise RuntimeError('Resposta SENSOR incompleta: ' + line)
            return fields
    raise TimeoutError('Nenhuma resposta SENSOR. Confira a COM, feche o monitor '
                       'serial e confirme que o firmware deste notebook foi gravado.')


def _s10_read_sensor(_):
    if not _s10_lock.acquire(blocking=False):
        return
    s10_sensor_status.value = _s10_message('DHT11', 'TESTANDO', 'Consultando por USB...')
    s10_read.disabled = s10_pulse.disabled = True
    try:
        connection, port = _s10_open()
        with connection:
            values = _s10_sensor_line(connection)
        def sensor(label, valid, temp, humidity):
            if valid == '0':
                return f'{label}: ausente ou leitura inválida'
            return f'{label}: {float(temp):.1f} °C; {float(humidity):.1f} % UR'
        internal = sensor('Interno (GPIO 16)', *values[1:4])
        external = sensor('Externo (GPIO 17)', *values[4:7])
        valid_count = int(values[1]) + int(values[4])
        state = 'OK' if valid_count else 'PENDENTE'
        detail = (f'ESP32 respondeu em {port}. {internal}. {external}. '
                  + ('Nenhum DHT retornou uma medida válida; conecte o sensor e repita.'
                     if not valid_count else 'Leitura USB concluída.'))
        s10_sensor_status.value = _s10_message('DHT11', state, detail)
        _s10_results['sensores_usb'] = {'porta': port, 'interno_valido': bool(int(values[1])),
                                      'externo_valido': bool(int(values[4])), 'resposta': values}
    except Exception as exc:
        s10_sensor_status.value = _s10_message('DHT11', 'FALHA', exc)
    finally:
        s10_read.disabled = s10_pulse.disabled = False
        _s10_lock.release()


_s10_channels = (
    ('motors', 'MOTOR_L', 0.15, 600, 'Esteira esquerda'),
    ('motors', 'MOTOR_R', 0.15, 600, 'Esteira direita'),
    ('servos', 'SERVO_PAN', 110, 700, 'Servo horizontal do canhão'),
    ('servos', 'SERVO_TILT', 110, 700, 'Servo vertical do canhão'),
    ('servos', 'SERVO_CAMERA', 110, 700, 'MG90S da plataforma da câmera'),
    ('relays', 'COOL_INT', 1, 300, 'Relé do arrefecimento interno'),
    ('relays', 'COOL_EXT', 1, 300, 'Relé do arrefecimento externo'),
)


def _s10_run_pulse(_):
    if not _s10_lock.acquire(blocking=False):
        return
    s10_read.disabled = s10_pulse.disabled = True
    try:
        data, hw, flags = _s10_config()
        drive=hw.get('drive_kind','bts7960_12v')
        cannon=hw.get('cannon_kind','mg996r')
        staged=[]
        if flags.get('motors') and drive=='bldc_ktc86':
            staged.append('BLDC: driver analógico, direção e retorno de falha ainda não integrados; não enviar PWM BTS.')
        if flags.get('servos') and cannon=='ht23_ec1000':
            staged.append('HT23: dois drivers pulso/direção e retorno de falha ainda não integrados; não enviar PWM MG996R.')
        permitted=[row for row in _s10_channels if flags.get(row[0],False)
                   and not (row[0]=='motors' and drive=='bldc_ktc86')
                   and not (row[0]=='servos' and row[1] in ('SERVO_PAN','SERVO_TILT') and cannon=='ht23_ec1000')]
        if not permitted:
            raise RuntimeError('Nenhum grupo foi autorizado. Na seção 6, selecione '
                               'Motores, Servos e/ou Relés; salve e ative o setup oficial. '+ '; '.join(staged))
        if not hw.get('physical_outputs', False):
            raise RuntimeError('Saídas físicas estão desabilitadas no setup oficial. '
                               'Configure na seção 6, ative o setup e regrave o firmware.')
        if _s10_output_macro() != 1:
            raise RuntimeError('O código atual indica saídas físicas desabilitadas. '
                               'Gere, compile e grave o firmware da seção 8 com o setup oficial.')
        if not s10_ready.value:
            raise RuntimeError('Marque a confirmação de fiação, alimentação e área livre '
                               'após conferir fisicamente os grupos autorizados.')
        s10_pulse_status.value = _s10_message(
            'Atuadores', 'TESTANDO', 'Abrindo a COM e iniciando pulsos limitados...')
        log = []
        connection, port = _s10_open()
        with connection:
            _s10_version_line(connection,data)
            for _, channel, value, duration, label in permitted:
                connection.reset_input_buffer()
                connection.write(f'TEST {channel} {value} {duration}\n'.encode('ascii'))
                connection.flush()
                deadline = time.monotonic() + 3.5
                start = done = refused = False
                while time.monotonic() < deadline:
                    line = connection.readline().decode('utf-8', 'replace').strip()
                    if line.startswith('DIAG START'):
                        start = True
                    elif line == 'DIAG DONE':
                        done = True
                        break
                    elif line.startswith('DIAG REFUSED'):
                        refused = line
                        break
                result = ('Comando aceito e encerrado; confirme o movimento observado'
                          if start and done else refused or 'Sem resposta completa do firmware')
                log.append(f'{label}: {result}')
                s10_pulse_status.value = _s10_message(
                    'Atuadores', 'TESTANDO', f'Porta {port}. ' + '; '.join(log))
                if refused or not (start and done):
                    break
                time.sleep(0.2)
        _s10_results['atuadores_usb'] = {'porta': port, 'mensagens': log}
        state = ('PENDENTE' if staged else 'OK') if len(log)==len(permitted) and all(
            'aceito e encerrado' in item for item in log) else 'FALHA'
        s10_pulse_status.value = _s10_message(
            'Atuadores', state, '; '.join(log+staged) +
            '. O retorno DIAG confirma o comando, não comprova movimento físico. '
            'Registre abaixo apenas o que observou.')
    except Exception as exc:
        s10_pulse_status.value = _s10_message('Atuadores', 'FALHA', exc)
    finally:
        s10_read.disabled = s10_pulse.disabled = False
        _s10_lock.release()


s10_read = widgets.Button(description='Ler DHT11 por USB', button_style='info',
                          layout=widgets.Layout(min_width='190px', height='38px'))
s10_pulse = widgets.Button(description='Executar pulsos autorizados',
                           button_style='warning',
                           layout=widgets.Layout(min_width='245px', height='38px'))
s10_ready = widgets.Checkbox(value=False, indent=False,
                             description='Conferi fiação, alimentação e área livre dos grupos autorizados',
                             layout=widgets.Layout(width='100%'))
s10_sensor_status = widgets.HTML(layout=widgets.Layout(width='100%'))
s10_pulse_status = widgets.HTML(layout=widgets.Layout(width='100%'))
try:
    _s10_data,_s10_hw,_s10_flags=_s10_config()
    _s10_drive=_s10_hw.get('drive_kind','bts7960_12v')
    _s10_cannon=_s10_hw.get('cannon_kind','mg996r')
    _s10_pins_desc = ('GPIO25/26/27: BTS esquerdo RPWM/LPWM/EN; GPIO32/33/23: BTS direito RPWM/LPWM/EN.'
                       if _s10_drive in ('dc_3_6v','bts7960_12v') else
                       'BLDC: interface 0–5 V e pinos de direção/falha aguardam projeto e teste do driver.')
    _s10_cannon_desc = ('GPIO18 pan e GPIO19 tilt, PWM MG996R.' if _s10_cannon=='mg996r'
                        else 'HT23: pulso/direção/encoder sem GPIO de atuação atribuído.')
    s10_profile_status=widgets.HTML(_s10_message('Arranjo selecionado','OK',
        f"Tração {_s10_drive} / driver {_s10_hw.get('drive_driver_model','BTS7960')}; "
        f"canhão {_s10_cannon} / driver {_s10_hw.get('cannon_driver_model','MG996R-PWM')}. "
        +_s10_pins_desc+' '+_s10_cannon_desc+
        ' Relés interno GPIO22 e externo GPIO13 são ensaiados separadamente.'))
except Exception as exc:
    s10_profile_status=widgets.HTML(_s10_message('Arranjo selecionado','PENDENTE',exc))
s10_notes = widgets.Textarea(description='Observação física:',
                             placeholder='Ex.: a esteira esquerda girou; o servo pan se moveu; o relé comutou.',
                             style={'description_width': '145px'},
                             layout=widgets.Layout(width='100%', height='95px'))
s10_read.on_click(_s10_read_sensor)
s10_pulse.on_click(_s10_run_pulse)

_s10_sensor_card = _s10_card(
    '1. Ler DHT11 por USB',
    'Envia STATUS ao ESP32 pela porta COM e apresenta validade, temperatura e umidade '
    'do sensor interno (GPIO 16) e do externo (GPIO 17). Não aciona saídas.',
    'ESP32 com firmware gravado, cabo USB de dados, COM correta no setup oficial '
    'e monitor serial fechado. O DHT11 pode estar desconectado para testar apenas '
    'a resposta USB. Para obter medidas reais, conecte DATA ao GPIO correspondente, '
    'VCC a 3V3 e GND a GND com a placa desligada.',
    'Sensor conectado e funcionando: leitura válida. Sensor ausente: leitura '
    'inválida/pendente, embora a comunicação USB possa estar OK.')
_s10_pulse_card = _s10_card(
    '2. Executar pulsos autorizados',
    'Envia, um de cada vez, pulsos curtos às esteiras, servos e relés marcados '
    'na seção 6. Exibe DIAG START/DONE ou o motivo da recusa pelo firmware.',
    'No setup oficial: Saídas físicas habilitadas e ao menos um grupo Motores, '
    'Servos ou Relés autorizado. Após ativá-lo, gere, compile e grave novamente '
    'o firmware. Conecte e confira drivers, fontes, terra comum, cargas e área '
    'livre; feche o monitor serial e marque a confirmação abaixo. Não use com '
    'o controle PC já comandando o robô.',
    'DIAG START/DONE confirma o recebimento e término do comando. Confira '
    'visualmente cada componente e anote o comportamento observado.', '#f0d5a5')

display(widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 6px">10. Sensores e acionamentos de bancada</h3>'
                 '<p style="margin:0 0 10px">Siga a ordem: leia os sensores; '
                 'depois teste somente os atuadores já montados e autorizados.</p>'),
    s10_profile_status, _s10_sensor_card, s10_read, s10_sensor_status,
    _s10_pulse_card, s10_ready, s10_pulse, s10_pulse_status,
    widgets.HTML('<b>Registro do ensaio</b><br>Escreva o que foi observado '
                 'fisicamente. Este campo é apenas uma anotação e não envia '
                 'comandos ao ESP32.'), s10_notes,
], layout=widgets.Layout(width='100%', max_width='1250px', grid_gap='9px')))


## 11. Ensaio integrado PC + ESP32

**Objetivo:** verificar, durante 20 segundos, se o PC lê o controle Xbox e a câmera, executa a visão, envia comandos ao ESP32 e recebe telemetria. Depois, o ensaio interrompe a comunicação e verifica se ela retorna com o controle desarmado. A telemetria não comprova movimento físico dos atuadores.

**Antes de iniciar:**

1. **Setup oficial:** defina na seção 7 uma combinação com câmera, modelos de fogo e pessoas, controle e hardware. Se alterou a configuração do ESP32, gere, compile e grave novamente o firmware na seção 8.
2. **ESP32:** mantenha a placa alimentada e disponível pela porta USB selecionada ou rede local criada pelo ESP32. Conclua o teste de comunicação da seção 9 e feche qualquer monitor serial que esteja ocupando a porta COM.
3. **Xbox:** conecte o controle ao PC e confirme na seção de controle que os comandos são recebidos. Encerre a simulação ou outro teste que esteja lendo o controle.
4. **Câmera e modelos:** disponibilize a câmera selecionada e os modelos de detecção de fogo e pessoas. O vídeo chega diretamente ao PC; não passa pelo ESP32.
5. **Atuadores:** para testar apenas comunicação e telemetria, eles podem ficar desconectados **se as saídas físicas estiverem desabilitadas no firmware gravado**. Com saídas habilitadas, confira as conexões, as fontes e a área livre antes do ensaio. O DHT11 é opcional; sem ele, sua leitura aparecerá inválida.

**Como executar:** rode a célula de código e clique em **Iniciar ensaio integrado**, com os comandos inicialmente neutros. Siga as instruções exibidas em cada fase. A barra mostra os 20 segundos do ensaio principal; a verificação de reconexão pode levar alguns segundos adicionais.

**Relatórios:** ao concluir, a célula mostra os caminhos completos dos arquivos. O resumo fica em `output/testes_pc/ensaio_integrado_resumo.json`; quando o registro de pacotes for gerado, ele fica em `output/testes_pc/integrado_pacotes.json`.

In [ ]:
# Seção 11 — célula completa. Execute depois das seções 6–9.
import html
import json
import os
import queue
import subprocess
import sys
import threading
import time
from datetime import datetime as _integrated_datetime
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display


class _S11Pending(RuntimeError):
    """Pré-requisito que pode ser resolvido antes de executar o ensaio."""


_S11_XBOX_CODE = r'''
import json, pygame, sys, time
pygame.init()
pygame.joystick.init()
count = pygame.joystick.get_count()
index = int(sys.argv[1])
if count == 1 and (index < 0 or index >= count):
    index = 0
if not (0 <= index < count):
    print('ERR:' + f'Índice {index} indisponível; pygame encontrou {count} controle(s).', flush=True)
    sys.exit(2)
joy = pygame.joystick.Joystick(index)
joy.init()
print('INFO:' + json.dumps({'index':index, 'name':joy.get_name(),
     'axes':joy.get_numaxes(), 'buttons':joy.get_numbuttons(),
     'hats':joy.get_numhats()}), flush=True)
try:
    while True:
        pygame.event.pump()
        print('DATA:' + json.dumps({
            'axes':[float(joy.get_axis(i)) for i in range(joy.get_numaxes())],
            'buttons':[int(joy.get_button(i)) for i in range(joy.get_numbuttons())],
            'hats':[list(joy.get_hat(i)) for i in range(joy.get_numhats())]
        }, separators=(',', ':')), flush=True)
        time.sleep(.05)
finally:
    joy.quit()
    pygame.joystick.quit()
    pygame.quit()
'''


class _S11XboxReader:
    """Mesmo método da seção de controle: SDL lê no processo filho."""
    def __init__(self, index):
        env = dict(os.environ)
        env['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
        env['SDL_JOYSTICK_ALLOW_BACKGROUND_EVENTS'] = '1'
        creationflags = getattr(subprocess, 'CREATE_NO_WINDOW', 0) if os.name == 'nt' else 0
        self.process = subprocess.Popen(
            [sys.executable, '-u', '-c', _S11_XBOX_CODE, str(index)],
            stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.PIPE, text=True, bufsize=1, env=env,
            creationflags=creationflags)
        self.lines = queue.Queue()
        self.lock = threading.Lock()
        self.info = None
        self.data = None
        self.stamp = 0.0
        self.error = ''
        threading.Thread(target=self._read_stdout, daemon=True).start()
        deadline = time.monotonic() + 9
        while time.monotonic() < deadline:
            if self.info and self.sample() is not None:
                return
            try:
                line = self.lines.get(timeout=.2)
            except queue.Empty:
                if self.process.poll() is not None:
                    break
                continue
            if line.startswith('INFO:'):
                self.info = json.loads(line[5:])
            elif line.startswith('DATA:'):
                with self.lock:
                    self.data = json.loads(line[5:])
                    self.stamp = time.monotonic()
                if self.info:
                    return
            elif line.startswith('ERR:'):
                self.error = line[4:]
                break
        detail = self.error or 'Não chegaram leituras do Xbox pelo processo de aquisição.'
        self.close()
        raise _S11Pending(detail + ' Confira a seção de controle e o índice salvo no setup.')

    def _read_stdout(self):
        try:
            for line in self.process.stdout:
                line = line.strip()
                if line.startswith('DATA:') and self.info is not None:
                    with self.lock:
                        self.data = json.loads(line[5:])
                        self.stamp = time.monotonic()
                else:
                    self.lines.put(line)
        except (OSError, ValueError, json.JSONDecodeError) as exc:
            self.error = str(exc)

    def sample(self):
        with self.lock:
            age = time.monotonic() - self.stamp
            if self.process.poll() is not None or self.data is None or age > 1.0:
                return None
            return self.data

    def close(self):
        if self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=2)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=2)
        if self.process.stdout:
            self.process.stdout.close()
        if self.process.stderr:
            self.process.stderr.close()


class _S11JoystickProxy:
    def __init__(self, reader):
        self.reader = reader

    def get_init(self):
        return self.reader.sample() is not None

    def get_name(self):
        return self.reader.info['name']

    def get_numaxes(self):
        return int(self.reader.info['axes'])

    def get_numbuttons(self):
        return int(self.reader.info['buttons'])

    def get_numhats(self):
        return int(self.reader.info['hats'])

    def get_axis(self, index):
        value = self.reader.sample()
        return float(value['axes'][index]) if value and 0 <= index < len(value['axes']) else 0.0

    def get_button(self, index):
        value = self.reader.sample()
        return int(value['buttons'][index]) if value and 0 <= index < len(value['buttons']) else 0

    def get_hat(self, index):
        value = self.reader.sample()
        hats = value['hats'] if value else []
        return tuple(hats[index]) if 0 <= index < len(hats) else (0, 0)

    def init(self):
        pass

    def quit(self):
        pass


class _S11FakePygame:
    class event:
        @staticmethod
        def pump():
            pass  # O processo filho atualiza o controle a cada 50 ms.

    class joystick:
        @staticmethod
        def get_count():
            return 1


def _s11_status(state, title, message):
    palette = {'OK': ('#ecfdf3', '#166534'),
               'PENDENTE': ('#fff8e6', '#875d00'),
               'FALHA': ('#fef0ef', '#b42318'),
               'TESTANDO': ('#eff6ff', '#1d4ed8')}
    bg, fg = palette[state]
    s11_status.value = (
        f'<div style="background:{bg};color:{fg};padding:11px 13px;'
        'border-radius:8px;line-height:1.5;overflow-wrap:anywhere">'
        f'<b>{html.escape(title)} — {state}</b><br>'
        f'{html.escape(str(message))}</div>')


def _s11_show_checklist(report, controller_name, hardware):
    valid, pending = [], []

    def item(passed, yes, no):
        (valid if passed else pending).append(yes if passed else no)

    valid.append(f'Xbox {controller_name}: dados de eixos e botões recebidos pelo PC. '
                 'O acionamento de cada botão ainda depende das verificações abaixo.')
    received = int(report.get('pacotes_recebidos') or 0)
    sent = int(report.get('pacotes_enviados') or 0)
    item(bool(report.get('enlace_real') and received),
         f'PC ↔ ESP32: {received} telemetrias recebidas e {sent} comandos gerados.',
         'PC ↔ ESP32: nenhuma telemetria válida recebida.')
    item(bool(report.get('camera_recente')),
         'Câmera: quadros recentes chegaram ao processamento no PC.',
         'Câmera: a chegada de quadros recentes não foi confirmada.')
    item(bool(report.get('reconectou')),
         'Comunicação: retorno confirmado após a interrupção de teste.',
         'Comunicação: retorno após a interrupção ainda não confirmado.')
    item(bool(report.get('desarmado_apos_reconexao')),
         'Intertravamento: ESP32 reportou estado desarmado após reconectar.',
         'Intertravamento: estado desarmado após reconectar não confirmado.')
    if report.get('parada_operador'):
        valid.append('Estado de parada reportado. Isso não testa isoladamente '
                     'o botão de parada de emergência.')
    item(bool(report.get('rearmado')),
         'Rearme: ESP32 confirmou o comando durante o ensaio.',
         'Rearme pelo Xbox: nenhum estado armado foi registrado.')
    item(bool(report.get('tracao_comandada')),
         'Tração: telemetria registrou comando para as esteiras.',
         'Tração: não houve comando de esteiras confirmado na telemetria.')
    item(bool(report.get('servo_comandado')),
         'Canhão: telemetria registrou posição comandada fora da posição neutra.',
         'Canhão: não houve deslocamento comandado confirmado na telemetria.')
    item(bool(report.get('modo_automatico')),
         'Modo automático: estado registrado na telemetria.',
         'Modo automático: não foi registrado durante este ensaio.')
    item(bool(report.get('arrefecimento_comandado')),
         'Arrefecimento: comando registrado na telemetria.',
         'Arrefecimento: não houve comando registrado.')
    item(bool(report.get('jato_virtual')),
         'Jato virtual: estado ativo registrado na telemetria.',
         'Jato virtual: não foi ativado ou confirmado neste ensaio.')
    item(bool(report.get('pessoa_bloqueada')),
         'Pessoa: bloqueio registrado na telemetria.',
         'Bloqueio por pessoa: não foi exercitado ou confirmado.')
    for key, name in (('sensor_internal_valid', 'DHT interno'),
                      ('sensor_external_valid', 'DHT externo')):
        # Os campos da função de ensaio usam nomes em português.
        result_key = ('sensor_interno_valido' if 'internal' in key
                      else 'sensor_externo_valido')
        item(bool(report.get(result_key)),
             f'{name}: leitura válida recebida.',
             f'{name}: nenhuma leitura válida recebida (sensor opcional).')
    if not bool(hardware.get('physical_outputs', False)):
        pending.append('Movimento físico de motores, servos e relés: saídas '
                       'desabilitadas no setup; esta sessão não os comprova.')
    else:
        pending.append('Movimento físico de motores, servos e relés: '
                       'telemetria não substitui observação dos componentes.')
    pending.append('Inferência de fogo e pessoas: quadro recente da câmera '
                   'não confirma que os modelos detectaram alvos.')

    def box(title, items, background, border):
        lines = ''.join('<li style="margin:5px 0">' + html.escape(entry) + '</li>'
                        for entry in items)
        return (f'<div style="background:{background};border:1px solid {border};'
                'border-radius:8px;padding:12px;min-width:310px;flex:1 1 410px;'
                'box-sizing:border-box;overflow-wrap:anywhere">'
                f'<b>{title} ({len(items)})</b><ul style="padding-left:20px;'
                f'margin:7px 0 0">{lines}</ul></div>')

    s11_checklist.value = (
        '<div style="font-size:13px;line-height:1.45">'
        '<b>Resultado por função</b><br>'
        '<small>“Ainda não validado” significa que este ensaio não produziu '
        'evidência suficiente; não indica defeito do componente. Os comandos '
        'e as telemetrias podem ter cadências diferentes.</small>'
        '<div style="display:flex;flex-wrap:wrap;gap:10px;margin-top:8px">'
        + box('Validado', valid, '#ecfdf3', '#a7d8b8')
        + box('Ainda não validado', pending, '#fff8e6', '#e7cf8e')
        + '</div></div>')
    return valid, pending


def _s11_official():
    root = Path(ROOT)
    folder = Path(globals().get('SETUPS_DIR', root / 'pc_operador' / 'parameters_setups'))
    filename = folder / 'official' / 'active.json'
    if not filename.is_file():
        raise _S11Pending('Defina um setup oficial na seção 7 antes de iniciar.')
    data = json.loads(filename.read_text(encoding='utf-8'))
    if not isinstance(data, dict) or not data.get('_official', {}).get('name'):
        raise _S11Pending('O arquivo oficial está incompleto; ative o setup novamente.')
    return data


def _s11_release_previous_reader():
    # O leitor da seção 5 pode continuar ativo após a simulação.
    if 'control_stop' in globals():
        control_stop.set()
    if 'xbox_read_stop' in globals():
        xbox_read_stop.set()
    if 'xbox_process' in globals() and xbox_process is not None and xbox_process.poll() is None:
        xbox_process.terminate()
        try:
            xbox_process.wait(timeout=2)
        except subprocess.TimeoutExpired:
            xbox_process.kill()
    if 'xbox_read_thread' in globals() and xbox_read_thread is not None:
        xbox_read_thread.join(timeout=2)
    if 'stop_xbox_preview' in globals():
        stop_xbox_preview()


def _s11_prepare_cfg(official):
    if 'AppConfig' not in globals():
        raise _S11Pending('Execute as células iniciais do notebook para carregar AppConfig.')
    cfg = AppConfig()
    # Preserva defaults do runtime e aplica os parâmetros da combinação oficial.
    for key, value in official.items():
        if key.startswith('_'):
            continue
        if isinstance(value, dict) and isinstance(cfg.data.get(key), dict):
            cfg.data[key] = {**cfg.data[key], **value}
        else:
            cfg.data[key] = value
    hw = official.get('hardware') or {}
    cfg.data.setdefault('esp', {})
    if hw.get('transport'):
        cfg.data['esp']['transport'] = hw['transport']
    if hw.get('upload_port'):
        cfg.data['esp']['serial_port'] = hw['upload_port']
    if hw.get('host'):
        cfg.data['esp']['host'] = hw['host']
    if hasattr(cfg, '_vision_settings'):
        cfg.vision = cfg._vision_settings()
    return cfg


def _s11_run(_):
    if s11_start.disabled:
        return
    s11_start.disabled = True
    _s11_status('TESTANDO', 'Ensaio integrado', 'Conferindo setup e Xbox...')
    s11_progress.value = 0
    s11_progress.bar_style = 'info'
    s11_progress.layout.display = ''
    s11_phase.value = '<b>Preparação:</b> conferindo o controle e o setup oficial.'
    s11_report.value = ''
    s11_checklist.value = ''

    def worker():
        reader = None
        original_inputs = None
        runtime_module = None
        ticker_stop = threading.Event()
        try:
            official = _s11_official()
            cfg = _s11_prepare_cfg(official)
            if 'test' not in globals() or not callable(getattr(test, 'stage_integrated', None)):
                raise _S11Pending('Execute a célula inicial que define test.stage_integrated.')
            _s11_release_previous_reader()
            index = int(cfg.data.get('xbox_index', 0))
            reader = _S11XboxReader(index)
            _s11_status('OK', 'Xbox detectado',
                        f"{reader.info['name']} — índice {reader.info['index']}; "
                        f"{reader.info['axes']} eixos e {reader.info['buttons']} botões. "
                        'Iniciando câmera, visão e comunicação...')
            import runtime as runtime_module
            original_inputs = runtime_module.OperatorInputs

            class IntegratedInputs(original_inputs):
                def __init__(self, *args, **kwargs):
                    super().__init__(*args, **kwargs)
                    # Conserva a interpretação, os botões e os intertravamentos
                    # do runtime; troca somente a aquisição SDL no kernel.
                    self.joy = _S11JoystickProxy(reader)
                    self.pygame = _S11FakePygame()

            runtime_module.OperatorInputs = IntegratedInputs
            duration = 20
            begin = time.monotonic()

            def update_progress():
                while not ticker_stop.wait(.2):
                    elapsed = time.monotonic() - begin
                    s11_progress.value = min(duration, int(elapsed))
                    if elapsed < duration:
                        s11_phase.value = (
                            f'<b>Ensaio:</b> {min(duration, int(elapsed))} de '
                            f'{duration} segundos. Siga as instruções no registro abaixo.')
                    else:
                        s11_phase.value = (
                            '<b>Reconexão:</b> aguardando a verificação final. '
                            'A barra mede os 20 segundos da fase principal.')

            ticker = threading.Thread(target=update_progress, daemon=True)
            ticker.start()
            packet_file = Path(ROOT) / 'output' / 'testes_pc' / 'integrado_pacotes.json'
            packet_before = packet_file.stat().st_mtime_ns if packet_file.is_file() else None
            with s11_log:
                print('Xbox lido por processo separado:', reader.info['name'])
                print('Setup oficial:', official['_official']['name'])
                print('Comece com o controle neutro; acompanhe as fases abaixo.')
                print('Fase principal: 20 segundos; a checagem da reconexão ocorre em seguida.')
                report = test.stage_integrated(cfg, seconds=duration)
                print('\nO resultado por função está na lista abaixo do progresso.')
            if not isinstance(report, dict):
                raise RuntimeError('O ensaio não retornou um relatório estruturado.')
            ticker_stop.set()
            ticker.join(timeout=1)
            s11_progress.value = duration
            s11_progress.bar_style = 'success'
            s11_phase.value = '<b>Concluído:</b> fase principal e verificação de reconexão encerradas.'
            globals().setdefault('results', {})['integrado'] = report
            folder = Path(ROOT) / 'output' / 'testes_pc'
            folder.mkdir(parents=True, exist_ok=True)
            summary_file = folder / 'ensaio_integrado_resumo.json'
            packet_fresh = packet_file.is_file() and (
                packet_before is None or packet_file.stat().st_mtime_ns != packet_before)
            # A versão atual de stage_integrated conserva um roteiro de 45 s.
            # Com 20 s, 'false' pode significar que a fase não foi executada.
            checks = {
                'comunicacao_pc_esp': 'OK' if report.get('enlace_real') else 'FALHA',
                'video_recente': 'OK' if report.get('camera_recente') else 'PENDENTE',
                'xbox_dados_recebidos': 'OK',
                'reconexao': 'OK' if report.get('reconectou') else 'PENDENTE',
                'desarme_apos_reconexao': (
                    'OK' if report.get('desarmado_apos_reconexao') else 'PENDENTE'),
                'rearme_pelo_xbox': 'OK' if report.get('rearmado') else 'NAO_VERIFICADO',
                'tracao': 'OK' if report.get('tracao_comandada') else 'NAO_VERIFICADO',
                'servo': 'OK' if report.get('servo_comandado') else 'NAO_VERIFICADO',
                'modo_automatico': 'OK' if report.get('modo_automatico') else 'NAO_VERIFICADO',
                'bloqueio_por_pessoa': (
                    'OK' if report.get('pessoa_bloqueada') else 'NAO_VERIFICADO'),
            }
            validated, not_validated = _s11_show_checklist(
                report, reader.info['name'], official.get('hardware') or {})
            summary_file.write_text(json.dumps({
                'data_hora_local': _integrated_datetime.now().astimezone().isoformat(),
                'setup_oficial': official['_official']['name'],
                'duracao_fase_principal_s': duration,
                'controle': reader.info,
                'resultado': report,
                'avaliacao': checks,
                'validado': validated,
                'ainda_nao_validado': not_validated,
                'nota_avaliacao': (
                    'O roteiro legado reserva fases para depois de 20 s e menciona '
                    'botões anteriores ao mapeamento atual. NAO_VERIFICADO não '
                    'significa defeito do controle nem do ESP32. A quantidade '
                    'de comandos enviados e telemetrias recebidas não mede '
                    'perda de pacotes porque as cadências são diferentes.'),
                'registro_pacotes': str(packet_file.resolve()) if packet_fresh else None,
            }, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
            locations = [f'Resumo JSON: {summary_file.resolve()}']
            if packet_fresh:
                locations.append(f'Comandos e telemetria: {packet_file.resolve()}')
            s11_report.value = (
                '<div style="padding:10px 12px;border:1px solid #b8d5c0;'
                'border-radius:8px;white-space:normal;overflow-wrap:anywhere">'
                '<b>Arquivos do relatório</b><br>' + '<br>'.join(
                    html.escape(item) for item in locations) + '</div>')
            with s11_log:
                print('\nArquivos do relatório:')
                for item in locations:
                    print(item)
            if isinstance(report, dict):
                packets = report.get('pacotes_recebidos', 0)
                reconnect = report.get('reconectou', False)
                control_ok = report.get('rearmado') and (
                    report.get('tracao_comandada') or report.get('servo_comandado'))
                state = 'OK' if packets and reconnect and control_ok else 'PENDENTE'
                detail = (f'Telemetrias recebidas: {packets}; reconexão: '
                          f'{"confirmada" if reconnect else "não confirmada"}. ')
                if packets and reconnect and not control_ok:
                    detail += 'Comunicação aprovada; comandos do Xbox ainda não verificados. '
                detail += f'Relatório: {summary_file.resolve()}'
                _s11_status(state, 'Ensaio integrado',
                            detail)
            else:
                _s11_status('OK', 'Ensaio integrado', 'Execução concluída; confira o relatório.')
        except Exception as exc:
            s11_progress.bar_style = 'danger'
            s11_phase.value = '<b>Interrompido:</b> veja o motivo no status.'
            _s11_status('PENDENTE' if isinstance(exc, _S11Pending) else 'FALHA',
                        'Ensaio integrado', exc)
            with s11_log:
                print(f'{type(exc).__name__}: {exc}')
        finally:
            ticker_stop.set()
            if runtime_module is not None and original_inputs is not None:
                runtime_module.OperatorInputs = original_inputs
            if reader is not None:
                reader.close()
            s11_start.disabled = False

    threading.Thread(target=worker, daemon=True, name='blaze-integrado').start()


s11_start = widgets.Button(description='Iniciar ensaio integrado',
                           button_style='warning',
                           layout=widgets.Layout(min_width='230px', height='40px'))
s11_status = widgets.HTML(layout=widgets.Layout(width='100%'))
s11_phase = widgets.HTML(layout=widgets.Layout(width='100%'))
s11_checklist = widgets.HTML(layout=widgets.Layout(width='100%'))
s11_report = widgets.HTML(layout=widgets.Layout(width='100%'))
s11_progress = widgets.IntProgress(value=0, min=0, max=20,
                                   description='Tempo:', bar_style='info',
                                   layout=widgets.Layout(width='100%', display='none'))
s11_log = widgets.Output(layout=widgets.Layout(width='100%', max_height='380px',
                                              overflow_y='auto', border='1px solid #cbd5e1',
                                              padding='8px'))
s11_start.on_click(_s11_run)
_s11_status('PENDENTE', 'Ensaio integrado',
            'Ative o setup oficial e conclua a comunicação da seção 9; '
            'depois clique em Iniciar ensaio integrado.')
display(widgets.VBox([
    widgets.HTML('<b>PC + ESP32 — ensaio integrado</b><br>'
                 'Lê Xbox e câmera, envia comandos e recebe telemetria durante 20 s. '
                 'Em seguida, testa a interrupção e a reconexão.'),
    s11_start, s11_status, s11_progress, s11_phase,
    s11_checklist, s11_report, s11_log,
], layout=widgets.Layout(width='100%', max_width='1250px', grid_gap='8px')))


## 12. Abrir a central independente do notebook

Esta seção abre o programa do PC em processo separado. Depois, é possível iniciar diretamente `BLAZE_Simulacao.cmd` ou `BLAZE_Combate_Real.cmd`: na janela inicial selecione **uma combinação completa** salva na seção 7. A central lê a câmera USB ou o arquivo de vídeo selecionado e usa USB ou a rede local do ESP32 para comandos e telemetria, conforme o setup. Ao preparar o firmware, o SSID e a senha exclusivos são salvos em JSON local; a gravação do ESP32 é feita por cabo USB.

Na janela inicial também é possível selecionar binários gerados, conferir a versão gravada e abrir a calibração física quando câmera USB, servos e firmware de combate real estiverem disponíveis. A visão, os estados do controle e a telemetria são processados no PC; o ESP32 executa o firmware de sensores e atuadores.


In [ ]:
# Seção 12 — abre o runtime independente com uma combinação completa.
import subprocess
import sys
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from profile_manager import names, prepare

runtime_status = widgets.HTML(layout=widgets.Layout(width='100%'))
runtime_sim_button = widgets.Button(description='Abrir runtime de simulação',
    button_style='success',layout=widgets.Layout(min_width='250px'))
runtime_real_button = widgets.Button(description='Abrir runtime de combate real',
    button_style='warning',layout=widgets.Layout(min_width='270px'))
runtime_log_files = []

def _open_runtime(profile):
    try:
        available = names(profile)
        if not available:
            raise ValueError('Nenhuma combinação completa de ' + profile +
                ' está salva. Crie-a na seção 7; para combate real habilite as saídas físicas no hardware.')
        script = Path(ROOT) / 'pc_operador' / ('runtime_simulacao.py' if profile=='simulacao'
                                               else 'runtime_combate_real.py')
        log_dir = Path(ROOT) / 'output' / 'runtime_logs'
        log_dir.mkdir(parents=True,exist_ok=True)
        logfile = log_dir / (profile+'.log')
        stream = logfile.open('a',encoding='utf-8')
        flags = getattr(subprocess,'CREATE_NO_WINDOW',0) if sys.platform=='win32' else 0
        process = subprocess.Popen([sys.executable,str(script)],cwd=str(script.parent),
            stdout=stream,stderr=subprocess.STDOUT,creationflags=flags)
        stream.close()
        runtime_log_files.append((process,logfile))
        runtime_status.value = status_box('Central independente','OK',
            f'Processo {process.pid} iniciado. Escolha um setup completo na janela externa. Log: {logfile}').value
    except Exception as exc:
        runtime_status.value = status_box('Central independente','FALHA',
            f'{type(exc).__name__}: {exc}').value

runtime_sim_button.on_click(lambda _:_open_runtime('simulacao'))
runtime_real_button.on_click(lambda _:_open_runtime('combate_real'))
display(widgets.VBox([
    widgets.HTML('Os dois programas abaixo também podem ser executados diretamente, sem abrir este '
        'notebook. Em cada janela você escolhe <b>uma combinação completa</b>, abre a central, '
        'compila ou grava o firmware correspondente no ESP32.'),
    widgets.HBox([runtime_sim_button,runtime_real_button]),runtime_status
],layout=widgets.Layout(width='100%')))


## Envio de novos setups ao github

In [2]:
import os
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- CONFIGURAÇÕES DE AUTOMAÇÃO ---
MENSAGEM_COMMIT = "Atualização automática dos setups paramétricos (Notebook)"
PASTA_SETUPS = "pc_operador/parameters_setups/user/"

def verificar_identidade_git():
    """Verifica se o usuário atual possui nome e email configurados no Git."""
    try:
        email = subprocess.check_output(["git", "config", "user.email"], text=True, stderr=subprocess.DEVNULL).strip()
        name = subprocess.check_output(["git", "config", "user.name"], text=True, stderr=subprocess.DEVNULL).strip()
        return bool(email and name)
    except subprocess.CalledProcessError:
        return False

def configurar_identidade_anonimo(b):
    """Configura o Git local com os dados digitados nas caixas de texto."""
    with area_logs:
        if not txt_nome.value or not txt_email.value:
            print("❌ Por favor, preencha ambos os campos (Nome e E-mail) primeiro.")
            return
        try:
            subprocess.run(["git", "config", "--global", "user.name", txt_nome.value], check=True)
            subprocess.run(["git", "config", "--global", "user.email", txt_email.value], check=True)
            clear_output()
            print("✅ Identidade configurada com sucesso! Atualizando interface...")
            renderizar_interface_principal()
        except subprocess.CalledProcessError as e:
            print(f"❌ Erro ao configurar identidade: {e}")

def verificar_alteracoes_setups():
    """Verifica se há modificações especificamente na pasta de setups."""
    try:
        resultado = subprocess.check_output(
            ["git", "status", PASTA_SETUPS, "--porcelain"], text=True, stderr=subprocess.DEVNULL
        ).strip()
        return resultado != ""
    except subprocess.CalledProcessError:
        return False

def enviar_setups_github(b):
    """Executa o fluxo de git add, commit e push."""
    botao_enviar.disabled = True
    with area_logs:
        clear_output()
        print("🚀 Iniciando envio dos novos setups paramétricos...")
        try:
            subprocess.run(["git", "add", PASTA_SETUPS], check=True)
            print(f"✔️ Alterações em '{PASTA_SETUPS}' preparadas.")
            
            subprocess.run(["git", "commit", "-m", MENSAGEM_COMMIT], check=True)
            print(f"✔️ Registro salvo localmente: '{MENSAGEM_COMMIT}'")
            
            subprocess.run(["git", "push", "origin", "HEAD"], check=True)
            print("🎉 Perfeito! Novos setups enviados com sucesso ao GitHub!")
        except subprocess.CalledProcessError as e:
            print(f"❌ Erro no envio. Certifique-se de que tem permissão de escrita no repositório.\n{e}")
            botao_enviar.disabled = False

# --- CONTROLE DE INTERFACE ---
area_logs = widgets.Output()
txt_nome = widgets.Text(description="Nome Git:", placeholder="Seu Nome ou User")
txt_email = widgets.Text(description="E-mail Git:", placeholder="seu_email@exemplo.com")
btn_salvar_id = widgets.Button(description="Configurar Git", button_style="info", icon="user-check")

botao_enviar = widgets.Button(
    description="Confirmar e Enviar Setups",
    button_style="success",
    tooltip="Clique para sincronizar os setups com o GitHub",
    icon="upload"
)

def renderizar_interface_principal():
    """Renderiza o botão de envio se houver alterações."""
    with area_logs:
        clear_output()
    if verificar_alteracoes_setups():
        print("⚠️ Detecção: Existem novos setups ou parâmetros modificados!")
        botao_enviar.on_click(enviar_setups_github)
        display(botao_enviar, area_logs)
    else:
        print("✅ Tudo atualizado! Os setups salvos são idênticos aos do GitHub.")

# --- EXECUÇÃO INICIAL ---
if not verificar_identidade_git():
    print("👤 Identidade do Git não encontrada. Configure seus dados para liberar o envio:")
    btn_salvar_id.on_click(configurar_identidade_anonimo)
    display(txt_nome, txt_email, btn_salvar_id, area_logs)
else:
    renderizar_interface_principal()

✅ Tudo atualizado! Os setups salvos são idênticos aos do GitHub.
